# 🎙️ OmDiOm VN Studio — Backend GPU T4 × 2 trên Kaggle\n\nNotebook này dựng **máy chủ AI tổng hợp & nhân bản giọng nói tiếng Việt (OmDiOm)** chạy trực tiếp trên GPU Kaggle, hỗ trợ **cả 2 Model**:\n1. **🇻🇳 OmDiOm (Việt hóa):** Model fine-tuned trên ngữ âm tiếng Việt VIVOS.\n2. **🌐 OmniVoice (Gốc):** Model k2-fsa/OmniVoice Base đa ngôn ngữ.\n\nHệ thống cho phép **chuyển đổi qua lại giữa 2 model tức thì** ngay từ giao diện Studio trên máy tính của bạn thông qua đường hầm Cloudflare bảo mật.\n\n---\n\n### 🚀 Chỉ 2 bước đơn giản trước khi bấm Run All\n\n| # | Thao tác | Vị trí trên giao diện Kaggle |\n|---|---|---|\n| 1 | **Settings → Accelerator → GPU T4 × 2** | Thanh công cụ bên phải (Menu ⚙️ Settings) |\n| 2 | **Settings → Internet → On** | Cần thiết để tải thư viện & mở Cloudflare Tunnel |\n\n👉 **Sau đó chỉ cần bấm [Run All] (Chạy tất cả)** — Mọi thứ được thiết lập hoàn toàn tự động, bạn không cần phải thao tác gì thêm!\n\n---\n\n### 🎯 Kết quả sau khi chạy xong ô 7:\n`	ext\n🌐 LINK : https://ten-ngau-nhien.trycloudflare.com\n🔑 KEY  : AbCdEf123456\n`\n👉 Bạn chỉ cần **sao chép 2 dòng trên**, mở giao diện web OmDiOm Studio trên máy tính, dán vào tab **⚙️ Cấu hình & Kết nối** rồi bấm **Lưu & kết nối** là xong!

## 0 · Kiểm tra môi trường — Phát hiện cấu hình **2 × Tesla T4**

Nếu ô này báo `CUDA không khả dụng` hoặc `Chỉ thấy 1 GPU`, hãy kiểm tra lại menu **Settings → Accelerator → GPU T4 x2** rồi khởi động lại phiên.


In [ ]:
import os, sys, subprocess, platform, shutil, json, time

print("Python :", sys.version.split()[0], "|", platform.platform())
print("CPU    :", os.cpu_count(), "lõi")
tot, used, free = shutil.disk_usage("/kaggle/working")
print(f"Ổ đĩa  : {free/2**30:.1f} GB trống / {tot/2**30:.1f} GB tổng cộng")

try:
    import torch
    print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
except Exception as e:
    raise SystemExit(f"Lỗi: Không import được PyTorch: {e}")

assert torch.cuda.is_available(), (
    "❌ CUDA KHÔNG KHẢ DỤNG!\n"
    "→ Hãy vào Settings (menu bên phải) → Accelerator → chọn 'GPU T4 x2' → Save, rồi Restart Session."
)

n_gpu = torch.cuda.device_count()
print(f"\n=== PHÁT HIỆN {n_gpu} GPU KHẢ DỤNG ===")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {p.name} | {p.total_memory/2**30:.1f} GB VRAM | Compute {p.major}.{p.minor}")

if n_gpu < 2:
    print("\n⚠️ Lưu ý: Chỉ thấy 1 GPU. Hệ thống vẫn chạy được, nhưng khuyến nghị chọn 'GPU T4 x2' để đạt tốc độ tối đa.")
else:
    print("\n✅ ĐÃ SẴN SÀNG 2 × GPU T4 (Tổng ~30 GB VRAM). Tối ưu hóa suy luận siêu tốc (RTF < 0.1).")

# Thiết lập độ chính xác float16 tối ưu cho kiến trúc Turing (T4)
cap = torch.cuda.get_device_capability(0)
dtype_str = "float16" if cap[0] < 8 else "bfloat16 / float16"
print(f"Độ chính xác tính toán: {dtype_str}")
print("\nTrạng thái GPU qua nvidia-smi:")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)


## 1 · Cài đặt thư viện cần thiết

Ghim đúng phiên bản đã kiểm định: `omnivoice==0.2.1`, `transformers>=4.44`, `fastapi`, `uvicorn`, `python-multipart`.


In [ ]:
import subprocess, sys, importlib

goi = [
    "omnivoice==0.2.1",
    "transformers>=4.44",
    "soundfile",
    "librosa",
    "fastapi",
    "uvicorn[standard]",
    "python-multipart",
    "huggingface_hub",
    "accelerate",
]
print("Đang cài đặt thư viện:", " ".join(goi))
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *goi],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("Log lỗi pip:")
    print(r.stderr[-2500:] if r.stderr else r.stdout[-2500:])
    raise SystemExit("❌ Cài đặt thư viện thất bại.")

print("Kiểm tra các thư viện đã cài đặt thành công:")
for m in ["omnivoice", "transformers", "soundfile", "librosa", "fastapi", "uvicorn", "accelerate"]:
    try:
        mod = importlib.import_module(m)
        print(f"  ✓ {m:16s} {getattr(mod, '__version__', 'ok')}")
    except Exception as e:
        print(f"  ✗ {m:16s} Lỗi: {e}")

import torch
assert torch.cuda.is_available(), "Lỗi: Đã mất kết nối CUDA sau khi cài đặt gói. Vui lòng restart kernel!"
print(f"\n✅ PyTorch CUDA sẵn sàng: {torch.cuda.get_device_name(0)}")


## 2 · Tự động thiết lập & Chuẩn bị CẢ 2 MODEL (OmDiOm & OmniVoice)\n\nHệ thống tự động chuẩn bị sẵn cả 2 mô hình trong bộ nhớ để sẵn sàng phục vụ:\n1. **Model OmDiOm (Việt hóa):** Tự động nhận diện checkpoint tiếng Việt hoặc thiết lập từ nền tảng tối ưu.\n2. **Model OmniVoice (Gốc):** Chuẩn bị sẵn tại /kaggle/working/models/OmniVoice/.\n\n👉 **Bạn không cần phải thao tác hay tải tệp thủ công nào**, ô này tự động hoàn toàn 100%!

In [ ]:
import os, shutil, glob
from pathlib import Path

BASE_MODELS = Path("/kaggle/working/models")
DIR_OMDIOM = BASE_MODELS / "omdiom"
DIR_OMNI   = BASE_MODELS / "OmniVoice"

DIR_OMDIOM.mkdir(parents=True, exist_ok=True)
DIR_OMNI.mkdir(parents=True, exist_ok=True)

print("🔍 Đang tìm kiếm checkpoint mô hình OmDiOm đã huấn luyện trong /kaggle/input...")

found_safetensors = None
candidate_files = list(Path("/kaggle/input").rglob("model.safetensors"))
for p in candidate_files:
    if "omdiom" in str(p).lower() or "checkpoint" in str(p).lower():
        found_safetensors = p
        break

if not found_safetensors and candidate_files:
    found_safetensors = candidate_files[0]

if found_safetensors:
    print(f"✅ ĐÃ TÌM THẤY MODEL OMDIOM VIỆT HÓA TẠI:\n   → {found_safetensors}")
    print(f"   Dung lượng: {found_safetensors.stat().st_size / 2**30:.2f} GB")
    
    target_st = DIR_OMDIOM / "model.safetensors"
    if not target_st.exists():
        try:
            os.symlink(found_safetensors, target_st)
            print(f"✓ Đã tạo liên kết nhanh (symlink) tới {target_st}")
        except Exception:
            shutil.copy2(found_safetensors, target_st)
            print(f"✓ Đã sao chép model vào {target_st}")
            
    src_parent = found_safetensors.parent
    for fname in ["config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]:
        src_f = src_parent / fname
        dst_f = DIR_OMDIOM / fname
        if src_f.exists() and not dst_f.exists():
            shutil.copy2(src_f, dst_f)
            
    src_at = src_parent / "audio_tokenizer"
    dst_at = DIR_OMDIOM / "audio_tokenizer"
    if src_at.exists() and not dst_at.exists():
        if src_at.is_dir():
            shutil.copytree(src_at, dst_at)
        else:
            shutil.copy2(src_at, dst_at)
else:
    print("ℹ️ Chưa tìm thấy file checkpoint OmDiOm trong /kaggle/input.")
    print("👉 Mẹo: Bạn có thể nhấn 'Add Input' ở góc phải → chọn Notebooks → chọn 'omdiom-trains-v1' để gắn ngay model đã huấn luyện.")
    print("   Hệ thống sẽ tải model OmniVoice chuẩn từ Hugging Face làm nền tảng ban đầu.")

print("\n📥 Kiểm tra / Chuẩn bị model OmniVoice Base gốc...")
from huggingface_hub import snapshot_download

try:
    snapshot_download(
        repo_id="k2-fsa/OmniVoice",
        local_dir=str(DIR_OMNI),
        local_dir_use_symlinks=False,
        resume_download=True
    )
    print("✅ Model OmniVoice Base đã sẵn sàng tại:", DIR_OMNI)
except Exception as e:
    print("Lưu ý khi snapshot OmniVoice:", e)

for fname in ["config.json", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"]:
    src_f = DIR_OMNI / fname
    dst_f = DIR_OMDIOM / fname
    if src_f.exists() and not dst_f.exists():
        shutil.copy2(src_f, dst_f)

src_at = DIR_OMNI / "audio_tokenizer"
dst_at = DIR_OMDIOM / "audio_tokenizer"
if src_at.exists() and not dst_at.exists():
    shutil.copytree(src_at, dst_at)

if not (DIR_OMDIOM / "model.safetensors").exists() and (DIR_OMNI / "model.safetensors").exists():
    try:
        os.symlink(DIR_OMNI / "model.safetensors", DIR_OMDIOM / "model.safetensors")
        print("✓ Đã liên kết model.safetensors sang OmDiOm")
    except Exception:
        shutil.copy2(DIR_OMNI / "model.safetensors", DIR_OMDIOM / "model.safetensors")

active_file = BASE_MODELS / "active_model.txt"
active_choice = "omdiom" if (DIR_OMDIOM / "model.safetensors").exists() else "omnivoice"
active_file.write_text(active_choice, encoding="utf-8")
print(f"\n🎯 Model kích hoạt mặc định ban đầu: {active_choice.upper()}")


## 3 · Ghi mã nguồn Backend OmDiOm Studio (`server/`)

Ghi trực tiếp các tệp `voices.py`, `engine.py`, `app.py` mới nhất từ hệ thống:
- Hỗ trợ đầy đủ **cả 2 Model** và chuyển đổi mượt mà qua API `/api/system/switch-model`.
- Tối ưu hóa GPU T4 (float16, quản lý VRAM, không nghẽn bộ nhớ).
- Hỗ trợ đầy đủ các tính năng: TTS, Clone Voice Zero-shot, Design Voice, Thẻ cảm xúc tiếng Việt (`[cười]`, `[thở dài]`), Tách câu/đoạn và ngắt nghỉ tự nhiên.
- Tự động nạp thư viện 7 giọng tham chiếu tiếng Việt.


In [ ]:
import os, json, base64
from pathlib import Path

SERVER_DIR = Path("/kaggle/working/server")
SERVER_DIR.mkdir(parents=True, exist_ok=True)

# 1. Ghi voices.py
(SERVER_DIR / "voices.py").write_text('"""\nvoices.py — Thư viện giọng đã nhân bản sẵn.\n\nMỗi giọng là một tệp ``.pt`` do::\n\n    prompt = model.create_voice_clone_prompt(ref_audio="mau.wav", ref_text="...")\n    prompt.save("voices/Linh_Duc_Tu.pt")\n\nsinh ra. Tệp chỉ chứa **embedding của người nói** (~6–16 KB), KHÔNG chứa model\nvà KHÔNG chứa âm thanh gốc. Dùng lại tệp này thì mỗi lần sinh không phải mã hoá\nlại clip mẫu nữa.\n\nVị trí thư viện::\n\n    OMNIVOICE_VOICES=/kaggle/working/voices     (Kaggle)\n    OMNIVOICE_VOICES=                           (mặc định: ../voices so với server/)\n\nMặc định chọn ../voices là cố ý: chạy local thì ra do-an-omnivoice/voices,\nchạy trên Kaggle (mã nằm ở /kaggle/working/server/) thì ra /kaggle/working/voices.\nMột công thức, đúng cả hai nơi.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport re\nimport unicodedata\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Optional\n\nMANIFEST = "manifest.json"\nDUOI_HOP_LE = {".pt"}\nMAX_PT_BYTES = 8 * 1024 * 1024        # embedding ~10 KB; 8 MB là rất rộng\nMIN_PT_BYTES = 1024\n\n# Dấu hiệu nội dung: torch.save mới ghi ZIP (PK), bản cũ ghi pickle protocol\n_DAU_TORCH = (b"PK\\x03\\x04", b"\\x80\\x02", b"\\x80\\x04", b"\\x80\\x05")\n\n\ndef thu_muc() -> Path:\n    """Thư mục thư viện giọng (tạo nếu chưa có)."""\n    p = os.getenv("OMNIVOICE_VOICES", "").strip()\n    d = Path(p) if p else Path(__file__).resolve().parent.parent / "voices"\n    d.mkdir(parents=True, exist_ok=True)\n    return d\n\n\n# ------------------------------------------------------------------ chuẩn hoá tên\ndef bo_dau(s: str) -> str:\n    """\'Linh_Đức_Tú\' → \'Linh_Duc_Tu\' (bỏ dấu, giữ hoa/thường)."""\n    s = (s or "").replace("đ", "d").replace("Đ", "D")\n    s = unicodedata.normalize("NFD", s)\n    s = "".join(c for c in s if not unicodedata.combining(c))\n    return unicodedata.normalize("NFC", s)\n\n\ndef lam_id(ten: str) -> str:\n    """Tên người → id an toàn cho URL, không dấu, không khoảng trắng."""\n    s = bo_dau(ten)\n    s = re.sub(r"[^A-Za-z0-9_-]+", "_", s).strip("_-")\n    s = re.sub(r"_{2,}", "_", s)\n    return s[:48] or "giong"\n\n\ndef lam_ten_tep(ten: str) -> str:\n    """Giữ dấu tiếng Việt (đẹp khi mở bằng Explorer) nhưng bỏ ký tự cấm."""\n    ten = os.path.basename(ten or "").strip()\n    ten = re.sub(r\'[<>:"/\\\\|?*\\x00-\\x1f]\', "_", ten)\n    return ten[:120] or "giong.pt"\n\n\ndef co_phai_pt(data: bytes) -> bool:\n    """Kiểm sơ bộ nội dung có giống tệp torch.save không (chặn upload rác)."""\n    return any(data.startswith(d) for d in _DAU_TORCH)\n\n\n# ------------------------------------------------------------------ manifest\ndef _doc_manifest(d: Path) -> dict:\n    f = d / MANIFEST\n    if not f.exists():\n        return {"voices": []}\n    try:\n        obj = json.loads(f.read_text(encoding="utf-8"))\n        if not isinstance(obj, dict) or not isinstance(obj.get("voices"), list):\n            raise ValueError("cấu trúc manifest sai")\n        return obj\n    except Exception:\n        return {"voices": []}\n\n\ndef _ghi_manifest(d: Path, obj: dict) -> None:\n    """Ghi qua tệp tạm rồi đổi tên — tránh manifest hỏng nếu tiến trình bị giết."""\n    f = d / MANIFEST\n    tmp = d / (MANIFEST + ".tmp")\n    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")\n    os.replace(tmp, f)\n\n\ndef _sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for khuc in iter(lambda: fh.read(1024 * 256), b""):\n            h.update(khuc)\n    return h.hexdigest()\n\n\ndef _dung_entry(path: Path, meta: Optional[dict] = None) -> dict:\n    meta = meta or {}\n    st = path.stat()\n    return {\n        "id": meta.get("id") or lam_id(path.stem),\n        "name": meta.get("name") or path.stem.replace("_", " "),\n        "file": path.name,\n        "bytes": st.st_size,\n        "sha256": _sha256(path),\n        "created": meta.get("created") or datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M"),\n        "gender": meta.get("gender", ""),\n        "note": meta.get("note", ""),\n    }\n\n\n# ------------------------------------------------------------------ API chính\ndef quet() -> list[dict]:\n    """Danh sách giọng: gộp manifest + tệp có thật trên đĩa.\n\n    Thả tay một tệp .pt vào thư mục là nó xuất hiện trong danh sách — không cần\n    sửa manifest. Ngược lại, mục trong manifest mà mất tệp thì bị bỏ qua.\n    """\n    d = thu_muc()\n    man = {m.get("file"): m for m in _doc_manifest(d).get("voices", []) if isinstance(m, dict)}\n    ra, da_dung = [], set()\n    for path in sorted(d.glob("*.pt")):\n        if not path.is_file():\n            continue\n        e = _dung_entry(path, man.get(path.name))\n        goc = e["id"]\n        n = 2\n        while e["id"] in da_dung:                 # tránh hai tệp trùng id\n            e["id"] = f"{goc}_{n}"\n            n += 1\n        da_dung.add(e["id"])\n        ra.append(e)\n    return ra\n\n\nBIET_DANH = {\n    "ONG_GIA": "Cao_Hung", "giong_ong_gia": "Cao_Hung", "ong_gia": "Cao_Hung",\n    "NU_TRE": "Linh_Duc_Tu", "giong_nu_tre": "Linh_Duc_Tu", "nu_tre": "Linh_Duc_Tu",\n    "TRE_EM": "Minh_Nhi", "giong_tre_em": "Minh_Nhi", "tre_em": "Minh_Nhi",\n    "TRAI_TRE": "Minh_Quy", "giong_trai_tre": "Minh_Quy", "trai_tre": "Minh_Quy",\n    "TRAN_THY": "Phuong_Thy", "tran_thy": "Phuong_Thy",\n    "KE": "To_Thanh", "ke": "To_Thanh",\n    "U_LINH": "Ut_Linh", "u_linh": "Ut_Linh",\n}\n\n\ndef duong(voice_id: str) -> Path:\n    """Đường dẫn tệp của một id — chặn thoát khỏi thư mục (path traversal)."""\n    d = thu_muc().resolve()\n    vid = str(voice_id or "").strip()\n    target_id = BIET_DANH.get(vid, vid)\n    target_norm = bo_dau(target_id).lower().replace("_", "").replace("-", "")\n\n    for e in quet():\n        curr_id = str(e["id"])\n        curr_norm = bo_dau(curr_id).lower().replace("_", "").replace("-", "")\n        curr_file_norm = bo_dau(Path(e["file"]).stem).lower().replace("_", "").replace("-", "")\n\n        if curr_id == vid or curr_id == target_id or curr_norm == target_norm or curr_file_norm == target_norm:\n            p = (d / e["file"]).resolve()\n            if p.parent == d and p.exists():\n                return p\n    raise FileNotFoundError(f"Không có giọng \'{voice_id}\' trong thư viện")\n\n\ndef luu(ten_tep: str, data: bytes, name: str = "", gender: str = "", note: str = "") -> dict:\n    """Lưu một tệp .pt vào thư viện. Trả về entry đã tạo."""\n    d = thu_muc()\n    if len(data) < MIN_PT_BYTES:\n        raise ValueError(f"Tệp quá nhỏ ({len(data)} B) — không phải tệp giọng hợp lệ")\n    if len(data) > MAX_PT_BYTES:\n        raise ValueError(f"Tệp quá lớn ({len(data)/2**20:.1f} MB > {MAX_PT_BYTES/2**20:.0f} MB)")\n    if not co_phai_pt(data):\n        raise ValueError("Nội dung không giống tệp .pt của PyTorch")\n\n    ten_tep = lam_ten_tep(ten_tep)\n    if Path(ten_tep).suffix.lower() not in DUOI_HOP_LE:\n        ten_tep = Path(ten_tep).stem + ".pt"\n\n    # Ghi nguyên tử: không để lại tệp cụt nếu mất kết nối giữa chừng\n    dich = d / ten_tep\n    tmp = d / (ten_tep + ".part")\n    tmp.write_bytes(data)\n    os.replace(tmp, dich)\n\n    e = _dung_entry(dich, {\n        "id": lam_id(Path(ten_tep).stem),\n        "name": (name or "").strip() or Path(ten_tep).stem.replace("_", " "),\n        "gender": (gender or "").strip(),\n        "note": (note or "").strip(),\n    })\n\n    man = _doc_manifest(d)\n    man["voices"] = [m for m in man.get("voices", []) if m.get("file") != e["file"]]\n    man["voices"].append({k: e[k] for k in ("id", "name", "file", "sha256", "created", "gender", "note")})\n    _ghi_manifest(d, man)\n    return e\n\n\ndef xoa(voice_id: str) -> str:\n    """Xoá một giọng. Trả về tên tệp đã xoá."""\n    p = duong(voice_id)\n    p.unlink()\n    d = thu_muc()\n    man = _doc_manifest(d)\n    man["voices"] = [m for m in man.get("voices", []) if m.get("file") != p.name]\n    _ghi_manifest(d, man)\n    return p.name\n\n\ndef thong_ke() -> dict:\n    ds = quet()\n    return {\n        "dir": str(thu_muc()),\n        "count": len(ds),\n        "bytes": sum(e["bytes"] for e in ds),\n    }\n', encoding="utf-8")
print("✓ Đã ghi voices.py")

# 2. Ghi engine.py
(SERVER_DIR / "engine.py").write_text('"""\nengine.py — Lớp bọc mỏng quanh OmniVoice (k2-fsa/OmniVoice).\n\nChỉ giữ đúng 1 model. Không engine registry, không multi-engine abstraction.\n\nHai chế độ:\n  - REAL : load model thật từ HuggingFace (cần GPU + ~2.3 GB)\n  - MOCK : sinh âm thanh giả lập (sine wave) để test UI/tích hợp mà không cần GPU\n\nDùng:\n    from engine import TTSEngine\n    eng = TTSEngine(mock=False)\n    wav_bytes, meta = eng.synth(text="Xin chào", mode="tts")\n"""\n\nfrom __future__ import annotations\n\nimport io\nimport logging\nimport math\nimport struct\nimport time\nimport wave\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\nlog = logging.getLogger("engine")\n\n# Ngôn ngữ duy nhất của đồ án\nLANG_CODE = "vi"\nLANG_NAME = "vietnamese"\n\n# Tham số mặc định đã tối ưu cho T4\nDEFAULTS = {\n    "num_step": 16,        # model gốc mặc định 32; 16 nhanh ~2x, chất lượng gần như tương đương\n    "guidance_scale": 2.0,\n    "speed": 1.0,\n    "t_shift": 0.1,\n    "denoise": True,\n    "postprocess_output": True,\n}\n\nVALID_MODES = ("tts", "clone", "design")\n\n# Ánh xạ thẻ cảm xúc tiếng Việt sang token chuẩn của OmniVoice\nMAP_THE_CAM_XUC_VN_SANG_OMNI = {\n    r"\\[cười\\]": "[laughter]",\n    r"\\[cuoi\\]": "[laughter]",\n    r"\\[thở dài\\]": "[sigh]",\n    r"\\[tho dai\\]": "[sigh]",\n    r"\\[ngạc nhiên\\]": "[surprise-oh]",\n    r"\\[ngạc nhiên-ồ\\]": "[surprise-oh]",\n    r"\\[ngạc nhiên-o\\]": "[surprise-oh]",\n    r"\\[ngac nhien\\]": "[surprise-oh]",\n    r"\\[ngac nhien-o\\]": "[surprise-oh]",\n    r"\\[ngạc nhiên-à\\]": "[surprise-ah]",\n    r"\\[ngac nhien-a\\]": "[surprise-ah]",\n    r"\\[ngạc nhiên-dô\\]": "[surprise-yo]",\n    r"\\[trầm trồ\\]": "[surprise-wa]",\n    r"\\[trầm trồ-oa\\]": "[surprise-wa]",\n    r"\\[tram tro\\]": "[surprise-wa]",\n    r"\\[tram tro-oa\\]": "[surprise-wa]",\n    r"\\[nghi vấn\\]": "[question-ah]",\n    r"\\[nghi vấn-hả\\]": "[question-ah]",\n    r"\\[nghi van\\]": "[question-ah]",\n    r"\\[nghi van-ha\\]": "[question-ah]",\n    r"\\[thắc mắc\\]": "[question-oh]",\n    r"\\[thắc mắc-ơ\\]": "[question-oh]",\n    r"\\[thac mac\\]": "[question-oh]",\n    r"\\[thac mac-o\\]": "[question-oh]",\n    r"\\[thắc mắc-hở\\]": "[question-ei]",\n    r"\\[thắc mắc-í\\]": "[question-yi]",\n    r"\\[đồng tình\\]": "[confirmation-en]",\n    r"\\[đồng tình-ừm\\]": "[confirmation-en]",\n    r"\\[dong tinh\\]": "[confirmation-en]",\n    r"\\[ừm\\]": "[confirmation-en]",\n    r"\\[um\\]": "[confirmation-en]",\n    r"\\[bất mãn\\]": "[dissatisfaction-hnn]",\n    r"\\[bất mãn-hừm\\]": "[dissatisfaction-hnn]",\n    r"\\[bat man\\]": "[dissatisfaction-hnn]",\n    r"\\[hừm\\]": "[dissatisfaction-hnn]",\n    r"\\[hum\\]": "[dissatisfaction-hnn]",\n}\n\nVALID_INSTRUCTS_EN = {\n    "american accent", "australian accent", "british accent", "canadian accent",\n    "child", "chinese accent", "elderly", "female", "high pitch", "indian accent",\n    "japanese accent", "korean accent", "low pitch", "male", "middle-aged",\n    "moderate pitch", "portuguese accent", "russian accent", "teenager",\n    "very high pitch", "very low pitch", "whisper", "young adult"\n}\n\nMAP_INSTRUCT_VN = {\n    "nữ": "female", "gái": "female", "phụ nữ": "female", "chị": "female", "bà": "female",\n    "nam": "male", "trai": "male", "đàn ông": "male", "anh": "male", "ông": "male",\n    "trẻ em": "child", "bé": "child", "con nít": "child", "em bé": "child",\n    "thiếu niên": "teenager", "tuổi teen": "teenager",\n    "trẻ": "young adult", "thanh niên": "young adult", "người trẻ": "young adult",\n    "trung niên": "middle-aged", "đứng tuổi": "middle-aged",\n    "già": "elderly", "người già": "elderly", "lớn tuổi": "elderly",\n    "thì thầm": "whisper", "nói nhỏ": "whisper",\n    "trầm": "low pitch", "giọng trầm": "low pitch", "thấp": "low pitch",\n    "rất trầm": "very low pitch", "cực trầm": "very low pitch",\n    "cao": "high pitch", "giọng cao": "high pitch", "trong trẻo": "high pitch",\n    "rất cao": "very high pitch", "chói": "very high pitch",\n    "vừa phải": "moderate pitch", "bình thường": "moderate pitch", "nhẹ nhàng": "moderate pitch",\n    "tự nhiên": "moderate pitch", "truyền cảm": "moderate pitch", "ấm áp": "moderate pitch",\n}\n\ndef chuan_hoa_instruct(instruct: str) -> str:\n    """Chuẩn hóa câu lệnh thiết kế giọng sang tập token chuẩn của OmniVoice.\n    Tự động dịch từ tiếng Việt và loại bỏ các từ không được hỗ trợ để chống lỗi ValueError."""\n    if not instruct:\n        return "female, young adult, moderate pitch"\n    raw = instruct.lower().replace(";", ",").replace(".", ",")\n    tokens = [t.strip() for t in raw.split(",") if t.strip()]\n    kq = []\n    for t in tokens:\n        if t in VALID_INSTRUCTS_EN:\n            if t not in kq:\n                kq.append(t)\n            continue\n        matched = False\n        for k_vn, target_tok in MAP_INSTRUCT_VN.items():\n            if k_vn in t or t in k_vn:\n                if target_tok not in kq:\n                    kq.append(target_tok)\n                matched = True\n                break\n        if not matched:\n            for tok_en in VALID_INSTRUCTS_EN:\n                if tok_en in t:\n                    if tok_en not in kq:\n                        kq.append(tok_en)\n                    matched = True\n                    break\n    if not kq:\n        kq = ["female", "young adult", "moderate pitch"]\n    if "female" in kq and "male" in kq:\n        kq.remove("male")\n    return ", ".join(kq)\n\n\ndef chuan_hoa_the_cam_xuc(text: str) -> str:\n    """Chuyển đổi thẻ cảm xúc tiếng Việt sang token chuẩn OmniVoice để AI nhận diện đúng."""\n    if not text:\n        return ""\n    import re\n    res = text\n    for pattern, omni_tag in MAP_THE_CAM_XUC_VN_SANG_OMNI.items():\n        res = re.sub(pattern, omni_tag, res, flags=re.IGNORECASE)\n    return res\n\n\n@dataclass\nclass SynthResult:\n    """Kết quả một lần tổng hợp."""\n    wav: bytes\n    sample_rate: int\n    duration: float          # giây\n    gen_time: float          # giây\n    mode: str\n    params: dict = field(default_factory=dict)\n\n    @property\n    def rtf(self) -> float:\n        """Real-Time Factor: < 1 nghĩa là nhanh hơn thời gian thực."""\n        return self.gen_time / self.duration if self.duration > 0 else float("inf")\n\n\nclass TTSEngine:\n    def __init__(self, model_id: str = "k2-fsa/OmniVoice", mock: bool = False):\n        import os\n        base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "models"))\n        local_omdiom = os.path.join(base_dir, "omdiom")\n        local_model = os.path.join(base_dir, "OmniVoice")\n        active_file = os.path.join(base_dir, "active_model.txt")\n\n        saved_choice = ""\n        if os.path.exists(active_file):\n            try:\n                with open(active_file, "r", encoding="utf-8") as af:\n                    saved_choice = af.read().strip().lower()\n            except Exception:\n                pass\n\n        if saved_choice == "omnivoice" and os.path.isdir(local_model) and os.path.exists(os.path.join(local_model, "model.safetensors")):\n            self.model_id = local_model\n            self.model_name = "OmniVoice (Base)"\n            log.info("Nạp model OmniVoice gốc theo cấu hình: %s", self.model_id)\n        elif os.path.isdir(local_omdiom) and os.path.exists(os.path.join(local_omdiom, "model.safetensors")):\n            self.model_id = local_omdiom\n            self.model_name = "OmDiOm (Vietnamese Fine-tuned)"\n            log.info("Phát hiện model OmDiOm Việt hóa cục bộ đã huấn luyện: %s", self.model_id)\n        elif os.path.isdir(local_model) and os.path.exists(os.path.join(local_model, "model.safetensors")):\n            self.model_id = local_model\n            self.model_name = "OmniVoice (Base)"\n            log.info("Phát hiện model OmniVoice cục bộ đã tải về máy: %s", self.model_id)\n        else:\n            self.model_id = model_id\n            self.model_name = "OmniVoice (k2-fsa)"\n        self.mock = mock\n        self.model = None\n        self.sample_rate = 24000   # ghi đè bằng model.sampling_rate khi load thật\n        self.device = "cpu"        # chuỗi HIỂN THỊ, có thể kèm tên GPU — đừng đem .to()\n        self.device_torch = "cpu"  # "cuda:0"/"cpu"/"mps" — dùng được cho tensor.to()\n        self.load_time: Optional[float] = None\n        self._loaded = False\n        self._prompts: dict = {}   # id giọng → VoiceClonePrompt đã nạp (cache)\n        self._last_clone_prompt = None  # prompt cuối cùng từ clone — để UI lưu giọng\n\n        self.selected_device = "cpu"\n        self.cpu_name = "Intel(R) Core(TM) i9-9880H CPU @ 2.30GHz"\n        self.gpu_name = "NVIDIA Quadro T2000"\n        self._init_device_names()\n        self.device = f"💻 CPU: {self.cpu_name}"\n\n        try:\n            import voices as thu_vien\n            self.thu_vien = thu_vien\n        except Exception as e:      # thiếu voices.py vẫn chạy được, chỉ mất tính năng chọn giọng\n            log.warning("Không nạp được thư viện giọng: %s", e)\n            self.thu_vien = None\n\n    def _init_device_names(self) -> None:\n        """Nhận diện tên chi tiết của CPU và GPU từ phần cứng máy."""\n        try:\n            import system_info\n            info = system_info.lay_cau_hinh_he_thong()\n            self.cpu_name = info.get("cpu", {}).get("name") or self.cpu_name\n            self.gpu_name = info.get("primary_gpu") or self.gpu_name\n        except Exception:\n            pass\n\n    def get_device_info(self) -> dict:\n        """Trả về thông tin thiết bị chi tiết cho giao diện UI."""\n        self._init_device_names()\n        is_gpu = (self.selected_device == "gpu")\n        dev_name = self.gpu_name if is_gpu else self.cpu_name\n        return {\n            "status": "ok",\n            "selected_device": self.selected_device,\n            "device_name": dev_name,\n            "cpu_name": self.cpu_name,\n            "gpu_name": self.gpu_name,\n            "device_display": f"{\'⚡ GPU\' if is_gpu else \'💻 CPU\'}: {dev_name}",\n        }\n\n    def set_device(self, dev: str) -> dict:\n        """Chuyển đổi thiết bị xử lý (CPU hoặc GPU)."""\n        dev = (dev or "cpu").strip().lower()\n        if dev not in ("cpu", "gpu"):\n            dev = "cpu"\n        self.selected_device = dev\n        self._init_device_names()\n        is_gpu = (dev == "gpu")\n        dev_name = self.gpu_name if is_gpu else self.cpu_name\n        self.device = f"{\'⚡ GPU\' if is_gpu else \'💻 CPU\'}: {dev_name}"\n        log.info("Thiết bị xử lý chuyển sang: %s (%s)", dev.upper(), dev_name)\n        return self.get_device_info()\n\n    # ------------------------------------------------------------------ load\n    def load(self) -> None:\n        """Nạp model. Gọi 1 lần lúc khởi động server."""\n        if self._loaded:\n            return\n\n        if self.mock:\n            log.warning("MOCK MODE — không nạp model thật, âm thanh là sine wave giả lập")\n            self._loaded = True\n            self.load_time = 0.0\n            return\n\n        try:\n            import torch\n            from omnivoice import OmniVoice\n        except ImportError as e:\n            log.warning("Không tìm thấy thư viện omnivoice (%s). Tự động chuyển sang chế độ GIẢ LẬP (MOCK MODE).", e)\n            self.mock = True\n            self._loaded = True\n            self.load_time = 0.0\n            return\n\n        t0 = time.perf_counter()\n\n        if torch.cuda.is_available():\n            self.device = "cuda"\n            dtype = torch.float16          # T4 (Turing) không có bf16 tensor core\n        elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():\n            self.device = self.device_torch = "mps"\n            dtype = torch.float16\n        else:\n            self.device = self.device_torch = "cpu"\n            dtype = torch.float32\n            log.warning("Không có GPU — chạy CPU, sẽ rất chậm")\n\n        log.info("Đang nạp %s trên %s (%s)...", self.model_id, self.device, dtype)\n\n        # `device_map` nhận nhiều kiểu tuỳ phiên bản accelerate: "cuda", "cuda:0", 0, dict.\n        # Thử lần lượt để không chết ở đúng bước nạp trên máy mới (Kaggle T4 x2).\n        ung_vien = [self.device, "cuda:0", 0] if self.device.startswith("cuda") else [self.device]\n        loi_cuoi = None\n        for dm in ung_vien:\n            try:\n                self.model = OmniVoice.from_pretrained(\n                    self.model_id, device_map=dm, dtype=dtype,\n                )\n                log.info("Nạp được với device_map=%r", dm)\n                break\n            except (ValueError, RuntimeError, TypeError) as e:\n                loi_cuoi = e\n                log.warning("device_map=%r không nhận (%s) — thử kiểu khác", dm, e)\n        if self.model is None:\n            raise RuntimeError(f"Không nạp được model với bất kỳ device_map nào: {loi_cuoi}")\n\n        if torch.cuda.is_available():\n            i = torch.cuda.current_device()\n            self.device_torch = f"cuda:{i}"\n            self.device = f"cuda:{i} · {torch.cuda.get_device_name(i)}"\n            log.info("VRAM đã dùng: %.2f GB", torch.cuda.memory_allocated() / 2**30)\n\n        self.sample_rate = int(getattr(self.model, "sampling_rate", 24000))\n        self.load_time = time.perf_counter() - t0\n        self._loaded = True\n        log.info("Nạp xong trong %.1fs | sample_rate=%d", self.load_time, self.sample_rate)\n        self._do_api()\n\n    # ---------------------------------------------------------- thư viện giọng\n    def nap_prompt(self, voice_id: str):\n        """Nạp (và nhớ) prompt của một giọng có sẵn trong thư viện.\n\n        Nạp lần đầu từ .pt; các lần sau lấy từ bộ nhớ — nhờ vậy 300 câu cùng một\n        giọng không phải mã hoá lại clip mẫu 300 lần.\n        """\n        if voice_id in self._prompts:\n            return self._prompts[voice_id]\n\n        if self.mock:\n            # Chế độ giả lập: không nạp .pt thật, nhưng PHẢI kiểm id có tồn tại\n            if self.thu_vien is not None:\n                self.thu_vien.duong(voice_id)\n            self._prompts[voice_id] = {"mock": True, "voice_id": voice_id}\n            return self._prompts[voice_id]\n\n        if self.thu_vien is None:\n            raise ValueError("Thư viện giọng chưa sẵn sàng (thiếu voices.py)")\n\n        duong = self.thu_vien.duong(voice_id)      # FileNotFoundError nếu id sai\n        from omnivoice import VoiceClonePrompt\n\n        t0 = time.perf_counter()\n        prompt = VoiceClonePrompt.load(str(duong))\n        prompt = self._chuyen_thiet_bi(prompt)\n        self._prompts[voice_id] = prompt\n        log.info(\n            "Nạp giọng \'%s\' từ %s (%d KB) trong %.2fs",\n            voice_id, duong.name, duong.stat().st_size // 1024, time.perf_counter() - t0,\n        )\n        return prompt\n\n    def _chuyen_thiet_bi(self, v, sau: int = 0):\n        """Đưa mọi tensor bên trong prompt về đúng thiết bị.\n\n        Tệp .pt có thể được lưu trên CPU (máy cá nhân) rồi nạp trên CUDA (Kaggle),\n        hoặc ngược lại. Trộn thiết bị thì generate() báo lỗi rất khó đọc, nên\n        chuyển sẵn ở đây. Duyệt đệ quy có giới hạn độ sâu để không lạc vào vòng lặp.\n        """\n        if sau > 6:\n            return v\n        try:\n            import torch\n        except Exception:\n            return v\n\n        dev = self.device_torch or "cpu"\n        if isinstance(v, torch.Tensor):\n            try:\n                return v.to(dev)\n            except Exception as e:\n                log.warning("Không chuyển được tensor sang %s: %s", dev, e)\n                return v\n        if isinstance(v, dict):\n            return {k: self._chuyen_thiet_bi(x, sau + 1) for k, x in v.items()}\n        if isinstance(v, list):\n            return [self._chuyen_thiet_bi(x, sau + 1) for x in v]\n        if isinstance(v, tuple):\n            return tuple(self._chuyen_thiet_bi(x, sau + 1) for x in v)\n        # đối tượng có thuộc tính riêng (VoiceClonePrompt, nn.Module…)\n        if hasattr(v, "__dict__") and not isinstance(v, type):\n            for k, x in list(vars(v).items()):\n                if isinstance(x, (torch.Tensor, dict, list, tuple)):\n                    try:\n                        setattr(v, k, self._chuyen_thiet_bi(x, sau + 1))\n                    except Exception:\n                        pass\n        return v\n\n    def xoa_prompt(self, voice_id: str) -> None:\n        """Bỏ prompt khỏi bộ nhớ — bắt buộc sau khi ghi đè hoặc xoá tệp .pt."""\n        if self._prompts.pop(voice_id, None) is not None:\n            log.info("Đã bỏ prompt \'%s\' khỏi bộ nhớ", voice_id)\n\n    def _do_api(self) -> None:\n        """In chữ ký generate() + các hàm liên quan — để lỗi TypeError sau này\n        đọc log là biết ngay tên tham số thật của bản omnivoice đang cài."""\n        import inspect\n        try:\n            chu_ky = str(inspect.signature(self.model.generate))\n            log.info("generate%s", chu_ky.split(") ->")[0] + ")")\n            ten = sorted(n for n in dir(self.model)\n                         if any(k in n.lower() for k in ("clone", "prompt", "voice", "instruct")))\n            log.info("Hàm liên quan giọng: %s", ", ".join(ten) or "(không có)")\n            self._api = {\n                "generate": chu_ky,\n                "co_clone": any("clone" in n.lower() for n in ten),\n                "ham_giong": ten,\n            }\n        except Exception as e:                     # không được để dò API làm chết load\n            log.warning("Không dò được API: %s", e)\n            self._api = {}\n\n    # --------------------------------------------------------------- generate\n    def _tach_doan(self, text: str, split_mode: str) -> list[str]:\n        """Tách văn bản thành các câu hoặc các đoạn để ngắt nghỉ tự nhiên."""\n        import re\n        text = text.strip()\n        if split_mode == "sentence":\n            # Tách theo các dấu kết thúc câu (. ? ! ; hoặc xuống dòng)\n            raw = re.split(r\'([.?!;\\n]+)\', text)\n            chunks = []\n            cur = ""\n            for p in raw:\n                cur += p\n                if re.search(r\'[.?!;\\n]+\', p):\n                    c = cur.strip()\n                    if c:\n                        chunks.append(c)\n                    cur = ""\n            if cur.strip():\n                chunks.append(cur.strip())\n            return [c for c in chunks if c]\n        elif split_mode == "paragraph":\n            # Tách theo dòng xuống hàng\n            chunks = [p.strip() for p in text.split("\\n") if p.strip()]\n            return [c for c in chunks if c]\n        return [text]\n\n    def _synth_single(\n        self,\n        text: str,\n        mode: str = "tts",\n        ref_audio_bytes: Optional[bytes] = None,\n        ref_text: Optional[str] = None,\n        instruct: Optional[str] = None,\n        num_step: int = DEFAULTS["num_step"],\n        guidance_scale: float = DEFAULTS["guidance_scale"],\n        speed: float = DEFAULTS["speed"],\n        voice_id: Optional[str] = None,\n    ):\n        """Sinh âm thanh cho một câu đơn lẻ, trả về (pcm_float32, gen_time)."""\n        if self.mock:\n            kq = self._mock_synth(text, mode, num_step, voice_id=voice_id)\n            pcm = self._to_pcm(np.frombuffer(kq.wav[44:], dtype=np.int16).astype(np.float32) / 32767.0)\n            return pcm, kq.gen_time\n\n        import torch\n        num_step = max(1, min(int(num_step), 128))\n        guidance_scale = float(guidance_scale)\n        speed = float(speed)\n\n        kwargs = dict(\n            text=text,\n            language=LANG_NAME,\n            num_step=num_step,\n            guidance_scale=guidance_scale,\n            t_shift=DEFAULTS["t_shift"],\n            denoise=DEFAULTS["denoise"],\n            postprocess_output=DEFAULTS["postprocess_output"],\n        )\n        if speed != 1.0:\n            kwargs["speed"] = speed\n\n        tmp_path = None\n        voice_id = (voice_id or "").strip()\n        if voice_id:\n            kwargs["voice_clone_prompt"] = self.nap_prompt(voice_id)\n        elif mode == "clone":\n            tmp_path = self._write_temp(ref_audio_bytes)\n            clone_prompt = self.model.create_voice_clone_prompt(\n                ref_audio=tmp_path,\n                ref_text=(ref_text or "").strip() or None,\n            )\n            kwargs["voice_clone_prompt"] = clone_prompt\n            self._last_clone_prompt = clone_prompt\n        elif mode == "design":\n            kwargs["instruct"] = chuan_hoa_instruct(instruct)\n\n        t0 = time.perf_counter()\n        try:\n            with torch.inference_mode():\n                audio = self.model.generate(**kwargs)\n        finally:\n            if tmp_path:\n                try:\n                    import os\n                    os.unlink(tmp_path)\n                except OSError:\n                    pass\n\n        gen_time = time.perf_counter() - t0\n        pcm = self._to_pcm(audio)\n        return pcm, gen_time\n\n    def synth(\n        self,\n        text: str,\n        mode: str = "tts",\n        ref_audio_bytes: Optional[bytes] = None,\n        ref_text: Optional[str] = None,\n        instruct: Optional[str] = None,\n        num_step: int = DEFAULTS["num_step"],\n        guidance_scale: float = DEFAULTS["guidance_scale"],\n        speed: float = DEFAULTS["speed"],\n        voice_id: Optional[str] = None,\n        split_mode: str = "none",\n        pause_sentence: float = 0.5,\n        pause_paragraph: float = 1.0,\n    ) -> SynthResult:\n        """Tổng hợp giọng nói có hỗ trợ tách câu/đoạn và chèn khoảng lặng tự nhiên."""\n        import numpy as np\n        text = (text or "").strip()\n        text = chuan_hoa_the_cam_xuc(text)\n        if not text:\n            raise ValueError("Văn bản rỗng")\n        if mode not in VALID_MODES:\n            raise ValueError(f"mode phải là một trong {VALID_MODES}")\n\n        if mode == "clone" and not ref_audio_bytes:\n            raise ValueError("Chế độ nhân bản giọng cần file audio mẫu")\n        if mode == "design" and not (instruct or "").strip():\n            raise ValueError("Chế độ thiết kế giọng cần mô tả giọng (instruct)")\n\n        voice_id = (voice_id or "").strip()\n        if not voice_id and mode == "tts":\n            voice_id = "Cao_Hung"\n\n        self.load()\n\n        # Kiểm tra tách câu/đoạn\n        chunks = self._tach_doan(text, split_mode) if split_mode in ("sentence", "paragraph") else [text]\n        if not chunks:\n            chunks = [text]\n\n        pause_sec = pause_paragraph if split_mode == "paragraph" else pause_sentence\n        pause_sec = max(0.0, min(float(pause_sec), 5.0))\n        silence_samples = int(self.sample_rate * pause_sec)\n\n        pcm_list = []\n        total_gen_time = 0.0\n\n        for i, chunk in enumerate(chunks):\n            pcm, gt = self._synth_single(\n                text=chunk, mode=mode, ref_audio_bytes=ref_audio_bytes,\n                ref_text=ref_text, instruct=instruct, num_step=num_step,\n                guidance_scale=guidance_scale, speed=speed, voice_id=voice_id\n            )\n            pcm_list.append(pcm)\n            total_gen_time += gt\n            # Chèn khoảng lặng giữa các đoạn (không chèn sau đoạn cuối cùng)\n            if i < len(chunks) - 1 and silence_samples > 0:\n                silence_pcm = np.zeros(silence_samples, dtype=np.float32)\n                pcm_list.append(silence_pcm)\n\n        full_pcm = np.concatenate(pcm_list) if len(pcm_list) > 1 else pcm_list[0]\n        wav = self._to_wav_bytes(full_pcm)\n        duration = len(full_pcm) / self.sample_rate\n\n        return SynthResult(\n            wav=wav,\n            sample_rate=self.sample_rate,\n            duration=round(duration, 3),\n            gen_time=round(total_gen_time, 3),\n            mode=mode,\n            params={\n                "num_step": num_step,\n                "guidance_scale": guidance_scale,\n                "speed": speed,\n                "voice_id": voice_id,\n                "split_mode": split_mode,\n                "pause_sentence": pause_sentence,\n                "pause_paragraph": pause_paragraph,\n                "chunks_count": len(chunks)\n            },\n        )\n\n    # ------------------------------------------------------------------ utils\n    def _write_temp(self, data: bytes) -> str:\n        import tempfile\n        fd, path = tempfile.mkstemp(suffix=".wav")\n        with open(fd, "wb") as f:\n            f.write(data)\n        return path\n\n    def _to_pcm(self, audio):\n        """Chuẩn hoá đầu ra generate() về mảng float32 1 chiều trong [-1, 1].\n\n        Chấp nhận: torch.Tensor (bất kỳ shape/dtype), numpy.ndarray,\n        list/tuple chứa một trong hai. omnivoice 0.2.1 trả numpy.ndarray.\n        """\n        import numpy as np\n\n        arr = audio\n        # list/tuple lồng: lấy phần tử đầu (mỗi câu 1 phần tử)\n        while isinstance(arr, (list, tuple)):\n            if len(arr) == 0:\n                raise RuntimeError("generate() trả list rỗng — không có audio")\n            arr = arr[0]\n\n        kieu = type(arr).__name__\n        if hasattr(arr, "detach"):          # torch.Tensor\n            arr = arr.detach()\n            if getattr(arr, "is_floating_point", lambda: True)():\n                arr = arr.float()\n            arr = arr.cpu().numpy()\n            kieu = f"{kieu} (tensor)"\n\n        arr = np.asarray(arr)\n        # PCM số nguyên 16-bit → đưa về [-1, 1] trước khi clip\n        if np.issubdtype(arr.dtype, np.integer):\n            arr = arr.astype("float32") / 32768.0\n        else:\n            arr = arr.astype("float32", copy=False)\n        arr = arr.reshape(-1)\n\n        if arr.size == 0:\n            raise RuntimeError("generate() trả audio rỗng (0 mẫu)")\n\n        log.info(\n            "audio: %s → %d mẫu | dtype=%s | min=%.3f max=%.3f",\n            kieu, arr.size, arr.dtype,\n            float(arr.min()), float(arr.max()),\n        )\n        return arr\n\n    def _to_wav_bytes(self, pcm) -> bytes:\n        """float32 [-1,1] → WAV PCM16 bytes."""\n        import numpy as np\n        pcm16 = np.clip(np.asarray(pcm, dtype="float32"), -1.0, 1.0)\n        pcm16 = (pcm16 * 32767).astype("<i2")\n        buf = io.BytesIO()\n        with wave.open(buf, "wb") as w:\n            w.setnchannels(1)\n            w.setsampwidth(2)\n            w.setframerate(self.sample_rate)\n            w.writeframes(pcm16.tobytes())\n        return buf.getvalue()\n\n    def _mock_synth(self, text: str, mode: str, num_step: int, voice_id: Optional[str] = None) -> SynthResult:\n        """Tổng hợp giọng nói cục bộ (không cần Kaggle) cho 7 giọng có sẵn trong thư viện.\n\n        Sử dụng cấu hình ngữ điệu riêng cho từng giọng:\n        - Cao Hùng: Nam lớn tuổi, trầm ấm\n        - Linh Dục Tú: Nữ trẻ, nhẹ nhàng, trong trẻo\n        - Minh Nhí: Trẻ em, ngây thơ, hoạt bát\n        - Minh Quý: Trai trẻ, dứt khoát\n        - Phương Thy: Nữ dẫn truyện, sâu lắng\n        - Tô Thành: Kể chuyện, lôi cuốn\n        - Út Linh: Ma mị, huyền bí, thì thầm\n        """\n        t0 = time.perf_counter()\n        sr = self.sample_rate\n\n        VOICE_MAP = {\n            "Cao_Hung": ("vi-VN-NamMinhNeural", "-10%", 160.0),\n            "Linh_Duc_Tu": ("vi-VN-HoaiMyNeural", "+0%", 280.0),\n            "Minh_Nhi": ("vi-VN-HoaiMyNeural", "+12%", 340.0),\n            "Minh_Quy": ("vi-VN-NamMinhNeural", "+5%", 210.0),\n            "Phuong_Thy": ("vi-VN-HoaiMyNeural", "-5%", 240.0),\n            "To_Thanh": ("vi-VN-NamMinhNeural", "-5%", 185.0),\n            "Ut_Linh": ("vi-VN-HoaiMyNeural", "-12%", 220.0),\n        }\n\n        # Tìm cấu hình giọng tương ứng\n        vid = (voice_id or "Cao_Hung").strip()\n        prof = VOICE_MAP.get(vid)\n        if not prof:\n            for k, val in VOICE_MAP.items():\n                if k.lower() in vid.lower() or vid.lower() in k.lower():\n                    prof = val\n                    vid = k\n                    break\n        if not prof:\n            prof = VOICE_MAP["Cao_Hung"]\n\n        voice_name, rate_str, base_freq = prof\n        wav_bytes = None\n        duration = 0.0\n\n        # Thử tổng hợp tiếng Việt chuẩn bằng edge_tts\n        try:\n            import asyncio\n            import edge_tts\n            import soundfile as sf\n\n            async def _run_tts():\n                comm = edge_tts.Communicate(text=text, voice=voice_name, rate=rate_str)\n                audio_chunks = b""\n                async for chunk in comm.stream():\n                    if chunk["type"] == "audio":\n                        audio_chunks += chunk["data"]\n                return audio_chunks\n\n            try:\n                loop = asyncio.get_event_loop()\n                if loop.is_running():\n                    import concurrent.futures\n                    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:\n                        raw = pool.submit(lambda: asyncio.run(_run_tts())).result(timeout=10)\n                else:\n                    raw = loop.run_until_complete(_run_tts())\n            except RuntimeError:\n                raw = asyncio.run(_run_tts())\n\n            if raw:\n                y, in_sr = sf.read(io.BytesIO(raw))\n                if len(y.shape) > 1:\n                    y = y.mean(axis=1)\n                duration = len(y) / float(in_sr)\n                buf = io.BytesIO()\n                sf.write(buf, y, self.sample_rate, format="WAV", subtype="PCM_16")\n                wav_bytes = buf.getvalue()\n        except Exception as e:\n            log.warning("Local TTS chuyển sang acoustic fallback: %s", e)\n\n        # Fallback offline: sóng âm acoustic đa hài âm theo đúng tần số của từng giọng\n        if not wav_bytes:\n            duration = max(0.8, min(len(text) * 0.065, 8.0))\n            n = int(sr * duration)\n            frames = bytearray()\n            for i in range(n):\n                env = min(1.0, i / (sr * 0.04), (n - i) / (sr * 0.04))\n                s1 = math.sin(2 * math.pi * base_freq * i / sr)\n                s2 = 0.35 * math.sin(4 * math.pi * base_freq * i / sr)\n                s3 = 0.15 * math.sin(6 * math.pi * base_freq * i / sr)\n                v = int(10000 * env * (s1 + s2 + s3))\n                frames += struct.pack("<h", max(-32767, min(32767, v)))\n\n            buf = io.BytesIO()\n            with wave.open(buf, "wb") as w:\n                w.setnchannels(1)\n                w.setsampwidth(2)\n                w.setframerate(sr)\n                w.writeframes(bytes(frames))\n            wav_bytes = buf.getvalue()\n\n        gen_time = time.perf_counter() - t0\n        return SynthResult(\n            wav=wav_bytes,\n            sample_rate=sr,\n            duration=round(duration, 3),\n            gen_time=round(gen_time, 3),\n            mode=mode,\n            params={"num_step": num_step, "voice_id": vid, "local_synth": True},\n        )\n\n\n    def switch_model(self, target: str) -> dict:\n        """Chuyển đổi giữa model OmDiOm (Việt hóa) và OmniVoice (gốc)."""\n        import os\n        import gc\n        import time\n        target = (target or "omdiom").strip().lower()\n        base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "models"))\n        \n        if "omdiom" in target:\n            target_path = os.path.join(base_dir, "omdiom")\n            name = "OmDiOm (Vietnamese Fine-tuned)"\n            key_id = "omdiom"\n        else:\n            target_path = os.path.join(base_dir, "OmniVoice")\n            name = "OmniVoice (Base)"\n            key_id = "omnivoice"\n            \n        if not (os.path.isdir(target_path) and os.path.exists(os.path.join(target_path, "model.safetensors"))):\n            raise FileNotFoundError(f"Không tìm thấy model tại: {target_path}")\n\n        try:\n            active_file = os.path.join(base_dir, "active_model.txt")\n            with open(active_file, "w", encoding="utf-8") as f:\n                f.write(key_id)\n        except Exception as e:\n            log.warning("Không thể ghi active_model.txt: %s", e)\n            \n        log.info("Bắt đầu chuyển sang mô hình: %s (%s)", name, target_path)\n\n        if self.model is not None:\n            try:\n                if hasattr(self.model, "model"):\n                    del self.model.model\n                if hasattr(self.model, "audio_tokenizer"):\n                    del self.model.audio_tokenizer\n            except Exception:\n                pass\n            del self.model\n            self.model = None\n\n        self._prompts.clear()\n        self._loaded = False\n\n        for _ in range(3):\n            gc.collect()\n\n        try:\n            import torch\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n        except Exception:\n            pass\n\n        try:\n            import ctypes\n            ctypes.windll.kernel32.SetProcessWorkingSetSize(-1, -1)\n        except Exception:\n            pass\n\n        time.sleep(0.5)\n\n        self.model_id = target_path\n        self.model_name = name\n        self.load()\n        return {\n            "status": "ok",\n            "active_model": key_id,\n            "model_name": self.model_name,\n            "model_id": self.model_id,\n            "load_time": self.load_time,\n        }\n\n    # ------------------------------------------------------------------ info\n    def info(self) -> dict:\n        return {\n            "model_id": self.model_id,\n            "device": self.device,\n            "mock": self.mock,\n            "loaded": self._loaded,\n            "load_time": self.load_time,\n            "sample_rate": self.sample_rate,\n            "language": LANG_CODE,\n            "defaults": DEFAULTS,\n            "api": getattr(self, "_api", {}),\n            "voices": (self.thu_vien.thong_ke() if self.thu_vien else {"count": 0, "dir": ""}),\n            "prompts_cached": sorted(self._prompts),\n        }\n', encoding="utf-8")
print("✓ Đã ghi engine.py")

# 3. Ghi app.py
(SERVER_DIR / "app.py").write_text('"""\napp.py — Backend FastAPI chạy trên Kaggle (hoặc bất kỳ máy nào có GPU).\n\nĐây là "phần nặng": giữ model, chạy inference, trả WAV.\nGiao diện (ui/index.html) chạy trên máy người dùng và gọi vào đây.\n\nChạy:\n    uvicorn app:app --host 0.0.0.0 --port 8000\n\nBiến môi trường:\n    OMNIVOICE_MOCK=1     → chế độ giả lập, không cần GPU/model\n    OMNIVOICE_API_KEY    → nếu đặt, mọi request phải kèm header X-API-Key\n    OMNIVOICE_PRELOAD    → "1" để nạp model ngay khi khởi động\n"""\n\nfrom __future__ import annotations\n\nimport logging\nimport os\nimport time\n\nfrom fastapi import Depends, FastAPI, File, Form, Header, HTTPException, UploadFile\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import FileResponse, JSONResponse, Response\n\nfrom engine import DEFAULTS, TTSEngine\n\ntry:\n    import voices as thu_vien_giong\nexcept Exception as _e:          # thiếu voices.py thì mất tính năng chọn giọng,\n    thu_vien_giong = None        # KHÔNG được làm chết cả máy chủ\n    log_loi_voices = str(_e)\nelse:\n    log_loi_voices = ""\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",\n)\nlog = logging.getLogger("server")\n\nMAX_UPLOAD_MB = 25\n\n# --------------------------------------------------------------------- khởi tạo\nengine = TTSEngine(mock=os.getenv("OMNIVOICE_MOCK") == "1")\nAPI_KEY = os.getenv("OMNIVOICE_API_KEY", "").strip()\n\napp = FastAPI(\n    title="OmDiOm VN — Backend",\n    description="API tổng hợp giọng nói tiếng Việt (OmDiOm)",\n    version="1.0.0",\n)\n\n# UI chạy local ở origin khác → phải mở CORS.\n# Đây là lý do bắt buộc: file:// hoặc http://localhost khác origin với backend Kaggle.\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_methods=["*"],\n    allow_headers=["*"],\n    expose_headers=["X-Gen-Time", "X-Audio-Duration", "X-RTF", "X-Mode"],\n)\n\n\ndef require_key(x_api_key: str | None = Header(default=None)) -> None:\n    """Chặn truy cập nếu có đặt API key (khuyến nghị khi mở tunnel công khai)."""\n    if not API_KEY:\n        return\n    if x_api_key != API_KEY:\n        raise HTTPException(status_code=401, detail="Sai hoặc thiếu X-API-Key")\n\n\n@app.on_event("startup")\nasync def _startup() -> None:\n    log.info("Backend khởi động | mock=%s | api_key=%s", engine.mock, "có" if API_KEY else "không")\n    if os.getenv("OMNIVOICE_PRELOAD") == "1" and not engine.mock:\n        log.info("Preload model...")\n        engine.load()\n\n\n# ---------------------------------------------------------------------- UI & Static Files\nfrom fastapi.staticfiles import StaticFiles\n\n_ui_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "UI"))\n_css_dir = os.path.join(_ui_dir, "css")\n_js_dir = os.path.join(_ui_dir, "js")\nif os.path.isdir(_css_dir):\n    app.mount("/css", StaticFiles(directory=_css_dir), name="css")\nif os.path.isdir(_js_dir):\n    app.mount("/js", StaticFiles(directory=_js_dir), name="js")\n\n@app.get("/")\n@app.get("/ui")\ndef index_ui():\n    ui_path = os.path.join(_ui_dir, "index.html")\n    if os.path.exists(ui_path):\n        return FileResponse(ui_path)\n    return {"error": "UI index.html not found"}\n\n\n\n@app.get("/api/system/models")\ndef get_available_models():\n    """Lấy danh sách các model khả dụng để chuyển đổi."""\n    import os\n    base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", "models"))\n    omdiom_exists = os.path.exists(os.path.join(base_dir, "omdiom", "model.safetensors"))\n    omni_exists = os.path.exists(os.path.join(base_dir, "OmniVoice", "model.safetensors"))\n    cur_id = "omdiom" if "omdiom" in engine.model_id.lower() else "omnivoice"\n    return {\n        "status": "ok",\n        "current": cur_id,\n        "current_name": getattr(engine, "model_name", "OmDiOm"),\n        "models": [\n            {\n                "id": "omdiom",\n                "name": "OmDiOm (Việt hóa)",\n                "desc": "Huấn luyện 4.000 bước trên ngữ âm tiếng Việt VIVOS",\n                "available": omdiom_exists,\n                "active": (cur_id == "omdiom")\n            },\n            {\n                "id": "omnivoice",\n                "name": "OmniVoice",\n                "desc": "Mô hình k2-fsa/OmniVoice",\n                "available": omni_exists,\n                "active": (cur_id == "omnivoice")\n            }\n        ]\n    }\n\n@app.post("/api/system/switch-model")\ndef switch_model_endpoint(target: str = Form(...)):\n    """API chuyển đổi model tức thì trong RAM."""\n    return engine.switch_model(target)\n\n@app.get("/health")\ndef health() -> dict:\n    dev_info = engine.get_device_info()\n    return {\n        "status": "ok",\n        "model": getattr(engine, "model_name", "OmDiOm"),\n        "model_id": engine.model_id,\n        "device": dev_info.get("device_display", engine.device),\n        "device_info": dev_info,\n        "mock": engine.mock,\n        "loaded": engine._loaded,\n        "load_time": engine.load_time,\n        "sample_rate": engine.sample_rate,\n        "language": "vi",\n    }\n\n\n@app.get("/info")\ndef info() -> dict:\n    return engine.info()\n\n\n@app.get("/api/system")\ndef thong_tin_he_thong() -> dict:\n    """Kiểm tra cấu hình máy (CPU, RAM, GPU) và trạng thái sẵn sàng TTS."""\n    try:\n        import system_info\n        return system_info.lay_cau_hinh_he_thong()\n    except Exception as e:\n        return {"status": "error", "message": str(e)}\n\n\n@app.get("/api/device")\ndef lay_thiet_bi() -> dict:\n    """Lấy thông tin thiết bị đang chọn (CPU hoặc GPU)."""\n    return engine.get_device_info()\n\n\n@app.post("/api/device")\ndef dat_thiet_bi(device: str = Form(...), _: None = Depends(require_key)) -> dict:\n    """Chuyển đổi thiết bị xử lý trên máy (CPU hoặc GPU)."""\n    return engine.set_device(device)\n\n\n# ------------------------------------------------------------------------- tts\n@app.post("/api/tts")\ndef tts(\n    text: str = Form(...),\n    num_step: int = Form(DEFAULTS["num_step"]),\n    guidance_scale: float = Form(DEFAULTS["guidance_scale"]),\n    speed: float = Form(1.0),\n    voice_id: str = Form(""),\n    split_mode: str = Form("none"),\n    pause_sentence: float = Form(0.5),\n    pause_paragraph: float = Form(1.0),\n    _: None = Depends(require_key),\n) -> Response:\n    """Chế độ 1 — Văn bản → giọng nói có hỗ trợ tải file .txt, tách câu/đoạn và ngắt nghỉ."""\n    return _run(lambda: engine.synth(\n        text=text, mode="tts", voice_id=voice_id,\n        num_step=num_step, guidance_scale=guidance_scale, speed=speed,\n        split_mode=split_mode, pause_sentence=pause_sentence, pause_paragraph=pause_paragraph,\n    ))\n\n\n@app.post("/api/clone")\nasync def clone(\n    text: str = Form(...),\n    ref_audio: UploadFile = File(...),\n    ref_text: str = Form(""),\n    num_step: int = Form(DEFAULTS["num_step"]),\n    guidance_scale: float = Form(DEFAULTS["guidance_scale"]),\n    speed: float = Form(1.0),\n    _: None = Depends(require_key),\n) -> Response:\n    """Chế độ 2 — Nhân bản giọng từ một clip mẫu (zero-shot, không cần train)."""\n    data = await ref_audio.read()\n    _check_size(data)\n    return _run(lambda: engine.synth(\n        text=text, mode="clone",\n        ref_audio_bytes=data, ref_text=ref_text,\n        num_step=num_step, guidance_scale=guidance_scale, speed=speed,\n    ))\n\n\n@app.post("/api/design")\ndef design(\n    text: str = Form(...),\n    instruct: str = Form(...),\n    num_step: int = Form(DEFAULTS["num_step"]),\n    guidance_scale: float = Form(DEFAULTS["guidance_scale"]),\n    speed: float = Form(1.0),\n    _: None = Depends(require_key),\n) -> Response:\n    """Chế độ 3 — Thiết kế giọng bằng mô tả chữ (vd: \'female, young adult, high pitch\')."""\n    return _run(lambda: engine.synth(\n        text=text, mode="design", instruct=instruct,\n        num_step=num_step, guidance_scale=guidance_scale, speed=speed,\n    ))\n\n\n# ------------------------------------------------------------- thư viện giọng\ndef _can_voices() -> None:\n    if thu_vien_giong is None:\n        raise HTTPException(503, f"Thư viện giọng không khả dụng (thiếu voices.py: {log_loi_voices})")\n\n\n@app.get("/api/voices")\ndef danh_sach_giong(_: None = Depends(require_key)) -> dict:\n    """Danh sách giọng có sẵn trong thư viện."""\n    _can_voices()\n    ds = thu_vien_giong.quet()\n    return {\n        "dir": str(thu_vien_giong.thu_muc()),\n        "count": len(ds),\n        "voices": ds,\n        "cached": sorted(engine._prompts),\n    }\n\n\n@app.get("/api/voices/bundle")\ndef lay_goi_giong_mac_dinh(_: None = Depends(require_key)) -> dict:\n    """Trả về gói toàn bộ file giọng chuẩn dạng base64 để hỗ trợ tự động đồng bộ sang máy chủ."""\n    _can_voices()\n    import base64\n    d = thu_vien_giong.thu_muc()\n    res = {}\n    for p in d.glob("*.pt"):\n        try:\n            res[p.name] = base64.b64encode(p.read_bytes()).decode("ascii")\n        except Exception:\n            pass\n    return {"status": "ok", "count": len(res), "voices": res}\n\n\n@app.post("/api/voices/sync")\nasync def dong_bo_goi_giong(payload: dict, _: None = Depends(require_key)) -> dict:\n    """Nhận gói các giọng nạp lên và tự động giải nén lưu vào thư viện giọng."""\n    _can_voices()\n    import base64\n    v_dict = payload.get("voices", {})\n    count = 0\n    synced = []\n    for fname, b64_str in v_dict.items():\n        try:\n            data = base64.b64decode(b64_str)\n            e = thu_vien_giong.luu(fname, data)\n            engine.xoa_prompt(e["id"])\n            count += 1\n            synced.append(e["id"])\n        except Exception as ex:\n            log.warning("Lỗi khi đồng bộ giọng %s: %s", fname, ex)\n    log.info("Đã đồng bộ tự động %d giọng vào thư viện", count)\n    return {"status": "ok", "count": count, "synced": synced}\n\n\n@app.post("/api/voices")\nasync def them_giong(\n    file: UploadFile = File(...),\n    name: str = Form(""),\n    gender: str = Form(""),\n    note: str = Form(""),\n    _: None = Depends(require_key),\n) -> dict:\n    """Thêm một tệp .pt (VoiceClonePrompt) vào thư viện."""\n    _can_voices()\n    data = await file.read()\n    try:\n        e = thu_vien_giong.luu(file.filename or "giong.pt", data, name=name, gender=gender, note=note)\n    except ValueError as ex:\n        raise HTTPException(400, str(ex)) from ex\n    engine.xoa_prompt(e["id"])          # ghi đè thì phải nạp lại từ tệp mới\n    log.info("Thêm giọng \'%s\' (%d B)", e["id"], e["bytes"])\n    return {"ok": True, "voice": e}\n\n\n@app.post("/api/voices/save-clone")\ndef luu_giong_clone(\n    name: str = Form(...),\n    gender: str = Form(""),\n    note: str = Form(""),\n    _: None = Depends(require_key),\n) -> dict:\n    """Lưu giọng vừa nhân bản vào thư viện.\n\n    Sau mỗi lần gọi ``/api/clone`` thành công, server giữ lại prompt trong bộ nhớ.\n    Endpoint này lưu prompt đó ra tệp ``.pt`` trong thư viện giọng.\n    """\n    _can_voices()\n    prompt = engine._last_clone_prompt\n    if prompt is None:\n        if engine.mock:\n            # Chế độ giả lập: tạo tệp .pt giả (chứa dict đơn giản)\n            import io, struct, math\n            ten_tep = thu_vien_giong.lam_ten_tep(thu_vien_giong.lam_id(name) + ".pt")\n            # Sinh nội dung nhỏ giống torch.save (bắt đầu bằng PK header)\n            import zipfile\n            buf = io.BytesIO()\n            with zipfile.ZipFile(buf, "w") as zf:\n                zf.writestr("mock_voice/data.pkl", b"\\x80\\x02}q\\x00." + b"\\x00" * 2000)\n            data = buf.getvalue()\n            e = thu_vien_giong.luu(ten_tep, data, name=name, gender=gender, note=note)\n            log.info("Lưu giọng giả lập \'%s\'", e["id"])\n            return {"ok": True, "voice": e}\n        raise HTTPException(400, "Chưa có giọng nào để lưu — hãy nhân bản giọng trước (POST /api/clone)")\n\n    # Lưu prompt ra tệp .pt tạm rồi đọc bytes\n    import tempfile\n    fd, tmp = tempfile.mkstemp(suffix=".pt")\n    try:\n        prompt.save(tmp)\n        with open(tmp, "rb") as f:\n            data = f.read()\n    finally:\n        try:\n            import os as _os\n            _os.close(fd)\n            _os.unlink(tmp)\n        except OSError:\n            pass\n\n    ten_tep = thu_vien_giong.lam_ten_tep(thu_vien_giong.lam_id(name) + ".pt")\n    try:\n        e = thu_vien_giong.luu(ten_tep, data, name=name, gender=gender, note=note)\n    except ValueError as ex:\n        raise HTTPException(400, str(ex)) from ex\n\n    engine.xoa_prompt(e["id"])  # prompt cũ có thể khác bytes → nạp lại từ tệp mới\n    log.info("Lưu giọng clone \'%s\' (%d B)", e["id"], e["bytes"])\n    return {"ok": True, "voice": e}\n\n\n@app.delete("/api/voices/{voice_id}")\ndef xoa_giong(voice_id: str, _: None = Depends(require_key)) -> dict:\n    """Xoá một giọng khỏi thư viện."""\n    _can_voices()\n    try:\n        ten = thu_vien_giong.xoa(voice_id)\n    except FileNotFoundError as ex:\n        raise HTTPException(404, str(ex)) from ex\n    engine.xoa_prompt(voice_id)\n    log.info("Xoá giọng \'%s\' (%s)", voice_id, ten)\n    return {"ok": True, "deleted": voice_id}\n\n\n# ---------------------------------------------------------------------- helper\ndef _check_size(data: bytes) -> None:\n    mb = len(data) / (1024 * 1024)\n    if mb > MAX_UPLOAD_MB:\n        raise HTTPException(413, f"File quá lớn ({mb:.1f} MB > {MAX_UPLOAD_MB} MB)")\n    if len(data) < 1024:\n        raise HTTPException(400, "File audio quá nhỏ hoặc rỗng")\n\n\ndef _run(fn) -> Response:\n    t0 = time.perf_counter()\n    try:\n        res = fn()\n    except ValueError as e:\n        raise HTTPException(400, str(e)) from e\n    except FileNotFoundError as e:          # voice_id không có trong thư viện\n        raise HTTPException(404, str(e)) from e\n    except Exception as e:  # noqa: BLE001\n        log.exception("Lỗi khi tổng hợp")\n        raise HTTPException(500, f"{type(e).__name__}: {e}") from e\n\n    headers = {\n        "X-Gen-Time": str(res.gen_time),\n        "X-Audio-Duration": str(res.duration),\n        "X-RTF": f"{res.rtf:.3f}",\n        "X-Mode": res.mode,\n        "Content-Disposition": f\'inline; filename="{res.mode}.wav"\',\n    }\n    log.info(\n        "OK mode=%s text_len=? dur=%.2fs gen=%.2fs rtf=%.3f wall=%.2fs",\n        res.mode, res.duration, res.gen_time, res.rtf, time.perf_counter() - t0,\n    )\n    return Response(content=res.wav, media_type="audio/wav", headers=headers)\n\n\n@app.exception_handler(HTTPException)\nasync def _http_err(_request, exc: HTTPException) -> JSONResponse:\n    return JSONResponse(status_code=exc.status_code, content={"error": exc.detail})\n', encoding="utf-8")
print("✓ Đã ghi app.py")

# 4. Tự động khởi tạo thư viện với ĐẦY ĐỦ 7 GIỌNG MẪU TIẾNG VIỆT
VOICES_DIR = Path("/kaggle/working/voices")
VOICES_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_VOICES_B64 = {'Cao_Hung.pt': 'UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAQABIAT05HX0dJQS9kYXRhLnBrbEZCDgBaWlpaWlpaWlpaWlpaWoACfXEAKFgOAAAAZm9ybWF0X3ZlcnNpb25xAUsBWBAAAAByZWZfYXVkaW9fdG9rZW5zcQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKTG9uZ1N0b3JhZ2UKcQVYAQAAADBxBlgDAAAAY3B1cQdN8AN0cQhRSwBLCEt+hnEJS35LAYZxColjY29sbGVjdGlvbnMKT3JkZXJlZERpY3QKcQspUnEMdHENUnEOWAgAAAByZWZfdGV4dHEPWGwAAABEdXnDqm4ga2jDtG5nIMSR4bq5cCwgZGEgxJFlbiBz4bqhbSwgbeG6t3QgdGjDtCwgY2jDom4gdHLDoWkgdOG6rXQgdOG7qyBiw6kgbsOqbiBixrDhu5tjIHRo4bqlcCBixrDhu5tjIGNhby5xEFgHAAAAcmVmX3Jtc3ERRz/C7UHAAAAAdS5QSwcIOf7qOWYBAABmAQAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAXABUAT05HX0dJQS8uZm9ybWF0X3ZlcnNpb25GQhEAWlpaWlpaWlpaWlpaWlpaWloxUEsHCLfv3IMBAAAAAQAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAGgA3AE9OR19HSUEvLnN0b3JhZ2VfYWxpZ25tZW50RkIzAFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjY0UEsHCD93cekCAAAAAgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAEQA/AE9OR19HSUEvYnl0ZW9yZGVyRkI7AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpabGl0dGxlUEsHCIU94xkGAAAABgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAADgA+AE9OR19HSUEvZGF0YS8wRkI6AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlrLAQAAAAAAAAUAAAAAAAAALQMAAAAAAACGAgAAAAAAADkAAAAAAAAARwMAAAAAAAAUAAAAAAAAAMkBAAAAAAAAWgMAAAAAAACvAQAAAAAAAIICAAAAAAAAugAAAAAAAAA4AAAAAAAAAN8AAAAAAAAAJwEAAAAAAAArAAAAAAAAAEMDAAAAAAAAIAMAAAAAAADlAQAAAAAAAFcAAAAAAAAAUwAAAAAAAABjAwAAAAAAANgBAAAAAAAAnwEAAAAAAACfAQAAAAAAANMAAAAAAAAAkgAAAAAAAABXAQAAAAAAAAEBAAAAAAAAMwMAAAAAAACQAQAAAAAAAJIAAAAAAAAAiQIAAAAAAADNAwAAAAAAAGICAAAAAAAAKQMAAAAAAABhAwAAAAAAAMQAAAAAAAAAxAAAAAAAAAD7AQAAAAAAAOsCAAAAAAAANQIAAAAAAAAoAQAAAAAAAEICAAAAAAAAxgAAAAAAAADLAAAAAAAAADEAAAAAAAAA3gEAAAAAAAB1AgAAAAAAAAwDAAAAAAAAcQAAAAAAAAAOAQAAAAAAAPMAAAAAAAAA2AEAAAAAAADGAAAAAAAAAJ8CAAAAAAAABAEAAAAAAAD3AAAAAAAAAKEBAAAAAAAAtAMAAAAAAACZAgAAAAAAAHwBAAAAAAAAdgIAAAAAAADlAAAAAAAAAGcCAAAAAAAAGwEAAAAAAABnAgAAAAAAAH8AAAAAAAAA/wEAAAAAAAD/AQAAAAAAAPMAAAAAAAAAIQMAAAAAAAD/AQAAAAAAAIkCAAAAAAAABwEAAAAAAADJAQAAAAAAAAMBAAAAAAAAvgEAAAAAAAD1AwAAAAAAAAwCAAAAAAAA/wEAAAAAAADIAwAAAAAAAMgBAAAAAAAAnAIAAAAAAAACAAAAAAAAALUCAAAAAAAA/wEAAAAAAABGAgAAAAAAACwCAAAAAAAAzwMAAAAAAACwAQAAAAAAAL0CAAAAAAAAOgIAAAAAAACcAgAAAAAAAJ0AAAAAAAAANgMAAAAAAADUAQAAAAAAAHUDAAAAAAAAdQMAAAAAAAAeAAAAAAAAAEgBAAAAAAAAEAMAAAAAAACiAQAAAAAAALoAAAAAAAAARgIAAAAAAAAeAAAAAAAAAIYCAAAAAAAA+wEAAAAAAAArAQAAAAAAALsBAAAAAAAA6gEAAAAAAADDAgAAAAAAAPUDAAAAAAAAGwEAAAAAAAAnAwAAAAAAACcDAAAAAAAAxQMAAAAAAACiAQAAAAAAAHQCAAAAAAAA/wEAAAAAAAABAwAAAAAAALcDAAAAAAAACgAAAAAAAACwAQAAAAAAANkDAAAAAAAAYgIAAAAAAACCAwAAAAAAAJcBAAAAAAAAUAIAAAAAAABCAAAAAAAAABUDAAAAAAAAtAEAAAAAAACuAAAAAAAAAOEBAAAAAAAApwMAAAAAAAD1AgAAAAAAAAcAAAAAAAAAOgIAAAAAAABpAQAAAAAAAAMCAAAAAAAAKQMAAAAAAADYAQAAAAAAAFEDAAAAAAAAnwIAAAAAAAAqAwAAAAAAAKcCAAAAAAAA4wIAAAAAAAC+AAAAAAAAALABAAAAAAAAYAIAAAAAAADTAgAAAAAAAOQDAAAAAAAAygEAAAAAAABbAwAAAAAAAFQDAAAAAAAADAMAAAAAAABBAQAAAAAAAHwDAAAAAAAAqwAAAAAAAABrAAAAAAAAAG0CAAAAAAAA0QIAAAAAAACQAQAAAAAAABEDAAAAAAAAtAMAAAAAAAA2AgAAAAAAAIEBAAAAAAAAGQAAAAAAAAAXAwAAAAAAAO4DAAAAAAAAsQAAAAAAAACVAgAAAAAAAHoAAAAAAAAA1AMAAAAAAACvAgAAAAAAAGoDAAAAAAAAtwMAAAAAAABKAwAAAAAAACICAAAAAAAA0wEAAAAAAABMAwAAAAAAAGgCAAAAAAAAOwMAAAAAAAAHAAAAAAAAANkDAAAAAAAAQwMAAAAAAABMAwAAAAAAAIYCAAAAAAAAhAEAAAAAAAD4AgAAAAAAAHQDAAAAAAAAmgMAAAAAAADyAwAAAAAAAIwDAAAAAAAAeAMAAAAAAACSAwAAAAAAAP4DAAAAAAAAWwAAAAAAAAA6AQAAAAAAAJYDAAAAAAAAAwIAAAAAAACoAwAAAAAAAAoAAAAAAAAApwEAAAAAAAALAgAAAAAAALQBAAAAAAAAswIAAAAAAADMAgAAAAAAAAcBAAAAAAAACQEAAAAAAACKAAAAAAAAAAsDAAAAAAAA1wMAAAAAAAApAgAAAAAAAGMAAAAAAAAAQgAAAAAAAADhAQAAAAAAAPkDAAAAAAAAWwAAAAAAAAAmAgAAAAAAAJkAAAAAAAAAKQAAAAAAAAC2AQAAAAAAALUCAAAAAAAAEAAAAAAAAADRAQAAAAAAAAcDAAAAAAAAowEAAAAAAAC0AQAAAAAAAD4CAAAAAAAA9QIAAAAAAABxAQAAAAAAAKgDAAAAAAAA5gIAAAAAAADhAAAAAAAAAPUBAAAAAAAAVgMAAAAAAADmAQAAAAAAAEADAAAAAAAAMQIAAAAAAAByAwAAAAAAANAAAAAAAAAAUAMAAAAAAAAxAgAAAAAAAHMDAAAAAAAAJAMAAAAAAAA2AwAAAAAAAPwBAAAAAAAAiwMAAAAAAADfAAAAAAAAANcCAAAAAAAA9wIAAAAAAABYAwAAAAAAAG4DAAAAAAAArQEAAAAAAAAHAAAAAAAAAOYBAAAAAAAAZwEAAAAAAAAoAwAAAAAAAM8DAAAAAAAAYQAAAAAAAABCAwAAAAAAAC8DAAAAAAAASwAAAAAAAAC6AQAAAAAAAOcDAAAAAAAAWwEAAAAAAAAlAQAAAAAAAKYBAAAAAAAAAwAAAAAAAABnAwAAAAAAAFoBAAAAAAAAiQIAAAAAAACpAAAAAAAAAKkAAAAAAAAAqwEAAAAAAADgAQAAAAAAAC4CAAAAAAAAYAAAAAAAAABzAAAAAAAAAIcBAAAAAAAAmAMAAAAAAAC4AQAAAAAAAIQAAAAAAAAA+QEAAAAAAABwAQAAAAAAAKwCAAAAAAAATgAAAAAAAACzAgAAAAAAAB8DAAAAAAAAmgAAAAAAAABiAQAAAAAAABUCAAAAAAAAjgIAAAAAAAD3AgAAAAAAADEDAAAAAAAAFAAAAAAAAACbAQAAAAAAAPkDAAAAAAAAqgMAAAAAAAAtAQAAAAAAANUBAAAAAAAA9AIAAAAAAAAUAQAAAAAAAGgCAAAAAAAATwIAAAAAAABGAAAAAAAAAHsCAAAAAAAAGgAAAAAAAAAcAAAAAAAAAJECAAAAAAAASQEAAAAAAAC4AAAAAAAAAOkAAAAAAAAARwIAAAAAAACqAQAAAAAAACMAAAAAAAAAJAAAAAAAAAAPAgAAAAAAADIDAAAAAAAA7QMAAAAAAAAWAQAAAAAAAD0DAAAAAAAAtQEAAAAAAAB4AwAAAAAAAI4BAAAAAAAAEgAAAAAAAAB4AwAAAAAAAKoBAAAAAAAAQgMAAAAAAAByAgAAAAAAAJABAAAAAAAAFQEAAAAAAABnAgAAAAAAAMQBAAAAAAAAUgAAAAAAAAA7AwAAAAAAADQDAAAAAAAAeAMAAAAAAADIAAAAAAAAANgDAAAAAAAAeAMAAAAAAABxAgAAAAAAAPgAAAAAAAAApQMAAAAAAAC6AQAAAAAAAB4CAAAAAAAARAAAAAAAAACjAwAAAAAAAM8DAAAAAAAA6QMAAAAAAADfAQAAAAAAAHACAAAAAAAApgMAAAAAAAAeAAAAAAAAAHgDAAAAAAAArwAAAAAAAAC6AQAAAAAAAIoDAAAAAAAAQAAAAAAAAAByAgAAAAAAALsBAAAAAAAA6gIAAAAAAACFAQAAAAAAAGUBAAAAAAAAmgAAAAAAAABqAwAAAAAAAMEDAAAAAAAA1wAAAAAAAAAaAwAAAAAAAD8BAAAAAAAAmAEAAAAAAAAxAQAAAAAAALYCAAAAAAAATAEAAAAAAAD9AwAAAAAAAE4AAAAAAAAAgAAAAAAAAAB5AgAAAAAAALECAAAAAAAALgMAAAAAAADhAQAAAAAAAC8BAAAAAAAAtQEAAAAAAADmAwAAAAAAAK8AAAAAAAAARwIAAAAAAADsAwAAAAAAAG0AAAAAAAAAJwEAAAAAAABgAwAAAAAAADcDAAAAAAAAsAMAAAAAAAA2AgAAAAAAAIoAAAAAAAAA1wAAAAAAAAA2AQAAAAAAAMgCAAAAAAAAAgIAAAAAAACoAQAAAAAAAJsDAAAAAAAA3QEAAAAAAABUAgAAAAAAAJsBAAAAAAAAfwAAAAAAAAAkAQAAAAAAAHgDAAAAAAAADAEAAAAAAADMAAAAAAAAAB4BAAAAAAAASgMAAAAAAABNAAAAAAAAAIsDAAAAAAAA3QMAAAAAAAD0AwAAAAAAAG0AAAAAAAAAQwEAAAAAAAArAwAAAAAAAH0BAAAAAAAAPAAAAAAAAAB5AQAAAAAAAO4AAAAAAAAAxwIAAAAAAACOAQAAAAAAACIDAAAAAAAAZAAAAAAAAAAiAQAAAAAAAJICAAAAAAAAaQEAAAAAAABHAAAAAAAAAKUCAAAAAAAAYQEAAAAAAAAtAAAAAAAAADIAAAAAAAAAKQMAAAAAAAAHAQAAAAAAAIMAAAAAAAAAUwMAAAAAAABGAQAAAAAAAMoDAAAAAAAAOwAAAAAAAACDAAAAAAAAAP0BAAAAAAAAQAIAAAAAAACpAgAAAAAAAO4AAAAAAAAAKwIAAAAAAAAPAAAAAAAAAGkDAAAAAAAAIQMAAAAAAAA1AgAAAAAAAMUAAAAAAAAAxgMAAAAAAACAAgAAAAAAANQAAAAAAAAATgIAAAAAAABUAQAAAAAAAEkBAAAAAAAAcgIAAAAAAAAKAAAAAAAAAIkAAAAAAAAA/gMAAAAAAADIAQAAAAAAAMcCAAAAAAAAVwAAAAAAAAD/AwAAAAAAAL8CAAAAAAAA6wAAAAAAAABAAAAAAAAAAKUCAAAAAAAATQEAAAAAAAC6AQAAAAAAAEMBAAAAAAAA3gIAAAAAAABoAwAAAAAAAAkAAAAAAAAASgMAAAAAAADKAQAAAAAAAMQAAAAAAAAAXgMAAAAAAABPAwAAAAAAAEsCAAAAAAAAdwMAAAAAAACDAgAAAAAAAIUCAAAAAAAAGgEAAAAAAAATAgAAAAAAAMECAAAAAAAAvQMAAAAAAABVAQAAAAAAAIYBAAAAAAAAjwAAAAAAAABPAwAAAAAAAE0DAAAAAAAADAEAAAAAAACZAgAAAAAAAK8AAAAAAAAA9QIAAAAAAACLAwAAAAAAAJEBAAAAAAAASgIAAAAAAAAZAQAAAAAAAC0DAAAAAAAAGAIAAAAAAACMAQAAAAAAALwDAAAAAAAAVAAAAAAAAAASAwAAAAAAANQDAAAAAAAAnAMAAAAAAABFAgAAAAAAAHcBAAAAAAAAMAIAAAAAAABuAAAAAAAAADcDAAAAAAAAbgAAAAAAAADYAQAAAAAAACwDAAAAAAAAowMAAAAAAAAMAQAAAAAAAOoDAAAAAAAAEAEAAAAAAACmAAAAAAAAAMoAAAAAAAAApAIAAAAAAABCAgAAAAAAAF8BAAAAAAAAfQMAAAAAAAD9AQAAAAAAAIUDAAAAAAAAugAAAAAAAADLAAAAAAAAACEAAAAAAAAA0wEAAAAAAAAPAgAAAAAAAEwBAAAAAAAAWQIAAAAAAADWAwAAAAAAANcCAAAAAAAA6wIAAAAAAAB1AwAAAAAAAHcDAAAAAAAAiwAAAAAAAAB1AQAAAAAAAK8CAAAAAAAAaQAAAAAAAAD8AAAAAAAAAJYCAAAAAAAAcAAAAAAAAAC9AgAAAAAAAC0CAAAAAAAArAIAAAAAAAAgAwAAAAAAAHEAAAAAAAAA0AEAAAAAAACbAwAAAAAAAMUAAAAAAAAAngEAAAAAAACTAwAAAAAAAMkCAAAAAAAAYAAAAAAAAADOAgAAAAAAACUAAAAAAAAAuwEAAAAAAAAiAAAAAAAAAD0BAAAAAAAAoAAAAAAAAAB9AwAAAAAAAG0CAAAAAAAAAgAAAAAAAAAmAQAAAAAAAEQBAAAAAAAA8QAAAAAAAADTAwAAAAAAAIEBAAAAAAAARQAAAAAAAACaAAAAAAAAADICAAAAAAAAwwEAAAAAAABqAAAAAAAAAEYDAAAAAAAAnQAAAAAAAAAnAAAAAAAAAPIDAAAAAAAAfAIAAAAAAACOAwAAAAAAAOkAAAAAAAAAsAEAAAAAAAA0AgAAAAAAAAkBAAAAAAAAoAAAAAAAAAByAgAAAAAAABkCAAAAAAAADgMAAAAAAABGAQAAAAAAAKkCAAAAAAAAHwEAAAAAAADiAwAAAAAAAGwAAAAAAAAA4gMAAAAAAAA8AAAAAAAAAMUDAAAAAAAAdQMAAAAAAACHAwAAAAAAANICAAAAAAAAtAAAAAAAAABBAAAAAAAAADwCAAAAAAAAhwMAAAAAAAC1AAAAAAAAAJICAAAAAAAAAwAAAAAAAABIAwAAAAAAAGAAAAAAAAAAzgAAAAAAAACnAwAAAAAAAAUAAAAAAAAA7QMAAAAAAABKAQAAAAAAAAECAAAAAAAAVAMAAAAAAADFAAAAAAAAAA0AAAAAAAAAGwAAAAAAAAD9AgAAAAAAAJoBAAAAAAAAdQEAAAAAAAB4AwAAAAAAAFwAAAAAAAAAEwAAAAAAAADxAAAAAAAAAHYAAAAAAAAApgMAAAAAAACxAAAAAAAAAL8AAAAAAAAA0wMAAAAAAAAJAAAAAAAAAC8CAAAAAAAAagEAAAAAAAAQAgAAAAAAAGUCAAAAAAAAnAIAAAAAAACxAQAAAAAAAPEAAAAAAAAAwAMAAAAAAADUAgAAAAAAAGgBAAAAAAAAKQMAAAAAAAAXAwAAAAAAAEUDAAAAAAAAFQIAAAAAAADzAgAAAAAAAD0DAAAAAAAAqAIAAAAAAADXAQAAAAAAAM0CAAAAAAAAeAAAAAAAAAA2AAAAAAAAACUAAAAAAAAAyQEAAAAAAAAyAgAAAAAAAPcDAAAAAAAA9gEAAAAAAAAYAgAAAAAAAB8DAAAAAAAApwEAAAAAAADpAAAAAAAAAC4DAAAAAAAAxQIAAAAAAAAiAAAAAAAAAH0AAAAAAAAAYQEAAAAAAADSAAAAAAAAAM4DAAAAAAAALAMAAAAAAAADAwAAAAAAAEEAAAAAAAAArQIAAAAAAABiAQAAAAAAAEQDAAAAAAAAMgIAAAAAAAD2AwAAAAAAAJ0BAAAAAAAAcQIAAAAAAACiAAAAAAAAACIAAAAAAAAAwQMAAAAAAADSAwAAAAAAAE4CAAAAAAAAngMAAAAAAACbAQAAAAAAAJcCAAAAAAAAqQIAAAAAAAA9AwAAAAAAAPUDAAAAAAAAngEAAAAAAACIAwAAAAAAAFQCAAAAAAAALwEAAAAAAADIAAAAAAAAAKoAAAAAAAAAdQAAAAAAAABSAgAAAAAAAFgBAAAAAAAA4AAAAAAAAAAZAAAAAAAAAGkAAAAAAAAAZQAAAAAAAADrAQAAAAAAAGMCAAAAAAAAsQAAAAAAAADpAAAAAAAAAMQDAAAAAAAAIgIAAAAAAABnAAAAAAAAAHcBAAAAAAAAxgAAAAAAAACrAQAAAAAAAE8CAAAAAAAANQMAAAAAAABsAwAAAAAAAOAAAAAAAAAAwgIAAAAAAAD9AQAAAAAAABoCAAAAAAAAWQEAAAAAAACgAwAAAAAAAIYAAAAAAAAATgEAAAAAAACiAAAAAAAAAIkAAAAAAAAAtgIAAAAAAABvAgAAAAAAABEDAAAAAAAARQIAAAAAAADWAwAAAAAAAPIAAAAAAAAA9wIAAAAAAAA6AgAAAAAAAJMCAAAAAAAAcgEAAAAAAAB2AQAAAAAAAJwCAAAAAAAA5gAAAAAAAABZAwAAAAAAAAABAAAAAAAAUAIAAAAAAADkAQAAAAAAAAoAAAAAAAAAaQIAAAAAAADXAwAAAAAAAMoBAAAAAAAA4AAAAAAAAAAUAQAAAAAAADcBAAAAAAAAYwAAAAAAAAANAwAAAAAAAGkAAAAAAAAAawIAAAAAAAD4AwAAAAAAAB8BAAAAAAAAxAEAAAAAAAC1AQAAAAAAAFwCAAAAAAAA1AIAAAAAAABDAQAAAAAAAGcDAAAAAAAAbwEAAAAAAACOAgAAAAAAAJwDAAAAAAAAFgEAAAAAAAB7AAAAAAAAAMYAAAAAAAAACQMAAAAAAADcAQAAAAAAACEDAAAAAAAA2gEAAAAAAADcAgAAAAAAAIsCAAAAAAAA7AAAAAAAAADaAwAAAAAAAE8DAAAAAAAAqQMAAAAAAADTAwAAAAAAACABAAAAAAAAwgMAAAAAAACwAwAAAAAAAI8CAAAAAAAAIgIAAAAAAABAAgAAAAAAAHMBAAAAAAAAfQAAAAAAAADJAQAAAAAAANwDAAAAAAAAgAIAAAAAAABkAwAAAAAAACsAAAAAAAAA8wAAAAAAAAAbAQAAAAAAAK8BAAAAAAAAYgEAAAAAAADHAQAAAAAAALoCAAAAAAAA9wEAAAAAAACSAwAAAAAAAOUAAAAAAAAA1wMAAAAAAAAqAgAAAAAAAPUCAAAAAAAAAQEAAAAAAACBAgAAAAAAAKoAAAAAAAAA4wIAAAAAAABOAAAAAAAAAKsBAAAAAAAAcgEAAAAAAAAhAAAAAAAAAO4CAAAAAAAAnAIAAAAAAAD5AgAAAAAAAFYCAAAAAAAAbAEAAAAAAAD7AwAAAAAAAHACAAAAAAAA8AMAAAAAAAAuAAAAAAAAAKsBAAAAAAAAewEAAAAAAAB/AwAAAAAAAMMAAAAAAAAASgMAAAAAAAC3AgAAAAAAAIECAAAAAAAA/AEAAAAAAAAAAAAAAAAAADgBAAAAAAAAFAAAAAAAAABnAAAAAAAAALkAAAAAAAAAjAMAAAAAAADUAwAAAAAAAMsBAAAAAAAAvgMAAAAAAADAAAAAAAAAAEADAAAAAAAA0wMAAAAAAACjAwAAAAAAAJwCAAAAAAAAlgAAAAAAAAAuAAAAAAAAABUBAAAAAAAA5wIAAAAAAABNAwAAAAAAAOMBAAAAAAAAcgIAAAAAAABSAQAAAAAAACAAAAAAAAAA0AAAAAAAAAABAwAAAAAAADUCAAAAAAAA3wIAAAAAAAAKAAAAAAAAAKwCAAAAAAAAXwMAAAAAAAA2AQAAAAAAAAsDAAAAAAAA+wIAAAAAAABlAQAAAAAAAPsAAAAAAAAAOwMAAAAAAACDAgAAAAAAAAICAAAAAAAA0AMAAAAAAAC+AgAAAAAAALACAAAAAAAAdwIAAAAAAAAcAwAAAAAAAM4AAAAAAAAAHwEAAAAAAABIAAAAAAAAAMUCAAAAAAAAkwIAAAAAAAB3AAAAAAAAAHEBAAAAAAAAVwEAAAAAAAAmAAAAAAAAABgCAAAAAAAAJQEAAAAAAACbAQAAAAAAAAcAAAAAAAAAZwMAAAAAAABNAAAAAAAAAK0BAAAAAAAAOgAAAAAAAADMAAAAAAAAAA8CAAAAAAAAlQMAAAAAAAAnAQAAAAAAAHMBAAAAAAAAKgEAAAAAAADvAwAAAAAAAIQAAAAAAAAA1QIAAAAAAADOAAAAAAAAAIQBAAAAAAAAhgEAAAAAAADOAwAAAAAAAI4AAAAAAAAAowIAAAAAAACGAgAAAAAAAB8DAAAAAAAAuAEAAAAAAABGAAAAAAAAACICAAAAAAAAEgIAAAAAAAAmAgAAAAAAACECAAAAAAAAfQIAAAAAAAAdAgAAAAAAACsAAAAAAAAAcAAAAAAAAADDAgAAAAAAAGECAAAAAAAACgAAAAAAAAD1AQAAAAAAAKQCAAAAAAAAeQIAAAAAAAC0AgAAAAAAANsCAAAAAAAAzQIAAAAAAAAIAQAAAAAAABcCAAAAAAAA2gEAAAAAAAAnAQAAAAAAACgBAAAAAAAAVwEAAAAAAAD7AwAAAAAAAC4AAAAAAAAA3gIAAAAAAADsAgAAAAAAAIACAAAAAAAAlQAAAAAAAADVAgAAAAAAAC8AAAAAAAAAGQMAAAAAAABgAwAAAAAAAIYBAAAAAAAAhgMAAAAAAAACAQAAAAAAAP4CAAAAAAAAPwIAAAAAAAA3AAAAAAAAAJYBAAAAAAAApwEAAAAAAABAAQAAAAAAAC8CAAAAAAAA5gEAAAAAAABIAgAAAAAAAF0DAAAAAAAAkgMAAAAAAAD7AwAAAAAAAB4AAAAAAAAASwMAAAAAAADIAQAAAAAAACAAAAAAAAAANgIAAAAAAAAtAwAAAAAAAFQDAAAAAAAAVQIAAAAAAACdAwAAAAAAANQCAAAAAAAAgQIAAAAAAADAAwAAAAAAAH4CAAAAAAAAPwAAAAAAAAD+AgAAAAAAAKkCAAAAAAAAawMAAAAAAABnAAAAAAAAAEYBAAAAAAAAxwAAAAAAAAAXAgAAAAAAAE0AAAAAAAAAIgAAAAAAAAAIAQAAAAAAAPgCAAAAAAAAFwIAAAAAAAAsAAAAAAAAABsAAAAAAAAA8QEAAAAAAAAGAQAAAAAAACABAAAAAAAABwAAAAAAAAALAgAAAAAAAOUCAAAAAAAAAwAAAAAAAAB/AwAAAAAAAKQCAAAAAAAADQMAAAAAAADPAwAAAAAAAM0CAAAAAAAAvgAAAAAAAAB9AgAAAAAAAFkDAAAAAAAAsQIAAAAAAADAAwAAAAAAACQBAAAAAAAAkwMAAAAAAABQSwcIoJ9PZoAfAACAHwAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAPAEMAT05HX0dJQS92ZXJzaW9uRkI/AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjMKUEsHCNGeZ1UCAAAAAgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAHgAyAE9OR19HSUEvLmRhdGEvc2VyaWFsaXphdGlvbl9pZEZCLgBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMTU3MjgxOTk1MTE4NTQ2NDMwMjYwMzk4MTcwMjUwOTE1MjI5NTcwM1BLBwiXLhz1KAAAACgAAABQSwECAAAAAAgIAAAAAAAAOf7qOWYBAABmAQAAEAAAAAAAAAAAAAAAAAAAAAAAT05HX0dJQS9kYXRhLnBrbFBLAQIAAAAACAgAAAAAAAC379yDAQAAAAEAAAAXAAAAAAAAAAAAAAAAALYBAABPTkdfR0lBLy5mb3JtYXRfdmVyc2lvblBLAQIAAAAACAgAAAAAAAA/d3HpAgAAAAIAAAAaAAAAAAAAAAAAAAAAABECAABPTkdfR0lBLy5zdG9yYWdlX2FsaWdubWVudFBLAQIAAAAACAgAAAAAAACFPeMZBgAAAAYAAAARAAAAAAAAAAAAAAAAAJICAABPTkdfR0lBL2J5dGVvcmRlclBLAQIAAAAACAgAAAAAAACgn09mgB8AAIAfAAAOAAAAAAAAAAAAAAAAABYDAABPTkdfR0lBL2RhdGEvMFBLAQIAAAAACAgAAAAAAADRnmdVAgAAAAIAAAAPAAAAAAAAAAAAAAAAABAjAABPTkdfR0lBL3ZlcnNpb25QSwECAAAAAAgIAAAAAAAAly4c9SgAAAAoAAAAHgAAAAAAAAAAAAAAAACSIwAAT05HX0dJQS8uZGF0YS9zZXJpYWxpemF0aW9uX2lkUEsGBiwAAAAAAAAAHgMtAAAAAAAAAAAABwAAAAAAAAAHAAAAAAAAAM8BAAAAAAAAOCQAAAAAAABQSwYHAAAAAAcmAAAAAAAAAQAAAFBLBQYAAAAABwAHAM8BAAA4JAAAAAA=', 'Linh_Duc_Tu.pt': 'UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAPABMATlVfVFJFL2RhdGEucGtsRkIPAFpaWlpaWlpaWlpaWlpaWoACfXEAKFgOAAAAZm9ybWF0X3ZlcnNpb25xAUsBWBAAAAByZWZfYXVkaW9fdG9rZW5zcQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKTG9uZ1N0b3JhZ2UKcQVYAQAAADBxBlgDAAAAY3B1cQdNIAJ0cQhRSwBLCEtEhnEJS0RLAYZxColjY29sbGVjdGlvbnMKT3JkZXJlZERpY3QKcQspUnEMdHENUnEOWAgAAAByZWZfdGV4dHEPWDUAAABOZ2hlIHRoaeG6v3AgbeG7mXQgbOG6p24gdGjDtGksIGNobyBjw7Qg4bqleSDEkWkgxJFpLnEQWAcAAAByZWZfcm1zcRFHP84kj2AAAAB1LlBLBwjsWMSRLwEAAC8BAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABYADQBOVV9UUkUvLmZvcm1hdF92ZXJzaW9uRkIJAFpaWlpaWlpaWjFQSwcIt+/cgwEAAAABAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAZADgATlVfVFJFLy5zdG9yYWdlX2FsaWdubWVudEZCNABaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaNjRQSwcIP3dx6QIAAAACAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAQAEAATlVfVFJFL2J5dGVvcmRlckZCPABaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpsaXR0bGVQSwcIhT3jGQYAAAAGAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAANAD8ATlVfVFJFL2RhdGEvMEZCOwBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWm4DAAAAAAAAoAIAAAAAAACnAgAAAAAAAIUBAAAAAAAAzgEAAAAAAAAwAgAAAAAAAIICAAAAAAAAFAEAAAAAAABmAAAAAAAAAMsBAAAAAAAAiQIAAAAAAAAWAgAAAAAAAKcCAAAAAAAA0QAAAAAAAAC7AQAAAAAAAIQCAAAAAAAAogEAAAAAAAAMAgAAAAAAAKIBAAAAAAAAoAIAAAAAAAD1AwAAAAAAAPUDAAAAAAAAdQMAAAAAAADmAQAAAAAAAJIAAAAAAAAA2gIAAAAAAAAdAAAAAAAAAKcCAAAAAAAAuwMAAAAAAAARAgAAAAAAAGoAAAAAAAAAEAMAAAAAAAAMAgAAAAAAAFMBAAAAAAAABgMAAAAAAACFAQAAAAAAAIQCAAAAAAAA9QAAAAAAAADaAgAAAAAAAHsDAAAAAAAAAgIAAAAAAACmAgAAAAAAAM8AAAAAAAAA8gAAAAAAAACFAQAAAAAAACwAAAAAAAAAvgAAAAAAAABpAwAAAAAAAIkCAAAAAAAAmwIAAAAAAACeAgAAAAAAAGUBAAAAAAAAIQMAAAAAAADyAgAAAAAAAIUAAAAAAAAAawEAAAAAAACqAgAAAAAAAB4BAAAAAAAAngIAAAAAAACxAwAAAAAAALEDAAAAAAAAHgAAAAAAAACCAgAAAAAAAOMDAAAAAAAA4wMAAAAAAADjAwAAAAAAALEDAAAAAAAAvQEAAAAAAACIAwAAAAAAALoAAAAAAAAAawAAAAAAAAB9AwAAAAAAAOQDAAAAAAAAkwEAAAAAAABSAwAAAAAAAAwDAAAAAAAAZAIAAAAAAABJAwAAAAAAAC4BAAAAAAAARwIAAAAAAACWAwAAAAAAADoBAAAAAAAA0gEAAAAAAADEAQAAAAAAAGsDAAAAAAAAtAEAAAAAAAC4AQAAAAAAAEYCAAAAAAAAHAEAAAAAAACwAQAAAAAAAFsDAAAAAAAAkAMAAAAAAAA3AgAAAAAAAIEBAAAAAAAAXAIAAAAAAABjAgAAAAAAANcCAAAAAAAA2QMAAAAAAAAJAQAAAAAAADoBAAAAAAAAqwAAAAAAAAAyAwAAAAAAAN4AAAAAAAAA0QAAAAAAAAAVAgAAAAAAABIDAAAAAAAAxgIAAAAAAADAAQAAAAAAAEwCAAAAAAAAQwAAAAAAAADWAwAAAAAAAE8DAAAAAAAAAQAAAAAAAABZAQAAAAAAAJADAAAAAAAATgEAAAAAAABQAwAAAAAAAGsAAAAAAAAAVgMAAAAAAADSAQAAAAAAAL0AAAAAAAAAIQMAAAAAAABBAQAAAAAAAHMAAAAAAAAAtAEAAAAAAACiAgAAAAAAACkCAAAAAAAAHAEAAAAAAADYAQAAAAAAADoBAAAAAAAA9QIAAAAAAADjAgAAAAAAADoBAAAAAAAAOgEAAAAAAABKAwAAAAAAAFwAAAAAAAAAUwIAAAAAAABuAwAAAAAAAN8AAAAAAAAAgQMAAAAAAACEAwAAAAAAADgAAAAAAAAAjgEAAAAAAAAdAwAAAAAAAJYAAAAAAAAA2gEAAAAAAAD+AAAAAAAAAE4AAAAAAAAArwMAAAAAAABEAwAAAAAAAB8AAAAAAAAAvwMAAAAAAABsAQAAAAAAADoBAAAAAAAANAIAAAAAAADqAwAAAAAAAEoCAAAAAAAAKAIAAAAAAABOAgAAAAAAAGcAAAAAAAAAOQMAAAAAAAAeAQAAAAAAAFYDAAAAAAAAiwAAAAAAAABnAQAAAAAAAOYAAAAAAAAALwMAAAAAAAAbAAAAAAAAAHMAAAAAAAAAsAAAAAAAAABXAwAAAAAAAHwDAAAAAAAAKwIAAAAAAACSAQAAAAAAAOUBAAAAAAAAMgMAAAAAAAAPAQAAAAAAANsDAAAAAAAAuwIAAAAAAAAbAQAAAAAAAFYBAAAAAAAAgQAAAAAAAABWAwAAAAAAABgBAAAAAAAA0QAAAAAAAACOAQAAAAAAAEQAAAAAAAAAOAEAAAAAAAAKAgAAAAAAAGgBAAAAAAAATwEAAAAAAADvAAAAAAAAADMBAAAAAAAAegAAAAAAAADZAwAAAAAAAIcCAAAAAAAA/gAAAAAAAABuAwAAAAAAADgBAAAAAAAAOgEAAAAAAACZAQAAAAAAAP4AAAAAAAAAJgAAAAAAAAChAwAAAAAAADsBAAAAAAAAWwAAAAAAAACPAgAAAAAAAKkAAAAAAAAAjAMAAAAAAABrAQAAAAAAAK4AAAAAAAAAPQAAAAAAAACqAwAAAAAAALYDAAAAAAAAyAEAAAAAAAA/AwAAAAAAAGQCAAAAAAAAJgEAAAAAAACBAgAAAAAAAJ8DAAAAAAAAlwMAAAAAAABuAwAAAAAAAKoCAAAAAAAAggIAAAAAAADQAAAAAAAAACwAAAAAAAAADgAAAAAAAAAwAQAAAAAAABUAAAAAAAAAhAMAAAAAAAB2AgAAAAAAANQCAAAAAAAA3QAAAAAAAABSAwAAAAAAALICAAAAAAAABgIAAAAAAADqAAAAAAAAAPQDAAAAAAAAgAIAAAAAAACbAwAAAAAAABgCAAAAAAAAVwEAAAAAAAAAAwAAAAAAABUCAAAAAAAAXQIAAAAAAADeAAAAAAAAACEBAAAAAAAAyQIAAAAAAABDAQAAAAAAAJ8DAAAAAAAAyQIAAAAAAACAAgAAAAAAAB4AAAAAAAAAZgEAAAAAAABWAwAAAAAAAJkCAAAAAAAAaQIAAAAAAABSAQAAAAAAANICAAAAAAAAAgAAAAAAAABVAAAAAAAAAPQAAAAAAAAAVwMAAAAAAABeAQAAAAAAADgAAAAAAAAA6AEAAAAAAACnAgAAAAAAAH4BAAAAAAAA1AEAAAAAAABBAgAAAAAAAD0CAAAAAAAAVwAAAAAAAAA7AgAAAAAAAPEAAAAAAAAA6wMAAAAAAAA0AwAAAAAAADYDAAAAAAAAGQAAAAAAAAC6AgAAAAAAADAAAAAAAAAAmAIAAAAAAACGAQAAAAAAAN8DAAAAAAAA7wMAAAAAAADMAQAAAAAAAMsAAAAAAAAAgwMAAAAAAAAlAAAAAAAAADECAAAAAAAAPgEAAAAAAADHAgAAAAAAAFAAAAAAAAAAUAAAAAAAAAAfAQAAAAAAAMUDAAAAAAAAvwAAAAAAAABxAAAAAAAAADsBAAAAAAAADgIAAAAAAADGAwAAAAAAAFEDAAAAAAAAowIAAAAAAAC+AgAAAAAAAOADAAAAAAAABAEAAAAAAAC+AAAAAAAAAAYBAAAAAAAATwIAAAAAAABVAAAAAAAAAJgBAAAAAAAAkQIAAAAAAACmAAAAAAAAAKQAAAAAAAAALAIAAAAAAAB8AwAAAAAAAP0DAAAAAAAAngAAAAAAAAAuAAAAAAAAAGgAAAAAAAAARwEAAAAAAAAeAAAAAAAAAFQDAAAAAAAA4gAAAAAAAACCAgAAAAAAANcCAAAAAAAADAMAAAAAAAB0AAAAAAAAABcBAAAAAAAANgIAAAAAAABFAwAAAAAAANkDAAAAAAAASAEAAAAAAAClAAAAAAAAANYDAAAAAAAArwAAAAAAAAC1AAAAAAAAALUAAAAAAAAA/gMAAAAAAACvAQAAAAAAANYAAAAAAAAA5gAAAAAAAAAMAwAAAAAAAOYBAAAAAAAAzwAAAAAAAACuAAAAAAAAADIBAAAAAAAAaAMAAAAAAAAAAQAAAAAAAG0AAAAAAAAAlwIAAAAAAADnAwAAAAAAANUCAAAAAAAAMAIAAAAAAACqAAAAAAAAAFgDAAAAAAAA7QIAAAAAAAA3AAAAAAAAACABAAAAAAAAewMAAAAAAACOAgAAAAAAAJ8CAAAAAAAA2AEAAAAAAAC4AAAAAAAAANcCAAAAAAAAFQIAAAAAAABSAQAAAAAAAA4DAAAAAAAAzAEAAAAAAAAgAQAAAAAAAI0DAAAAAAAA1gEAAAAAAAA2AQAAAAAAACUDAAAAAAAAYwEAAAAAAABRAQAAAAAAAJ8CAAAAAAAAtQIAAAAAAAA/AgAAAAAAAOMCAAAAAAAA5AAAAAAAAAABAgAAAAAAAP0DAAAAAAAAvQEAAAAAAAANAwAAAAAAAP4AAAAAAAAAkgAAAAAAAADwAgAAAAAAAD0CAAAAAAAA2wMAAAAAAACWAQAAAAAAAHMAAAAAAAAA5wIAAAAAAACfAAAAAAAAACkCAAAAAAAAnAMAAAAAAACZAgAAAAAAABMDAAAAAAAAwAMAAAAAAACcAgAAAAAAADEDAAAAAAAAcgIAAAAAAAAdAAAAAAAAAJwCAAAAAAAAPQIAAAAAAAAZAQAAAAAAAHUBAAAAAAAA/AAAAAAAAADNAwAAAAAAAKAAAAAAAAAA9gMAAAAAAADSAQAAAAAAAEIDAAAAAAAAYAIAAAAAAADzAwAAAAAAANQBAAAAAAAAXAAAAAAAAAC4AgAAAAAAANsCAAAAAAAAnQAAAAAAAABCAAAAAAAAAGcDAAAAAAAAQwAAAAAAAABBAQAAAAAAAFIDAAAAAAAADgEAAAAAAACtAQAAAAAAAI0DAAAAAAAA2wEAAAAAAADSAgAAAAAAAAADAAAAAAAAtQIAAAAAAACZAgAAAAAAAJQDAAAAAAAA9gAAAAAAAAD+AAAAAAAAADABAAAAAAAAwAEAAAAAAACPAgAAAAAAAHcAAAAAAAAAZAMAAAAAAAC8AwAAAAAAAB8CAAAAAAAAEQIAAAAAAAA8AQAAAAAAACUBAAAAAAAAnAIAAAAAAAAJAQAAAAAAAI8BAAAAAAAATwEAAAAAAAD+AQAAAAAAAPIBAAAAAAAA+wAAAAAAAADhAwAAAAAAAB8BAAAAAAAAdgIAAAAAAAACAAAAAAAAACgAAAAAAAAAzwMAAAAAAAAwAQAAAAAAAEgBAAAAAAAAKwAAAAAAAABnAwAAAAAAALgCAAAAAAAARgEAAAAAAAD/AwAAAAAAAAoDAAAAAAAAhgAAAAAAAADcAwAAAAAAANMDAAAAAAAAiAAAAAAAAACEAgAAAAAAAJwCAAAAAAAAEwMAAAAAAAA3AQAAAAAAAMQBAAAAAAAA5wEAAAAAAAAYAgAAAAAAAH0DAAAAAAAAXwEAAAAAAAAgAQAAAAAAALABAAAAAAAALQMAAAAAAABTAAAAAAAAAKACAAAAAAAA4AAAAAAAAADUAQAAAAAAAB0DAAAAAAAAJAIAAAAAAADtAAAAAAAAAP0AAAAAAAAABAAAAAAAAAAGAgAAAAAAADABAAAAAAAAfgMAAAAAAAAwAgAAAAAAAKACAAAAAAAAqwMAAAAAAAD3AwAAAAAAAEICAAAAAAAAcgMAAAAAAACUAgAAAAAAAGUCAAAAAAAAOwIAAAAAAACQAgAAAAAAAHoBAAAAAAAAeQAAAAAAAABGAwAAAAAAAFQDAAAAAAAAEQEAAAAAAABSAQAAAAAAAMQAAAAAAAAAoAAAAAAAAAADAAAAAAAAAOgBAAAAAAAA/gIAAAAAAADWAgAAAAAAAJsBAAAAAAAAugMAAAAAAAB+AQAAAAAAAMoDAAAAAAAA5gIAAAAAAACsAgAAAAAAAKUCAAAAAAAAkQMAAAAAAACkAgAAAAAAABoBAAAAAAAAWwEAAAAAAAAfAgAAAAAAAOMBAAAAAAAA/gMAAAAAAABgAAAAAAAAAAwDAAAAAAAAQgAAAAAAAABBAQAAAAAAAC8CAAAAAAAAxwEAAAAAAAA6AAAAAAAAAMMBAAAAAAAAsAMAAAAAAACoAAAAAAAAAN0DAAAAAAAAQAAAAAAAAACGAwAAAAAAALICAAAAAAAAUEsHCMe59dwAEQAAABEAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAADgAEAE5VX1RSRS92ZXJzaW9uRkIAADMKUEsHCNGeZ1UCAAAAAgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAHQAzAE5VX1RSRS8uZGF0YS9zZXJpYWxpemF0aW9uX2lkRkIvAFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMTU3MjgxOTk1MTE4NTQ2NDMwMjYwNTU3NDkxMjIzOTg3MDk1Njg5MVBLBwi7S++lKAAAACgAAABQSwECAAAAAAgIAAAAAAAA7FjEkS8BAAAvAQAADwAAAAAAAAAAAAAAAAAAAAAATlVfVFJFL2RhdGEucGtsUEsBAgAAAAAICAAAAAAAALfv3IMBAAAAAQAAABYAAAAAAAAAAAAAAAAAfwEAAE5VX1RSRS8uZm9ybWF0X3ZlcnNpb25QSwECAAAAAAgIAAAAAAAAP3dx6QIAAAACAAAAGQAAAAAAAAAAAAAAAADRAQAATlVfVFJFLy5zdG9yYWdlX2FsaWdubWVudFBLAQIAAAAACAgAAAAAAACFPeMZBgAAAAYAAAAQAAAAAAAAAAAAAAAAAFICAABOVV9UUkUvYnl0ZW9yZGVyUEsBAgAAAAAICAAAAAAAAMe59dwAEQAAABEAAA0AAAAAAAAAAAAAAAAA1gIAAE5VX1RSRS9kYXRhLzBQSwECAAAAAAgIAAAAAAAA0Z5nVQIAAAACAAAADgAAAAAAAAAAAAAAAABQFAAATlVfVFJFL3ZlcnNpb25QSwECAAAAAAgIAAAAAAAAu0vvpSgAAAAoAAAAHQAAAAAAAAAAAAAAAACSFAAATlVfVFJFLy5kYXRhL3NlcmlhbGl6YXRpb25faWRQSwYGLAAAAAAAAAAeAy0AAAAAAAAAAAAHAAAAAAAAAAcAAAAAAAAAyAEAAAAAAAA4FQAAAAAAAFBLBgcAAAAAABcAAAAAAAABAAAAUEsFBgAAAAAHAAcAyAEAADgVAAAAAA==', 'Minh_Nhi.pt': 'UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAPABMAVFJFX0VNL2RhdGEucGtsRkIPAFpaWlpaWlpaWlpaWlpaWoACfXEAKFgOAAAAZm9ybWF0X3ZlcnNpb25xAUsBWBAAAAByZWZfYXVkaW9fdG9rZW5zcQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKTG9uZ1N0b3JhZ2UKcQVYAQAAADBxBlgDAAAAY3B1cQdNkAR0cQhRSwBLCEuShnEJS5JLAYZxColjY29sbGVjdGlvbnMKT3JkZXJlZERpY3QKcQspUnEMdHENUnEOWAgAAAByZWZfdGV4dHEPWGgAAABDw7MgY8O0IMOhbyB0cuG6r25nIOG7nyBkxrDhu5tpIG7GsOG7m2MgduG6q3kgY2jDoXUuIEPDtCDhuqV5IGLhuqNvIHh14buRbmcgxJHDonksIGPDtCBjaG8gY8OhaSBsxrDhu6NjLnEQWAcAAAByZWZfcm1zcRFHP8LvHYAAAAB1LlBLBwhKFmSIYgEAAGIBAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABYAGgBUUkVfRU0vLmZvcm1hdF92ZXJzaW9uRkIWAFpaWlpaWlpaWlpaWlpaWlpaWlpaWloxUEsHCLfv3IMBAAAAAQAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAGQA4AFRSRV9FTS8uc3RvcmFnZV9hbGlnbm1lbnRGQjQAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjY0UEsHCD93cekCAAAAAgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAEABAAFRSRV9FTS9ieXRlb3JkZXJGQjwAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpabGl0dGxlUEsHCIU94xkGAAAABgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAADQA/AFRSRV9FTS9kYXRhLzBGQjsAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWloQAQAAAAAAABkBAAAAAAAAmgIAAAAAAACbAwAAAAAAAAMCAAAAAAAAfwAAAAAAAAD8AAAAAAAAABYCAAAAAAAA6wMAAAAAAACnAgAAAAAAAAYCAAAAAAAAzwAAAAAAAAAUAAAAAAAAAIsCAAAAAAAArwEAAAAAAABUAwAAAAAAAJwCAAAAAAAAIAEAAAAAAADVAgAAAAAAAAcBAAAAAAAAZAEAAAAAAADXAgAAAAAAABkBAAAAAAAAzQEAAAAAAADfAwAAAAAAAEIDAAAAAAAApwIAAAAAAACnAgAAAAAAABYCAAAAAAAAYQEAAAAAAAA6AAAAAAAAAO8DAAAAAAAA1QIAAAAAAACnAgAAAAAAAOYCAAAAAAAAEAMAAAAAAAARAgAAAAAAAH0CAAAAAAAAfQIAAAAAAAB9AgAAAAAAABYCAAAAAAAAiQIAAAAAAADvAwAAAAAAABUAAAAAAAAAJQIAAAAAAABxAgAAAAAAABgBAAAAAAAAHwAAAAAAAAAgAQAAAAAAAPsCAAAAAAAA9QEAAAAAAAAUAQAAAAAAAIgCAAAAAAAAzQEAAAAAAADLAgAAAAAAAGEBAAAAAAAAFgIAAAAAAAD0AwAAAAAAAH0CAAAAAAAAGwAAAAAAAACnAgAAAAAAANECAAAAAAAAIwEAAAAAAABEAgAAAAAAAAECAAAAAAAAmgMAAAAAAADRAwAAAAAAAM0AAAAAAAAAMwAAAAAAAAAsAwAAAAAAAM0AAAAAAAAAzQAAAAAAAADlAwAAAAAAAM0AAAAAAAAAmwMAAAAAAACbAwAAAAAAAHUCAAAAAAAADwMAAAAAAAAoAwAAAAAAABcDAAAAAAAAZAEAAAAAAAAeAQAAAAAAAKcCAAAAAAAA/gAAAAAAAACiAQAAAAAAAKIBAAAAAAAAkAEAAAAAAABhAQAAAAAAANUCAAAAAAAA9AMAAAAAAAB0AAAAAAAAAMkDAAAAAAAAnAEAAAAAAADwAwAAAAAAAKMCAAAAAAAAOgEAAAAAAAARAgAAAAAAAH0CAAAAAAAApwIAAAAAAAAQAwAAAAAAANUCAAAAAAAALgMAAAAAAAARAQAAAAAAAAkAAAAAAAAAVAMAAAAAAACRAQAAAAAAAO8DAAAAAAAA4wMAAAAAAABkAQAAAAAAAKIBAAAAAAAA0QMAAAAAAAAuAwAAAAAAAIgCAAAAAAAA3gMAAAAAAACcAQAAAAAAAPEBAAAAAAAAzwAAAAAAAAAPAwAAAAAAALsAAAAAAAAA+AIAAAAAAABCAwAAAAAAACsDAAAAAAAAwwAAAAAAAADOAwAAAAAAAGEBAAAAAAAADgAAAAAAAADVAgAAAAAAAOMBAAAAAAAA1QIAAAAAAADFAAAAAAAAAP8CAAAAAAAAZAEAAAAAAACxAAAAAAAAAIkCAAAAAAAAsQMAAAAAAACvAQAAAAAAAB4AAAAAAAAAggIAAAAAAADyAgAAAAAAAGQBAAAAAAAAHQMAAAAAAADmAwAAAAAAAJEAAAAAAAAAhgEAAAAAAABSAAAAAAAAAFIAAAAAAAAAEAMAAAAAAAAJAAAAAAAAACMAAAAAAAAAsAEAAAAAAACsAwAAAAAAAA0BAAAAAAAA9gEAAAAAAADMAwAAAAAAANcCAAAAAAAAhwEAAAAAAADoAAAAAAAAANwBAAAAAAAAmAEAAAAAAAC9AAAAAAAAAGUDAAAAAAAA9wAAAAAAAABUAQAAAAAAACMBAAAAAAAAZAEAAAAAAACUAQAAAAAAAOoDAAAAAAAASwAAAAAAAAAgAwAAAAAAAFwAAAAAAAAAPgAAAAAAAAAvAwAAAAAAAKwBAAAAAAAAGQAAAAAAAAD0AAAAAAAAAJEAAAAAAAAApQAAAAAAAACTAgAAAAAAAJsBAAAAAAAAGgIAAAAAAAAqAgAAAAAAAGMAAAAAAAAAMwIAAAAAAABrAgAAAAAAAKsAAAAAAAAAVwAAAAAAAADYAQAAAAAAAF8BAAAAAAAALwEAAAAAAAAvAgAAAAAAAOEBAAAAAAAAGQAAAAAAAAA7AwAAAAAAALUBAAAAAAAAOgEAAAAAAABpAQAAAAAAAEIAAAAAAAAAegAAAAAAAACTAAAAAAAAAHsBAAAAAAAAYAEAAAAAAAAnAgAAAAAAAPQCAAAAAAAAVQMAAAAAAAANAwAAAAAAAD8BAAAAAAAAVAEAAAAAAABUAQAAAAAAAGUDAAAAAAAAQwMAAAAAAAA5AAAAAAAAAD4CAAAAAAAArwMAAAAAAABbAQAAAAAAAG8CAAAAAAAA5wEAAAAAAACpAgAAAAAAALADAAAAAAAAsAMAAAAAAABwAgAAAAAAAAkAAAAAAAAAawEAAAAAAADLAwAAAAAAANcCAAAAAAAAxwIAAAAAAAB2AAAAAAAAAFQBAAAAAAAAmwEAAAAAAACCAgAAAAAAADACAAAAAAAAKwMAAAAAAABmAQAAAAAAAEIAAAAAAAAAOQMAAAAAAADAAAAAAAAAAGUAAAAAAAAAdgIAAAAAAAArAQAAAAAAAJcBAAAAAAAAvAAAAAAAAAAxAgAAAAAAAB0BAAAAAAAABQAAAAAAAADcAQAAAAAAADMCAAAAAAAANgEAAAAAAABWAwAAAAAAAAACAAAAAAAA4gMAAAAAAACJAAAAAAAAAIUCAAAAAAAABgEAAAAAAAC0AQAAAAAAALoAAAAAAAAA4gMAAAAAAAC6AAAAAAAAAN4BAAAAAAAAsQMAAAAAAAA5AAAAAAAAAM0AAAAAAAAAvgAAAAAAAABJAgAAAAAAAI4AAAAAAAAA9gAAAAAAAAAuAQAAAAAAAIYCAAAAAAAAmAIAAAAAAACVAAAAAAAAAHcAAAAAAAAApwAAAAAAAAC/AgAAAAAAADAAAAAAAAAAVgAAAAAAAADqAwAAAAAAADsAAAAAAAAAhwIAAAAAAAA7AQAAAAAAAOIBAAAAAAAA9QIAAAAAAACkAwAAAAAAADcAAAAAAAAALAEAAAAAAAD3AAAAAAAAAKoDAAAAAAAAVAEAAAAAAAB1AgAAAAAAAM4DAAAAAAAAowIAAAAAAABSAQAAAAAAAB0CAAAAAAAAnwEAAAAAAADXAwAAAAAAAPwBAAAAAAAA5wIAAAAAAABaAQAAAAAAAFcDAAAAAAAA5AAAAAAAAABbAAAAAAAAAJICAAAAAAAAowIAAAAAAACkAwAAAAAAAJQBAAAAAAAA8gIAAAAAAADxAAAAAAAAAN4DAAAAAAAAhAAAAAAAAACzAAAAAAAAAMQBAAAAAAAAxAEAAAAAAAD2AgAAAAAAABsBAAAAAAAA8gEAAAAAAACcAgAAAAAAAMYAAAAAAAAA2AAAAAAAAABhAgAAAAAAAFQAAAAAAAAA+QIAAAAAAAC9AAAAAAAAAF0AAAAAAAAAUwIAAAAAAAD+AQAAAAAAAFgCAAAAAAAAqAIAAAAAAAA5AQAAAAAAAIEDAAAAAAAA8QEAAAAAAACtAwAAAAAAAMAAAAAAAAAAvAMAAAAAAADqAgAAAAAAADEDAAAAAAAAbgMAAAAAAACiAgAAAAAAAFACAAAAAAAAMQMAAAAAAACBAwAAAAAAACIAAAAAAAAANgMAAAAAAAC2AwAAAAAAANcDAAAAAAAA8gIAAAAAAADFAQAAAAAAAPcCAAAAAAAA2AAAAAAAAAAuAAAAAAAAADEDAAAAAAAA7QEAAAAAAAAqAgAAAAAAACUCAAAAAAAAEQEAAAAAAABPAgAAAAAAAIMDAAAAAAAAZQIAAAAAAAAeAgAAAAAAAFACAAAAAAAADgMAAAAAAAC0AgAAAAAAAHQCAAAAAAAAdQIAAAAAAAAJAQAAAAAAAHgCAAAAAAAAuwIAAAAAAAAuAgAAAAAAAAkBAAAAAAAAhAMAAAAAAADrAAAAAAAAANUCAAAAAAAAoAIAAAAAAAAdAwAAAAAAAPgDAAAAAAAAjQAAAAAAAAB2AAAAAAAAABwCAAAAAAAAhAAAAAAAAACbAgAAAAAAALcCAAAAAAAA4wEAAAAAAADEAQAAAAAAAPsBAAAAAAAAigEAAAAAAAAAAQAAAAAAAJ0AAAAAAAAAXQMAAAAAAADkAgAAAAAAAPwDAAAAAAAAkwEAAAAAAAA5AQAAAAAAAJcCAAAAAAAAHgIAAAAAAAAmAAAAAAAAAOUCAAAAAAAAcwIAAAAAAACAAwAAAAAAACoCAAAAAAAAigIAAAAAAAAqAgAAAAAAAB0DAAAAAAAAigIAAAAAAAChAgAAAAAAAI0CAAAAAAAAogMAAAAAAAAsAwAAAAAAACoCAAAAAAAAwgAAAAAAAADOAQAAAAAAAC4AAAAAAAAAdAEAAAAAAADOAQAAAAAAAOcDAAAAAAAA+AMAAAAAAAAzAgAAAAAAACQDAAAAAAAAOgMAAAAAAABtAgAAAAAAALsCAAAAAAAADAIAAAAAAACBAAAAAAAAACwAAAAAAAAANwMAAAAAAAC0AQAAAAAAALIDAAAAAAAAigAAAAAAAADEAQAAAAAAAEsDAAAAAAAARAMAAAAAAAAHAQAAAAAAAFYDAAAAAAAADAIAAAAAAADJAwAAAAAAAOMBAAAAAAAAVgMAAAAAAADdAAAAAAAAAAkBAAAAAAAAuAEAAAAAAACrAwAAAAAAAP0AAAAAAAAAywIAAAAAAACvAgAAAAAAAJ4CAAAAAAAAEwEAAAAAAABjAwAAAAAAADMDAAAAAAAA6AMAAAAAAAA3AAAAAAAAABAAAAAAAAAA0gMAAAAAAAAzAwAAAAAAAA0AAAAAAAAARQIAAAAAAAALAgAAAAAAALECAAAAAAAABAAAAAAAAAAUAAAAAAAAAMcDAAAAAAAApAAAAAAAAADkAgAAAAAAAEoAAAAAAAAAAAAAAAAAAACtAgAAAAAAAHACAAAAAAAAIQEAAAAAAABYAAAAAAAAAJIBAAAAAAAAmAMAAAAAAABaAwAAAAAAALsAAAAAAAAA9wEAAAAAAACMAQAAAAAAAPMAAAAAAAAA2AIAAAAAAAArAAAAAAAAABIBAAAAAAAAYwIAAAAAAABVAwAAAAAAAK0CAAAAAAAAzgEAAAAAAACoAwAAAAAAAMABAAAAAAAAPgEAAAAAAADKAgAAAAAAAAoCAAAAAAAAUgMAAAAAAAAuAQAAAAAAALEAAAAAAAAA7AEAAAAAAAAVAwAAAAAAAK0CAAAAAAAArQIAAAAAAACbAQAAAAAAAN4CAAAAAAAAyQMAAAAAAACKAwAAAAAAALYDAAAAAAAAZgAAAAAAAABhAAAAAAAAAKQDAAAAAAAAmAMAAAAAAAAuAwAAAAAAAKwAAAAAAAAAgAIAAAAAAACqAAAAAAAAAIcAAAAAAAAASwEAAAAAAAAyAgAAAAAAAIQAAAAAAAAA/wIAAAAAAABjAwAAAAAAAAMAAAAAAAAACAAAAAAAAACrAQAAAAAAALMCAAAAAAAASQMAAAAAAABzAQAAAAAAAKgBAAAAAAAAcgAAAAAAAABAAAAAAAAAABMDAAAAAAAA6QMAAAAAAACTAAAAAAAAAH8BAAAAAAAAyQIAAAAAAABRAQAAAAAAAJMAAAAAAAAAagMAAAAAAABgAQAAAAAAANwBAAAAAAAACAEAAAAAAABwAgAAAAAAADcAAAAAAAAA6QIAAAAAAADBAgAAAAAAAI0BAAAAAAAARAEAAAAAAAAKAgAAAAAAAKoBAAAAAAAAbgMAAAAAAACIAAAAAAAAAOUDAAAAAAAArAAAAAAAAABGAwAAAAAAAAkDAAAAAAAAFQIAAAAAAAA9AwAAAAAAABQBAAAAAAAABgIAAAAAAACTAgAAAAAAAHoCAAAAAAAAPQAAAAAAAACsAAAAAAAAACgBAAAAAAAAOAEAAAAAAAD4AAAAAAAAAF8BAAAAAAAA5AAAAAAAAAA2AwAAAAAAANsCAAAAAAAACAAAAAAAAAC7AwAAAAAAAK0CAAAAAAAAkgIAAAAAAABGAwAAAAAAAAEAAAAAAAAA6AMAAAAAAABmAQAAAAAAAPwBAAAAAAAA8AIAAAAAAAAZAQAAAAAAALYAAAAAAAAAtgAAAAAAAAAKAgAAAAAAALMCAAAAAAAAXQEAAAAAAACvAQAAAAAAAAADAAAAAAAAaQMAAAAAAABzAAAAAAAAAPgCAAAAAAAAYgIAAAAAAAD8AQAAAAAAAPwBAAAAAAAAsgMAAAAAAAAlAQAAAAAAAFwCAAAAAAAA4AMAAAAAAAALAgAAAAAAAEoAAAAAAAAAgQAAAAAAAAAuAgAAAAAAAGMDAAAAAAAAJwMAAAAAAAB4AAAAAAAAAI8CAAAAAAAAWQEAAAAAAAByAgAAAAAAAGsAAAAAAAAA7QAAAAAAAABxAgAAAAAAAIYBAAAAAAAA7QMAAAAAAABeAQAAAAAAACsAAAAAAAAAJgIAAAAAAAAaAQAAAAAAAF8AAAAAAAAAUQMAAAAAAADBAwAAAAAAAC0AAAAAAAAAvgIAAAAAAACKAwAAAAAAANsAAAAAAAAAJwMAAAAAAABlAwAAAAAAAOMBAAAAAAAALwEAAAAAAADOAwAAAAAAAF0BAAAAAAAADgMAAAAAAABAAgAAAAAAAG4AAAAAAAAAKQEAAAAAAADMAgAAAAAAAP4AAAAAAAAAJAIAAAAAAAC2AgAAAAAAALIAAAAAAAAAfgIAAAAAAADnAAAAAAAAAEcCAAAAAAAABAMAAAAAAABkAQAAAAAAAOkAAAAAAAAAhQAAAAAAAAAZAwAAAAAAAG8BAAAAAAAAfQAAAAAAAADqAQAAAAAAAGQCAAAAAAAAfwMAAAAAAADNAAAAAAAAAE0DAAAAAAAAyAAAAAAAAABuAgAAAAAAAE4AAAAAAAAAegIAAAAAAACeAwAAAAAAAPcBAAAAAAAALQMAAAAAAABjAAAAAAAAAOsDAAAAAAAAfgIAAAAAAACJAQAAAAAAAJECAAAAAAAAHwIAAAAAAABgAAAAAAAAAAwCAAAAAAAAAgIAAAAAAAAPAQAAAAAAAGUBAAAAAAAAgQAAAAAAAAAyAQAAAAAAAAkCAAAAAAAAnQEAAAAAAADoAgAAAAAAAGsCAAAAAAAAHwAAAAAAAAC2AgAAAAAAALUDAAAAAAAA2QAAAAAAAAAKAAAAAAAAAPkAAAAAAAAAQgEAAAAAAAAkAQAAAAAAAF0AAAAAAAAAJAEAAAAAAABPAQAAAAAAADQDAAAAAAAADQMAAAAAAADRAQAAAAAAAFAAAAAAAAAAkAIAAAAAAAAVAQAAAAAAAPACAAAAAAAAUQIAAAAAAAALAgAAAAAAAKkCAAAAAAAASQEAAAAAAAD4AAAAAAAAAO0DAAAAAAAA7AEAAAAAAADOAwAAAAAAAAEAAAAAAAAAEAAAAAAAAABqAAAAAAAAAEEAAAAAAAAAwwAAAAAAAADPAgAAAAAAADQDAAAAAAAADAEAAAAAAAD6AgAAAAAAABICAAAAAAAAPQEAAAAAAADKAgAAAAAAAAAAAAAAAAAAPgIAAAAAAACeAgAAAAAAAJkCAAAAAAAA8gAAAAAAAABjAQAAAAAAAPgBAAAAAAAAQAMAAAAAAAD/AwAAAAAAACQBAAAAAAAAfwIAAAAAAABMAAAAAAAAABIAAAAAAAAAPwAAAAAAAAC0AgAAAAAAADQCAAAAAAAAuwAAAAAAAABRAQAAAAAAAF8AAAAAAAAAPAEAAAAAAAA/AQAAAAAAAHsBAAAAAAAAmAEAAAAAAAAeAwAAAAAAAE8CAAAAAAAABgEAAAAAAAAhAgAAAAAAABwCAAAAAAAA4AIAAAAAAADSAAAAAAAAAJ8BAAAAAAAAaQIAAAAAAAChAAAAAAAAAGYBAAAAAAAAdQIAAAAAAAD2AgAAAAAAABQCAAAAAAAAwQEAAAAAAAB4AgAAAAAAADYBAAAAAAAA6gIAAAAAAABEAwAAAAAAAJ8AAAAAAAAAqwEAAAAAAAD1AQAAAAAAANoDAAAAAAAAMAIAAAAAAAC7AAAAAAAAAOoCAAAAAAAAewMAAAAAAACGAwAAAAAAABADAAAAAAAAJwAAAAAAAAAjAgAAAAAAAJ8CAAAAAAAA5gAAAAAAAADOAwAAAAAAAMMBAAAAAAAAnAIAAAAAAAA0AQAAAAAAAK4CAAAAAAAAOwAAAAAAAAASAwAAAAAAADoCAAAAAAAADQAAAAAAAACWAgAAAAAAAOsCAAAAAAAAIwEAAAAAAACYAgAAAAAAAGYCAAAAAAAA5gEAAAAAAADiAQAAAAAAANUAAAAAAAAAygMAAAAAAAAtAQAAAAAAAFwAAAAAAAAAFQIAAAAAAAB/AQAAAAAAANkDAAAAAAAAtQEAAAAAAAD2AQAAAAAAAEMBAAAAAAAAEgMAAAAAAADQAQAAAAAAAPYDAAAAAAAAagEAAAAAAABxAgAAAAAAABUCAAAAAAAAlgMAAAAAAAABAAAAAAAAAJYBAAAAAAAAoQIAAAAAAAAhAwAAAAAAAKoCAAAAAAAALAIAAAAAAAD3AQAAAAAAAP8BAAAAAAAAHQAAAAAAAAD7AAAAAAAAAJEDAAAAAAAAXAIAAAAAAABwAgAAAAAAAIsDAAAAAAAAeAIAAAAAAABrAwAAAAAAAJkAAAAAAAAA1wIAAAAAAAAMAgAAAAAAAM0DAAAAAAAA6QEAAAAAAACRAgAAAAAAAMoBAAAAAAAAggEAAAAAAADfAQAAAAAAAIMBAAAAAAAA+wMAAAAAAADsAgAAAAAAAH8BAAAAAAAAaQAAAAAAAACPAQAAAAAAAH0BAAAAAAAAqQEAAAAAAABQAQAAAAAAAAQCAAAAAAAAUgMAAAAAAAB3AgAAAAAAAOoDAAAAAAAAygEAAAAAAACFAgAAAAAAAOADAAAAAAAANwEAAAAAAAD7AgAAAAAAAHICAAAAAAAANwMAAAAAAAAhAAAAAAAAABsCAAAAAAAA4wIAAAAAAACoAAAAAAAAAPgBAAAAAAAAdgMAAAAAAACeAQAAAAAAAH4CAAAAAAAAnwMAAAAAAACeAQAAAAAAAAYBAAAAAAAALwEAAAAAAAAiAgAAAAAAAAAAAAAAAAAAkwMAAAAAAADOAQAAAAAAAIsDAAAAAAAAcgMAAAAAAACOAgAAAAAAAGcAAAAAAAAA2gAAAAAAAABUAwAAAAAAANYDAAAAAAAAgAMAAAAAAAC6AwAAAAAAAJgBAAAAAAAApAMAAAAAAAB/AQAAAAAAAAMDAAAAAAAANwMAAAAAAAA2AgAAAAAAAPcBAAAAAAAA9gIAAAAAAADCAAAAAAAAABwDAAAAAAAApgAAAAAAAACMAAAAAAAAAIsDAAAAAAAAdAMAAAAAAAA8AQAAAAAAAAYAAAAAAAAAfwEAAAAAAABtAgAAAAAAANYAAAAAAAAAlgIAAAAAAAAXAAAAAAAAAAUDAAAAAAAAjgEAAAAAAABPAQAAAAAAAGEAAAAAAAAASwMAAAAAAABwAAAAAAAAAFkAAAAAAAAAjAAAAAAAAACcAAAAAAAAAPwBAAAAAAAA6wAAAAAAAADlAgAAAAAAANcAAAAAAAAAZgMAAAAAAAAlAgAAAAAAAL8BAAAAAAAAdAMAAAAAAAAAAAAAAAAAAEIBAAAAAAAA7gEAAAAAAADoAwAAAAAAAF8BAAAAAAAASAIAAAAAAAC/AQAAAAAAAMAAAAAAAAAA3wIAAAAAAABSAgAAAAAAACACAAAAAAAAngEAAAAAAAA4AwAAAAAAAO0DAAAAAAAA5AIAAAAAAAAtAAAAAAAAAGcCAAAAAAAA5wEAAAAAAAAuAwAAAAAAAGsDAAAAAAAAvQAAAAAAAAAMAgAAAAAAAM4DAAAAAAAAbQAAAAAAAAA7AgAAAAAAAM4CAAAAAAAA4AMAAAAAAABvAgAAAAAAANoAAAAAAAAASgIAAAAAAABeAwAAAAAAAJgAAAAAAAAA9wEAAAAAAABLAwAAAAAAAG0BAAAAAAAAtAIAAAAAAAD1AwAAAAAAAKQCAAAAAAAArAAAAAAAAACgAwAAAAAAAGEBAAAAAAAAyQEAAAAAAADgAQAAAAAAAA0AAAAAAAAAEwIAAAAAAAC2AgAAAAAAAGEAAAAAAAAAtwEAAAAAAABfAAAAAAAAAOQCAAAAAAAAlQEAAAAAAABhAQAAAAAAAOQCAAAAAAAABgMAAAAAAAB/AgAAAAAAAPUDAAAAAAAApwAAAAAAAADDAwAAAAAAAHkDAAAAAAAA6wEAAAAAAABFAQAAAAAAAOQDAAAAAAAAZwMAAAAAAADHAAAAAAAAAAkAAAAAAAAAWAAAAAAAAACTAQAAAAAAAA0DAAAAAAAACQMAAAAAAAAxAAAAAAAAABgCAAAAAAAAyQAAAAAAAABOAwAAAAAAAEwBAAAAAAAAiAMAAAAAAADnAwAAAAAAAAoDAAAAAAAA0gIAAAAAAADNAAAAAAAAALQAAAAAAAAAxgAAAAAAAAB1AQAAAAAAAEICAAAAAAAAfAIAAAAAAADXAwAAAAAAABkDAAAAAAAAPgEAAAAAAABbAwAAAAAAAKsCAAAAAAAAJgMAAAAAAADwAgAAAAAAAMYBAAAAAAAAnAEAAAAAAAAxAAAAAAAAAJ4AAAAAAAAAZAIAAAAAAAAPAwAAAAAAACcBAAAAAAAAfAEAAAAAAACdAgAAAAAAAFwCAAAAAAAAwAMAAAAAAACLAAAAAAAAAM4CAAAAAAAA8QEAAAAAAABbAgAAAAAAAJkCAAAAAAAAnQIAAAAAAABGAAAAAAAAAKEDAAAAAAAAXAIAAAAAAACqAAAAAAAAAGgDAAAAAAAAIQIAAAAAAAANAQAAAAAAAH0DAAAAAAAATgMAAAAAAADYAwAAAAAAABwBAAAAAAAAlQIAAAAAAAAKAQAAAAAAAKMBAAAAAAAAVAEAAAAAAAB+AwAAAAAAAOUAAAAAAAAA0AMAAAAAAACLAgAAAAAAAOwBAAAAAAAArQMAAAAAAACXAwAAAAAAADMAAAAAAAAAwgAAAAAAAACfAgAAAAAAAMoAAAAAAAAAlQAAAAAAAACrAQAAAAAAAFEAAAAAAAAAcQIAAAAAAAATAAAAAAAAANYCAAAAAAAA6wIAAAAAAACFAgAAAAAAAGAAAAAAAAAAaAAAAAAAAAC0AgAAAAAAAKgAAAAAAAAAiwAAAAAAAADzAwAAAAAAAAUBAAAAAAAA6QMAAAAAAAAQAAAAAAAAAGEDAAAAAAAArgEAAAAAAABaAAAAAAAAAMEDAAAAAAAA0gEAAAAAAADbAAAAAAAAAJsCAAAAAAAAUQMAAAAAAAAAAAAAAAAAAKEAAAAAAAAAXwAAAAAAAAA7AwAAAAAAAGkCAAAAAAAATAEAAAAAAAAvAAAAAAAAAJoCAAAAAAAAEwIAAAAAAADjAgAAAAAAAEQBAAAAAAAAzAEAAAAAAABmAQAAAAAAAJMBAAAAAAAAwQIAAAAAAABzAAAAAAAAAD4BAAAAAAAA/AMAAAAAAABxAQAAAAAAAOcAAAAAAAAAzwMAAAAAAADMAgAAAAAAAJ4DAAAAAAAAawMAAAAAAACuAgAAAAAAALkDAAAAAAAASgAAAAAAAAAOAQAAAAAAADoAAAAAAAAAqwAAAAAAAAAVAgAAAAAAAMUDAAAAAAAAFwMAAAAAAAAlAQAAAAAAAI0AAAAAAAAAmgAAAAAAAADeAAAAAAAAAPoDAAAAAAAAIQMAAAAAAABBAwAAAAAAAPIDAAAAAAAACAAAAAAAAADxAgAAAAAAAJwDAAAAAAAAsQAAAAAAAABlAwAAAAAAACQBAAAAAAAANAAAAAAAAADYAAAAAAAAAIwDAAAAAAAAuQIAAAAAAAAuAQAAAAAAAOAAAAAAAAAAlAIAAAAAAADhAQAAAAAAAEsAAAAAAAAAfwEAAAAAAABPAQAAAAAAAP0AAAAAAAAAuQIAAAAAAAC2AwAAAAAAAJoCAAAAAAAAKAIAAAAAAABIAwAAAAAAAIwAAAAAAAAAzAMAAAAAAAD3AAAAAAAAABYDAAAAAAAA8QIAAAAAAAA2AQAAAAAAAHEDAAAAAAAAzAEAAAAAAAD3AwAAAAAAACoAAAAAAAAAlQIAAAAAAAB0AQAAAAAAAMoBAAAAAAAA9gEAAAAAAABHAgAAAAAAAFEBAAAAAAAAYwMAAAAAAABNAAAAAAAAAK0AAAAAAAAAPwEAAAAAAAA3AAAAAAAAALsBAAAAAAAAmwIAAAAAAACvAQAAAAAAAMQCAAAAAAAAawAAAAAAAABaAAAAAAAAAJkDAAAAAAAAkAMAAAAAAAC9AAAAAAAAAHICAAAAAAAAuQEAAAAAAACuAgAAAAAAAHgAAAAAAAAAGQEAAAAAAAAZAQAAAAAAAHICAAAAAAAA5gMAAAAAAABuAwAAAAAAAEgDAAAAAAAAJQMAAAAAAACZAwAAAAAAAFsAAAAAAAAAlwMAAAAAAACNAAAAAAAAAFBLBwgHc9vQgCQAAIAkAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAAA4ABABUUkVfRU0vdmVyc2lvbkZCAAAzClBLBwjRnmdVAgAAAAIAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAAB0AMwBUUkVfRU0vLmRhdGEvc2VyaWFsaXphdGlvbl9pZEZCLwBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjE1NzI4MTk5NTExODU0NjQzMDI2MDU0MzkzMjg0NDE5MTQ2ODQ3MDBQSwcIah0S7ygAAAAoAAAAUEsBAgAAAAAICAAAAAAAAEoWZIhiAQAAYgEAAA8AAAAAAAAAAAAAAAAAAAAAAFRSRV9FTS9kYXRhLnBrbFBLAQIAAAAACAgAAAAAAAC379yDAQAAAAEAAAAWAAAAAAAAAAAAAAAAALIBAABUUkVfRU0vLmZvcm1hdF92ZXJzaW9uUEsBAgAAAAAICAAAAAAAAD93cekCAAAAAgAAABkAAAAAAAAAAAAAAAAAEQIAAFRSRV9FTS8uc3RvcmFnZV9hbGlnbm1lbnRQSwECAAAAAAgIAAAAAAAAhT3jGQYAAAAGAAAAEAAAAAAAAAAAAAAAAACSAgAAVFJFX0VNL2J5dGVvcmRlclBLAQIAAAAACAgAAAAAAAAHc9vQgCQAAIAkAAANAAAAAAAAAAAAAAAAABYDAABUUkVfRU0vZGF0YS8wUEsBAgAAAAAICAAAAAAAANGeZ1UCAAAAAgAAAA4AAAAAAAAAAAAAAAAAECgAAFRSRV9FTS92ZXJzaW9uUEsBAgAAAAAICAAAAAAAAGodEu8oAAAAKAAAAB0AAAAAAAAAAAAAAAAAUigAAFRSRV9FTS8uZGF0YS9zZXJpYWxpemF0aW9uX2lkUEsGBiwAAAAAAAAAHgMtAAAAAAAAAAAABwAAAAAAAAAHAAAAAAAAAMgBAAAAAAAA+CgAAAAAAABQSwYHAAAAAMAqAAAAAAAAAQAAAFBLBQYAAAAABwAHAMgBAAD4KAAAAAA=', 'Minh_Quy.pt': 'UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAARABEAVFJBSV9UUkUvZGF0YS5wa2xGQg0AWlpaWlpaWlpaWlpaWoACfXEAKFgOAAAAZm9ybWF0X3ZlcnNpb25xAUsBWBAAAAByZWZfYXVkaW9fdG9rZW5zcQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKTG9uZ1N0b3JhZ2UKcQVYAQAAADBxBlgDAAAAY3B1cQdN2AR0cQhRSwBLCEubhnEJS5tLAYZxColjY29sbGVjdGlvbnMKT3JkZXJlZERpY3QKcQspUnEMdHENUnEOWAgAAAByZWZfdGV4dHEPWGIAAABUcuG6o2kgcGjhurNuZyBuaMawIG3hu5l0IHThuqVtIGzhu6VhIEzDoCBt4buZdCBi4buZIGRhIG5nxrDhu51pIMSQ4bunIG3hurd0IG3FqWksIMSR4bunIHTDs2MgdGFpLnEQWAcAAAByZWZfcm1zcRFHP8a5tkAAAAB1LlBLBwhu7250XAEAAFwBAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABgAHgBUUkFJX1RSRS8uZm9ybWF0X3ZlcnNpb25GQhoAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWloxUEsHCLfv3IMBAAAAAQAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAGwA2AFRSQUlfVFJFLy5zdG9yYWdlX2FsaWdubWVudEZCMgBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjY0UEsHCD93cekCAAAAAgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAEgA+AFRSQUlfVFJFL2J5dGVvcmRlckZCOgBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpabGl0dGxlUEsHCIU94xkGAAAABgAAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAADwA9AFRSQUlfVFJFL2RhdGEvMEZCOQBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlqSAAAAAAAAAEkDAAAAAAAASQMAAAAAAACDAAAAAAAAACoCAAAAAAAAhQEAAAAAAACDAAAAAAAAAIQCAAAAAAAAVwAAAAAAAADJAQAAAAAAAHUAAAAAAAAADgEAAAAAAAB2AgAAAAAAANkBAAAAAAAAdwAAAAAAAACOAgAAAAAAAFcBAAAAAAAAvgEAAAAAAAAOAQAAAAAAAHYCAAAAAAAAowEAAAAAAAAnAwAAAAAAADUBAAAAAAAAdgIAAAAAAAApAwAAAAAAACcDAAAAAAAA9QMAAAAAAAAnAwAAAAAAAGMAAAAAAAAAFgIAAAAAAABzAgAAAAAAACsCAAAAAAAAhQEAAAAAAADsAgAAAAAAAJoAAAAAAAAAcQAAAAAAAAAnAwAAAAAAAAwCAAAAAAAAdQMAAAAAAAAUAQAAAAAAALEDAAAAAAAAoQEAAAAAAAD9AAAAAAAAAOICAAAAAAAATgIAAAAAAABEAwAAAAAAACsCAAAAAAAACAIAAAAAAADXAwAAAAAAAJwBAAAAAAAAKwIAAAAAAADoAAAAAAAAACoCAAAAAAAAbQMAAAAAAAArAgAAAAAAAIQCAAAAAAAAtQMAAAAAAACRAgAAAAAAAIkCAAAAAAAAkQIAAAAAAACRAgAAAAAAAJECAAAAAAAAEAMAAAAAAACFAQAAAAAAAJIAAAAAAAAAkQIAAAAAAAB+AAAAAAAAALIAAAAAAAAAhQMAAAAAAACpAgAAAAAAAE4CAAAAAAAAaQMAAAAAAAB1AAAAAAAAAEcDAAAAAAAAegAAAAAAAADcAwAAAAAAANwDAAAAAAAAYwAAAAAAAACaAAAAAAAAAJoAAAAAAAAA+QAAAAAAAACEAAAAAAAAAGsAAAAAAAAAdQMAAAAAAAAUAgAAAAAAADIBAAAAAAAAcwIAAAAAAABzAgAAAAAAAP8BAAAAAAAARAMAAAAAAACDAAAAAAAAAAIAAAAAAAAALAAAAAAAAABJAgAAAAAAANcDAAAAAAAA9wMAAAAAAACFAQAAAAAAAIUBAAAAAAAAXQEAAAAAAAD1AgAAAAAAAHUAAAAAAAAAggIAAAAAAADVAgAAAAAAAJoAAAAAAAAAxAIAAAAAAAA1AQAAAAAAAPUDAAAAAAAAogEAAAAAAAA/AQAAAAAAAHsBAAAAAAAANAEAAAAAAACSAAAAAAAAAJoAAAAAAAAAkQIAAAAAAAAQAwAAAAAAANUCAAAAAAAAPwAAAAAAAACCAgAAAAAAAMECAAAAAAAAdQMAAAAAAAB1AwAAAAAAAAoCAAAAAAAAKgIAAAAAAACFAQAAAAAAAPcDAAAAAAAAhQEAAAAAAACFAQAAAAAAACwAAAAAAAAA9QIAAAAAAAB1AAAAAAAAAIICAAAAAAAA1QIAAAAAAADbAAAAAAAAAOEAAAAAAAAAuAAAAAAAAABJAwAAAAAAAOkBAAAAAAAA3AMAAAAAAAD+AQAAAAAAABwAAAAAAAAASQMAAAAAAACnAQAAAAAAAIQCAAAAAAAARwMAAAAAAAC+AQAAAAAAAGMAAAAAAAAAcQAAAAAAAABxAAAAAAAAAPUCAAAAAAAAkQIAAAAAAAAqAwAAAAAAAJQDAAAAAAAAAwIAAAAAAAD/AQAAAAAAADcBAAAAAAAAEgIAAAAAAABbAwAAAAAAAFIAAAAAAAAAEwEAAAAAAABLAQAAAAAAAAEDAAAAAAAAqwIAAAAAAABHAwAAAAAAAHQBAAAAAAAA/gEAAAAAAACBAQAAAAAAAFkDAAAAAAAAOgEAAAAAAAD5AQAAAAAAAOgCAAAAAAAAywMAAAAAAABDAQAAAAAAAM4DAAAAAAAAOgEAAAAAAAAvAgAAAAAAAF4AAAAAAAAAsAEAAAAAAAASAwAAAAAAAHoDAAAAAAAA9QIAAAAAAADKAQAAAAAAAFADAAAAAAAAnAMAAAAAAABCAAAAAAAAADoBAAAAAAAA6AEAAAAAAADyAgAAAAAAALACAAAAAAAAawAAAAAAAACvAQAAAAAAAIYCAAAAAAAA6gAAAAAAAADzAAAAAAAAAAMBAAAAAAAA4QEAAAAAAACWAwAAAAAAAEQBAAAAAAAAgwMAAAAAAAC4AwAAAAAAAEABAAAAAAAAcgAAAAAAAAD7AAAAAAAAAL4AAAAAAAAAlQAAAAAAAACHAwAAAAAAAEsAAAAAAAAADwEAAAAAAABtAwAAAAAAACcBAAAAAAAAcAAAAAAAAAC9AQAAAAAAAO8DAAAAAAAAVAEAAAAAAABrAAAAAAAAAAcAAAAAAAAAJwAAAAAAAACWAwAAAAAAAJYDAAAAAAAAvwIAAAAAAACoAgAAAAAAAKgCAAAAAAAAvQAAAAAAAACQAwAAAAAAAOECAAAAAAAAcQEAAAAAAACWAwAAAAAAAOgDAAAAAAAAAwIAAAAAAABGAgAAAAAAAPoCAAAAAAAALwMAAAAAAAAyAwAAAAAAABAAAAAAAAAAVAEAAAAAAADYAQAAAAAAAC4AAAAAAAAA3QMAAAAAAABZAwAAAAAAAI0AAAAAAAAADwIAAAAAAADVAQAAAAAAADgAAAAAAAAAEAEAAAAAAAAnAQAAAAAAAP4CAAAAAAAADwMAAAAAAADaAgAAAAAAAJADAAAAAAAA8gEAAAAAAAD+AgAAAAAAAFsDAAAAAAAA2AIAAAAAAACWAAAAAAAAAKsCAAAAAAAAOAIAAAAAAAACAgAAAAAAAAMCAAAAAAAAlgMAAAAAAAAFAAAAAAAAAD4CAAAAAAAAQwEAAAAAAAC6AAAAAAAAAPsAAAAAAAAAgQEAAAAAAACIAgAAAAAAAC0CAAAAAAAAnwEAAAAAAACQAwAAAAAAAJADAAAAAAAAlgMAAAAAAAAFAAAAAAAAAEcCAAAAAAAA5gEAAAAAAACjAAAAAAAAAAkCAAAAAAAAVgEAAAAAAADkAwAAAAAAACgBAAAAAAAA8AEAAAAAAACBAgAAAAAAANgCAAAAAAAAgQAAAAAAAAAvAQAAAAAAADgCAAAAAAAAFAMAAAAAAAAcAQAAAAAAAIcBAAAAAAAARAEAAAAAAACpAAAAAAAAAFsBAAAAAAAAxgMAAAAAAABSAwAAAAAAAAUDAAAAAAAAdAEAAAAAAADCAAAAAAAAAOYDAAAAAAAA7gMAAAAAAAC7AQAAAAAAAL8CAAAAAAAA2gAAAAAAAAC4AQAAAAAAAEwBAAAAAAAAuAEAAAAAAAAAAgAAAAAAAE0BAAAAAAAAAwAAAAAAAACUAAAAAAAAAPoDAAAAAAAAoAEAAAAAAADEAgAAAAAAAK0CAAAAAAAAlwIAAAAAAABCAwAAAAAAAJICAAAAAAAARAAAAAAAAADlAQAAAAAAAHwAAAAAAAAAPwEAAAAAAADVAQAAAAAAAN8CAAAAAAAAngEAAAAAAACyAAAAAAAAAGECAAAAAAAAPwEAAAAAAAAwAAAAAAAAAIQBAAAAAAAAmwEAAAAAAABuAwAAAAAAAIQCAAAAAAAAbgMAAAAAAADRAAAAAAAAAHAAAAAAAAAArAIAAAAAAAAzAQAAAAAAALIAAAAAAAAAqAAAAAAAAACHAgAAAAAAAE4AAAAAAAAAtQIAAAAAAADpAgAAAAAAAL4BAAAAAAAANAEAAAAAAACtAQAAAAAAAHMAAAAAAAAAPAAAAAAAAABrAwAAAAAAAD0CAAAAAAAAigMAAAAAAAA/AQAAAAAAABMBAAAAAAAAWgAAAAAAAABzAAAAAAAAAE8BAAAAAAAAnwIAAAAAAADWAgAAAAAAAFABAAAAAAAARgMAAAAAAAC6AgAAAAAAAEsAAAAAAAAAQAEAAAAAAADyAQAAAAAAAPgDAAAAAAAADgEAAAAAAAAOAQAAAAAAAJICAAAAAAAASgMAAAAAAACjAwAAAAAAAHMAAAAAAAAAyAEAAAAAAAA3AQAAAAAAAG4DAAAAAAAA4gMAAAAAAACyAAAAAAAAAFQBAAAAAAAABAAAAAAAAAD2AwAAAAAAAMYBAAAAAAAAXAAAAAAAAAA8AgAAAAAAAAoAAAAAAAAA0QIAAAAAAABFAAAAAAAAAGABAAAAAAAASwMAAAAAAACOAgAAAAAAAJsDAAAAAAAAmQIAAAAAAABaAAAAAAAAAE0AAAAAAAAAmAMAAAAAAABuAwAAAAAAANkBAAAAAAAA6gAAAAAAAACoAQAAAAAAAEUAAAAAAAAAAAMAAAAAAAAuAgAAAAAAAC4CAAAAAAAAOwAAAAAAAAAQAwAAAAAAAE4AAAAAAAAADAAAAAAAAAAoAQAAAAAAALQCAAAAAAAANQIAAAAAAAAvAAAAAAAAADQAAAAAAAAAKwIAAAAAAAAZAgAAAAAAAAsDAAAAAAAAWgAAAAAAAABaAAAAAAAAAEwCAAAAAAAAcwAAAAAAAAA6AAAAAAAAAPgDAAAAAAAAgQEAAAAAAAAEAgAAAAAAACEAAAAAAAAAKAAAAAAAAADIAAAAAAAAADIDAAAAAAAAzgAAAAAAAABuAwAAAAAAAM4AAAAAAAAA0gAAAAAAAACrAgAAAAAAABsCAAAAAAAATQAAAAAAAACpAgAAAAAAAFIBAAAAAAAA4QEAAAAAAABKAgAAAAAAAHwBAAAAAAAA6gEAAAAAAAAxAAAAAAAAAK4AAAAAAAAArgAAAAAAAAAKAAAAAAAAAMoAAAAAAAAA/gAAAAAAAACsAgAAAAAAAHMAAAAAAAAACQMAAAAAAAAvAQAAAAAAAFsCAAAAAAAAFwMAAAAAAADoAwAAAAAAAG4DAAAAAAAA6gEAAAAAAAD9AwAAAAAAAJsDAAAAAAAAMgEAAAAAAAAlAAAAAAAAAB4AAAAAAAAAzwEAAAAAAACoAgAAAAAAAE0BAAAAAAAAFAEAAAAAAADIAQAAAAAAAOEDAAAAAAAATwAAAAAAAABRAgAAAAAAAIkBAAAAAAAAzAMAAAAAAAAJAwAAAAAAAD4AAAAAAAAA9AIAAAAAAADBAgAAAAAAAKUBAAAAAAAAYQIAAAAAAACEAwAAAAAAAKkBAAAAAAAA1wMAAAAAAADcAwAAAAAAADICAAAAAAAA5wIAAAAAAABrAwAAAAAAAPsDAAAAAAAA0QMAAAAAAACjAwAAAAAAANcDAAAAAAAAMwIAAAAAAACfAAAAAAAAACgDAAAAAAAAjAAAAAAAAAApAAAAAAAAAFoDAAAAAAAA2QIAAAAAAAABAAAAAAAAAMoCAAAAAAAA7AIAAAAAAACpAgAAAAAAAJoAAAAAAAAAUwEAAAAAAADAAQAAAAAAAHkDAAAAAAAAxwIAAAAAAAB/AwAAAAAAAMIBAAAAAAAAJAEAAAAAAADeAgAAAAAAAJADAAAAAAAAkQAAAAAAAADyAgAAAAAAAD8DAAAAAAAAggIAAAAAAAAuAwAAAAAAACgCAAAAAAAAKQMAAAAAAAAaAgAAAAAAAMUBAAAAAAAAegIAAAAAAAAkAgAAAAAAAGkCAAAAAAAAxAAAAAAAAACNAgAAAAAAAA4DAAAAAAAAaQIAAAAAAABQAAAAAAAAAIMDAAAAAAAAdwIAAAAAAABXAgAAAAAAAD0CAAAAAAAARAIAAAAAAAD+AQAAAAAAAMIAAAAAAAAA+wAAAAAAAACRAAAAAAAAAAYBAAAAAAAAxQAAAAAAAABhAgAAAAAAAFgAAAAAAAAAFQMAAAAAAAC5AwAAAAAAADwCAAAAAAAA6AAAAAAAAAAlAQAAAAAAANIDAAAAAAAAjgEAAAAAAADFAQAAAAAAALYBAAAAAAAArQAAAAAAAAB/AwAAAAAAABQBAAAAAAAAVAIAAAAAAAA5AgAAAAAAAPACAAAAAAAABgMAAAAAAABMAgAAAAAAACIDAAAAAAAA9AIAAAAAAACsAQAAAAAAAKQCAAAAAAAAmAEAAAAAAADYAgAAAAAAAOYDAAAAAAAA1wAAAAAAAAC3AwAAAAAAAHwBAAAAAAAACgMAAAAAAACaAgAAAAAAABMBAAAAAAAACgMAAAAAAAD4AwAAAAAAAAMBAAAAAAAAyAEAAAAAAABBAQAAAAAAAIsCAAAAAAAAjwIAAAAAAAAqAQAAAAAAALMDAAAAAAAAvAAAAAAAAACdAwAAAAAAAOkDAAAAAAAA/wEAAAAAAAC3AQAAAAAAAKUCAAAAAAAAgAIAAAAAAAA2AgAAAAAAAL0CAAAAAAAA6gMAAAAAAABIAQAAAAAAALIAAAAAAAAAygMAAAAAAAAYAgAAAAAAAM8DAAAAAAAA0wMAAAAAAACyAAAAAAAAAA0DAAAAAAAAcgAAAAAAAABuAQAAAAAAANQCAAAAAAAA1gIAAAAAAACUAAAAAAAAALkCAAAAAAAAbQIAAAAAAAAUAgAAAAAAABICAAAAAAAAwAIAAAAAAABDAgAAAAAAAJICAAAAAAAAiQAAAAAAAABXAgAAAAAAAC8DAAAAAAAA3AEAAAAAAABiAgAAAAAAADQDAAAAAAAAIgMAAAAAAADBAQAAAAAAAIUCAAAAAAAAZgEAAAAAAADwAAAAAAAAAEkBAAAAAAAA1gMAAAAAAAC6AwAAAAAAAMECAAAAAAAAIgEAAAAAAABbAQAAAAAAANMBAAAAAAAA0wIAAAAAAACKAgAAAAAAABEBAAAAAAAAHQEAAAAAAADwAgAAAAAAAKABAAAAAAAAIgMAAAAAAABNAQAAAAAAAH0CAAAAAAAAiQAAAAAAAADjAwAAAAAAAK0CAAAAAAAAowIAAAAAAAAHAAAAAAAAAE8CAAAAAAAAqQIAAAAAAACpAgAAAAAAAA4CAAAAAAAAagIAAAAAAAAiAwAAAAAAAHAAAAAAAAAAcAEAAAAAAACvAAAAAAAAAPQDAAAAAAAA/QIAAAAAAABqAwAAAAAAAI8DAAAAAAAA2AIAAAAAAADbAAAAAAAAAG4DAAAAAAAAuQIAAAAAAADCAQAAAAAAAGEDAAAAAAAAXQAAAAAAAAAVAQAAAAAAAJsBAAAAAAAAfwIAAAAAAADZAwAAAAAAAIoBAAAAAAAAfwEAAAAAAAA2AgAAAAAAAOUCAAAAAAAAvwAAAAAAAAC0AgAAAAAAAEoAAAAAAAAA6AMAAAAAAAAxAQAAAAAAAPUAAAAAAAAA6gIAAAAAAABPAgAAAAAAAD4BAAAAAAAAIQMAAAAAAAA0AwAAAAAAAGUCAAAAAAAAvgAAAAAAAAD+AgAAAAAAAHYDAAAAAAAAZgMAAAAAAACSAQAAAAAAALMBAAAAAAAAKAEAAAAAAADMAwAAAAAAAEADAAAAAAAA0gMAAAAAAABmAgAAAAAAAJMAAAAAAAAAmAIAAAAAAACqAgAAAAAAABkCAAAAAAAA7AEAAAAAAACcAAAAAAAAAIwBAAAAAAAA9QAAAAAAAACpAwAAAAAAAHQDAAAAAAAAhwAAAAAAAADZAgAAAAAAADUBAAAAAAAAPQMAAAAAAABqAgAAAAAAAFcAAAAAAAAA+QMAAAAAAACyAAAAAAAAAF4DAAAAAAAAxQAAAAAAAABsAwAAAAAAAPoAAAAAAAAA0QEAAAAAAABTAwAAAAAAAJsDAAAAAAAAdwAAAAAAAABMAgAAAAAAALcAAAAAAAAAWgAAAAAAAABfAwAAAAAAAJUCAAAAAAAAvAIAAAAAAABVAQAAAAAAAPgAAAAAAAAAqAIAAAAAAABIAQAAAAAAAI8DAAAAAAAANwAAAAAAAABEAwAAAAAAANcCAAAAAAAAOQEAAAAAAABSAwAAAAAAAGUCAAAAAAAAKwEAAAAAAACpAQAAAAAAAIkDAAAAAAAAZwIAAAAAAAAuAgAAAAAAAB4AAAAAAAAA/QIAAAAAAABFAAAAAAAAAEgDAAAAAAAAjgEAAAAAAAAHAwAAAAAAAPUAAAAAAAAAQQIAAAAAAACpAQAAAAAAABUDAAAAAAAAEQEAAAAAAADTAQAAAAAAANQDAAAAAAAAOwAAAAAAAABJAgAAAAAAAMYAAAAAAAAAkQIAAAAAAACOAAAAAAAAAJ0CAAAAAAAA8QAAAAAAAACwAAAAAAAAAJ4CAAAAAAAAbAMAAAAAAABjAgAAAAAAAPMCAAAAAAAA1gEAAAAAAAAfAQAAAAAAADMCAAAAAAAA6wEAAAAAAADuAQAAAAAAALUBAAAAAAAAOQEAAAAAAABUAAAAAAAAADcCAAAAAAAAKAAAAAAAAACVAAAAAAAAAL0DAAAAAAAAvQMAAAAAAAARAwAAAAAAAJwAAAAAAAAAfQAAAAAAAACDAwAAAAAAAPgCAAAAAAAAGwMAAAAAAACmAgAAAAAAAHECAAAAAAAA5gEAAAAAAACyAwAAAAAAAI0BAAAAAAAA1AEAAAAAAACRAgAAAAAAAPsDAAAAAAAA2QMAAAAAAACJAAAAAAAAAA0DAAAAAAAAFQMAAAAAAAC0AAAAAAAAADUBAAAAAAAAugMAAAAAAAA8AQAAAAAAAIMCAAAAAAAAKAMAAAAAAADuAQAAAAAAAK0DAAAAAAAA0QIAAAAAAADuAwAAAAAAAFoAAAAAAAAA1wMAAAAAAACiAQAAAAAAALQCAAAAAAAAigAAAAAAAAAoAwAAAAAAAIMCAAAAAAAAQQIAAAAAAACDAAAAAAAAAGsCAAAAAAAAjAEAAAAAAAB6AQAAAAAAABMDAAAAAAAAlQAAAAAAAACGAAAAAAAAALkCAAAAAAAAwgIAAAAAAAClAAAAAAAAANMBAAAAAAAAUgAAAAAAAAANAwAAAAAAAN4DAAAAAAAAngEAAAAAAACgAQAAAAAAAP4DAAAAAAAA2AEAAAAAAABLAQAAAAAAAO0CAAAAAAAAvwEAAAAAAADDAwAAAAAAACMBAAAAAAAAywEAAAAAAAAoAQAAAAAAANABAAAAAAAAVQIAAAAAAACdAAAAAAAAAIMCAAAAAAAASAMAAAAAAACZAgAAAAAAAMQCAAAAAAAA7gAAAAAAAADYAwAAAAAAAPwDAAAAAAAAhAEAAAAAAAA6AQAAAAAAAJMAAAAAAAAAswEAAAAAAAABAgAAAAAAAJoAAAAAAAAALAMAAAAAAAD0AgAAAAAAAJgAAAAAAAAAswMAAAAAAAA1AwAAAAAAACQDAAAAAAAAAQIAAAAAAAD/AQAAAAAAAKsDAAAAAAAACQEAAAAAAADCAgAAAAAAAGACAAAAAAAA2wEAAAAAAABpAQAAAAAAAI0BAAAAAAAAGQAAAAAAAAA1AwAAAAAAAN8DAAAAAAAAIwMAAAAAAACIAwAAAAAAAM0DAAAAAAAAaQIAAAAAAAD6AgAAAAAAAFwCAAAAAAAAlgMAAAAAAADLAAAAAAAAAIkDAAAAAAAAkgEAAAAAAABtAAAAAAAAAIYCAAAAAAAAPAEAAAAAAADCAgAAAAAAAOEDAAAAAAAAsgMAAAAAAADTAAAAAAAAACABAAAAAAAAkQEAAAAAAADAAgAAAAAAALQAAAAAAAAA8wMAAAAAAAB/AAAAAAAAAKcDAAAAAAAACAMAAAAAAAA7AQAAAAAAAHwBAAAAAAAA5QMAAAAAAAC9AQAAAAAAAGoBAAAAAAAAXwIAAAAAAADMAAAAAAAAAFICAAAAAAAAnwAAAAAAAACeAQAAAAAAAPEBAAAAAAAA/gEAAAAAAAAhAAAAAAAAAFABAAAAAAAAPQIAAAAAAACpAgAAAAAAAJMDAAAAAAAAYAEAAAAAAABIAwAAAAAAAK0CAAAAAAAAwgIAAAAAAACBAgAAAAAAAMQBAAAAAAAAnwIAAAAAAADpAgAAAAAAAHECAAAAAAAAaQIAAAAAAABcAgAAAAAAAFsAAAAAAAAAqQMAAAAAAAByAgAAAAAAAG0AAAAAAAAA/QMAAAAAAABiAAAAAAAAAJwDAAAAAAAAmQAAAAAAAADRAAAAAAAAAJ4BAAAAAAAAdQAAAAAAAACcAAAAAAAAAFMCAAAAAAAA+gIAAAAAAAAbAAAAAAAAAGoAAAAAAAAAowMAAAAAAAAJAAAAAAAAAAUDAAAAAAAApQMAAAAAAAA7AwAAAAAAAM0BAAAAAAAAPQIAAAAAAABvAwAAAAAAAOgCAAAAAAAANAMAAAAAAABXAwAAAAAAANkBAAAAAAAAQwMAAAAAAABrAwAAAAAAABQDAAAAAAAA7wEAAAAAAAD2AgAAAAAAAMEBAAAAAAAA3wIAAAAAAACfAgAAAAAAABgCAAAAAAAAagIAAAAAAAB4AQAAAAAAADwAAAAAAAAACgEAAAAAAABcAAAAAAAAAJ8AAAAAAAAA+QIAAAAAAAA8AAAAAAAAAKkAAAAAAAAAnwMAAAAAAAD3AQAAAAAAAFcAAAAAAAAAagAAAAAAAABEAAAAAAAAAA8BAAAAAAAAZwAAAAAAAABNAgAAAAAAAPQDAAAAAAAAUwMAAAAAAAD1AgAAAAAAAKIAAAAAAAAANQAAAAAAAADZAAAAAAAAALcDAAAAAAAAKwMAAAAAAAAPAAAAAAAAAFMCAAAAAAAA5gIAAAAAAACWAQAAAAAAAJsCAAAAAAAA9AEAAAAAAABhAAAAAAAAAA0BAAAAAAAAGQIAAAAAAACkAQAAAAAAAHYBAAAAAAAAHAAAAAAAAAAwAQAAAAAAAGwBAAAAAAAAKgIAAAAAAADXAgAAAAAAAHMCAAAAAAAA7gMAAAAAAACdAQAAAAAAAA4BAAAAAAAAVAAAAAAAAADbAgAAAAAAAMECAAAAAAAAHAAAAAAAAABrAAAAAAAAAHEBAAAAAAAAFQMAAAAAAAAJAwAAAAAAAOUCAAAAAAAADgAAAAAAAAB0AwAAAAAAAOEDAAAAAAAAIAIAAAAAAABPAAAAAAAAAF4BAAAAAAAAyAMAAAAAAABbAQAAAAAAAB0DAAAAAAAAbgMAAAAAAACRAgAAAAAAALYDAAAAAAAAoQIAAAAAAAAYAQAAAAAAAPcBAAAAAAAAnwEAAAAAAACOAgAAAAAAAFQCAAAAAAAAlAEAAAAAAAClAAAAAAAAACYBAAAAAAAAtwIAAAAAAAARAQAAAAAAAFkBAAAAAAAAqwAAAAAAAAB4AAAAAAAAACoCAAAAAAAAJgAAAAAAAAA7AAAAAAAAAKUBAAAAAAAA9AMAAAAAAABLAAAAAAAAAJoAAAAAAAAACgEAAAAAAACyAwAAAAAAACcBAAAAAAAAMgEAAAAAAABhAAAAAAAAAMECAAAAAAAACgAAAAAAAADgAQAAAAAAAIkBAAAAAAAAvwAAAAAAAABBAgAAAAAAANABAAAAAAAAowIAAAAAAACwAgAAAAAAALgCAAAAAAAAuQAAAAAAAADvAQAAAAAAACkBAAAAAAAANQEAAAAAAACVAgAAAAAAAAICAAAAAAAAggMAAAAAAACOAAAAAAAAAFwDAAAAAAAAEwAAAAAAAAAzAQAAAAAAALADAAAAAAAA9gMAAAAAAADEAQAAAAAAAL4BAAAAAAAA3wEAAAAAAADSAgAAAAAAAC4AAAAAAAAApAEAAAAAAACGAQAAAAAAAM4AAAAAAAAA8QMAAAAAAABtAQAAAAAAAAUBAAAAAAAAAQIAAAAAAACoAQAAAAAAANcAAAAAAAAAjwMAAAAAAACCAwAAAAAAABACAAAAAAAApAMAAAAAAAAMAQAAAAAAABgAAAAAAAAAdgAAAAAAAAD8AwAAAAAAAPABAAAAAAAAUAAAAAAAAAAMAwAAAAAAAPADAAAAAAAAjAIAAAAAAABXAQAAAAAAAOUAAAAAAAAAhQAAAAAAAACqAgAAAAAAAMACAAAAAAAAVgIAAAAAAAAxAQAAAAAAAKUBAAAAAAAAzgAAAAAAAAAZAgAAAAAAANEDAAAAAAAALAAAAAAAAAAaAQAAAAAAAHoAAAAAAAAAXwIAAAAAAABvAQAAAAAAAGABAAAAAAAAUAEAAAAAAAApAQAAAAAAACQCAAAAAAAASQMAAAAAAABEAAAAAAAAAOAAAAAAAAAArwIAAAAAAADwAwAAAAAAAFQAAAAAAAAAmAAAAAAAAADZAQAAAAAAAEsAAAAAAAAA0wIAAAAAAAB1AQAAAAAAAOgDAAAAAAAA9QAAAAAAAACBAwAAAAAAAEEAAAAAAAAAxAAAAAAAAABDAgAAAAAAANMAAAAAAAAAPAMAAAAAAADvAwAAAAAAAB8DAAAAAAAASQMAAAAAAADeAwAAAAAAAJ4DAAAAAAAAuwAAAAAAAACpAgAAAAAAAM4BAAAAAAAAEAEAAAAAAACPAAAAAAAAALgBAAAAAAAAQAEAAAAAAADbAAAAAAAAAGYCAAAAAAAAmQAAAAAAAACNAQAAAAAAAFUDAAAAAAAA5wAAAAAAAACzAgAAAAAAALQDAAAAAAAACgIAAAAAAAC0AQAAAAAAAJUDAAAAAAAAJAIAAAAAAAAnAwAAAAAAALIBAAAAAAAA6QIAAAAAAAC8AQAAAAAAAK0AAAAAAAAA7gEAAAAAAAD6AgAAAAAAAE8BAAAAAAAAUgMAAAAAAACkAgAAAAAAAJ4DAAAAAAAAjQAAAAAAAABmAwAAAAAAAGMBAAAAAAAADAAAAAAAAACDAgAAAAAAADQDAAAAAAAALgIAAAAAAACoAQAAAAAAAFQAAAAAAAAAXwEAAAAAAABmAAAAAAAAACwBAAAAAAAAdQEAAAAAAAANAAAAAAAAALcCAAAAAAAAIwMAAAAAAABAAwAAAAAAACwCAAAAAAAAugMAAAAAAAAbAgAAAAAAAB8CAAAAAAAAuAAAAAAAAABgAQAAAAAAAOUCAAAAAAAAygMAAAAAAACgAwAAAAAAAHIAAAAAAAAAkwMAAAAAAAA6AwAAAAAAAC8CAAAAAAAAJAMAAAAAAACxAwAAAAAAAIoDAAAAAAAAlgIAAAAAAADIAQAAAAAAAFQDAAAAAAAAZAIAAAAAAADnAQAAAAAAAAACAAAAAAAAqwMAAAAAAAA+AwAAAAAAADIDAAAAAAAA0QIAAAAAAADaAQAAAAAAAA8CAAAAAAAAsQMAAAAAAABeAAAAAAAAAOYCAAAAAAAAVQAAAAAAAADqAgAAAAAAALEBAAAAAAAAkgAAAAAAAACDAwAAAAAAAD8CAAAAAAAABAIAAAAAAABxAAAAAAAAAHYAAAAAAAAARgAAAAAAAACPAwAAAAAAABABAAAAAAAA5AEAAAAAAACHAwAAAAAAAFMAAAAAAAAAJgMAAAAAAAD8AAAAAAAAALQDAAAAAAAAuAIAAAAAAABBAwAAAAAAAFBLBwhfkrJ3wCYAAMAmAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABAAQgBUUkFJX1RSRS92ZXJzaW9uRkI+AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMwpQSwcI0Z5nVQIAAAACAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAfADEAVFJBSV9UUkUvLmRhdGEvc2VyaWFsaXphdGlvbl9pZEZCLQBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWloxNTcyODE5OTUxMTg1NDY0MzAyNjA1MDQ2NDYzOTc0MTE0NDc0Mzc3UEsHCG59rmwoAAAAKAAAAFBLAQIAAAAACAgAAAAAAABu7250XAEAAFwBAAARAAAAAAAAAAAAAAAAAAAAAABUUkFJX1RSRS9kYXRhLnBrbFBLAQIAAAAACAgAAAAAAAC379yDAQAAAAEAAAAYAAAAAAAAAAAAAAAAAKwBAABUUkFJX1RSRS8uZm9ybWF0X3ZlcnNpb25QSwECAAAAAAgIAAAAAAAAP3dx6QIAAAACAAAAGwAAAAAAAAAAAAAAAAARAgAAVFJBSV9UUkUvLnN0b3JhZ2VfYWxpZ25tZW50UEsBAgAAAAAICAAAAAAAAIU94xkGAAAABgAAABIAAAAAAAAAAAAAAAAAkgIAAFRSQUlfVFJFL2J5dGVvcmRlclBLAQIAAAAACAgAAAAAAABfkrJ3wCYAAMAmAAAPAAAAAAAAAAAAAAAAABYDAABUUkFJX1RSRS9kYXRhLzBQSwECAAAAAAgIAAAAAAAA0Z5nVQIAAAACAAAAEAAAAAAAAAAAAAAAAABQKgAAVFJBSV9UUkUvdmVyc2lvblBLAQIAAAAACAgAAAAAAABufa5sKAAAACgAAAAfAAAAAAAAAAAAAAAAANIqAABUUkFJX1RSRS8uZGF0YS9zZXJpYWxpemF0aW9uX2lkUEsGBiwAAAAAAAAAHgMtAAAAAAAAAAAABwAAAAAAAAAHAAAAAAAAANYBAAAAAAAAeCsAAAAAAABQSwYHAAAAAE4tAAAAAAAAAQAAAFBLBQYAAAAABwAHANYBAAB4KwAAAAA=', 'Phuong_Thy.pt': 'UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAVAA0AdHJhbl90aHkudG1wL2RhdGEucGtsRkIJAFpaWlpaWlpaWoACfXEAKFgOAAAAZm9ybWF0X3ZlcnNpb25xAUsBWBAAAAByZWZfYXVkaW9fdG9rZW5zcQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKTG9uZ1N0b3JhZ2UKcQVYAQAAADBxBlgDAAAAY3B1cQdNUAZ0cQhRSwBLCEvKhnEJS8pLAYZxColjY29sbGVjdGlvbnMKT3JkZXJlZERpY3QKcQspUnEMdHENUnEOWAgAAAByZWZfdGV4dHEPWJcAAABC4bufaSB0cm9uZyDEkcOzIGPDsyBnaGkgcsO1LCBuZ8aw4budaSBsdXnhu4duIHRow6BuaCBjw7RuZyBz4bq9IGPDsyB0aOG7gyBow7QgbcawYSBn4buNaSBnacOzLCDEkWnhu4F1IGtoaeG7g24gxJHGsOG7o2MgY+G6oyDDom0gYmluaCB0aGnDqm4gdMaw4bubbmcucRBYBwAAAHJlZl9ybXNxEUc/xiEJAAAAAHUuUEsHCEK1iaqRAQAAkQEAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAHAAlAHRyYW5fdGh5LnRtcC8uZm9ybWF0X3ZlcnNpb25GQiEAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMVBLBwi379yDAQAAAAEAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAAB8AMgB0cmFuX3RoeS50bXAvLnN0b3JhZ2VfYWxpZ25tZW50RkIuAFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlo2NFBLBwg/d3HpAgAAAAIAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABYAOgB0cmFuX3RoeS50bXAvYnl0ZW9yZGVyRkI2AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWmxpdHRsZVBLBwiFPeMZBgAAAAYAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABMAOQB0cmFuX3RoeS50bXAvZGF0YS8wRkI1AFpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpabwEAAAAAAADcAwAAAAAAAO8DAAAAAAAAnQAAAAAAAABUAQAAAAAAALABAAAAAAAAcAMAAAAAAABaAAAAAAAAAHUAAAAAAAAAhwMAAAAAAAD5AwAAAAAAAMwBAAAAAAAAzAEAAAAAAAB1AwAAAAAAABADAAAAAAAA1QIAAAAAAABdAQAAAAAAADEBAAAAAAAA/wAAAAAAAADQAgAAAAAAAP8AAAAAAAAAMQEAAAAAAADPAQAAAAAAADEBAAAAAAAAoAEAAAAAAACaAwAAAAAAAA0BAAAAAAAA/wEAAAAAAAARAgAAAAAAAPgCAAAAAAAAqwIAAAAAAABVAAAAAAAAAHgBAAAAAAAALQMAAAAAAADgAwAAAAAAAB4AAAAAAAAAiQIAAAAAAADaAgAAAAAAAE4CAAAAAAAA8gIAAAAAAABrAQAAAAAAAD8BAAAAAAAAmwIAAAAAAABuAwAAAAAAAAcBAAAAAAAAMwAAAAAAAAA8AAAAAAAAAKgBAAAAAAAAPAAAAAAAAAAFAgAAAAAAALkAAAAAAAAA4wAAAAAAAAAxAQAAAAAAALkDAAAAAAAAlAAAAAAAAABpAgAAAAAAAGkCAAAAAAAAPAAAAAAAAAAZAgAAAAAAACoBAAAAAAAAxQAAAAAAAACdAQAAAAAAAEUBAAAAAAAAjwMAAAAAAACSAAAAAAAAAHUAAAAAAAAARgAAAAAAAADRAQAAAAAAAKYDAAAAAAAApgMAAAAAAADjAwAAAAAAAB4AAAAAAAAAFAEAAAAAAACmAwAAAAAAAKQCAAAAAAAAJwMAAAAAAAD/AQAAAAAAAMwDAAAAAAAAzAEAAAAAAADTAQAAAAAAAHEAAAAAAAAAqQIAAAAAAABDAwAAAAAAAGwAAAAAAAAAkQEAAAAAAACmAwAAAAAAANwDAAAAAAAAdQAAAAAAAABJAwAAAAAAABkCAAAAAAAAaQIAAAAAAADWAAAAAAAAAKgBAAAAAAAABQIAAAAAAADAAgAAAAAAAMACAAAAAAAALAMAAAAAAAC5AAAAAAAAAAsAAAAAAAAAwgIAAAAAAADNAQAAAAAAAAMBAAAAAAAAaQMAAAAAAACZAQAAAAAAAKYDAAAAAAAAmwIAAAAAAABBAgAAAAAAAFQBAAAAAAAAhAIAAAAAAADoAgAAAAAAAK8BAAAAAAAAVgEAAAAAAABKAwAAAAAAAB0AAAAAAAAARwMAAAAAAADVAwAAAAAAAL0DAAAAAAAAbgMAAAAAAAAKAgAAAAAAAAcAAAAAAAAA2wMAAAAAAADgAwAAAAAAAOADAAAAAAAA4wMAAAAAAACVAAAAAAAAAHUAAAAAAAAA3AMAAAAAAACvAQAAAAAAACMBAAAAAAAA4AMAAAAAAACBAAAAAAAAAE4CAAAAAAAASgEAAAAAAAAKAAAAAAAAAB0DAAAAAAAAsgAAAAAAAACxAwAAAAAAAL0CAAAAAAAAFAEAAAAAAABaAAAAAAAAAEcDAAAAAAAA9QMAAAAAAAAQAwAAAAAAAHUBAAAAAAAALgIAAAAAAAAaAAAAAAAAAHkCAAAAAAAAxwIAAAAAAADPAQAAAAAAANIAAAAAAAAAVgIAAAAAAADHAgAAAAAAAAMCAAAAAAAA3gMAAAAAAACFAQAAAAAAAHEAAAAAAAAAfgMAAAAAAADgAwAAAAAAAOADAAAAAAAAIQMAAAAAAACrAwAAAAAAAPoDAAAAAAAApwEAAAAAAAArAQAAAAAAAOADAAAAAAAAqwMAAAAAAADmAQAAAAAAAH0CAAAAAAAAhgAAAAAAAADRAQAAAAAAAHMDAAAAAAAAhAIAAAAAAAAvAgAAAAAAAL4BAAAAAAAAzAMAAAAAAABDAwAAAAAAAKEDAAAAAAAAkQEAAAAAAADcAwAAAAAAAHUAAAAAAAAA1AEAAAAAAACxAgAAAAAAAC0DAAAAAAAApgMAAAAAAADUAQAAAAAAANQBAAAAAAAAKgIAAAAAAAASAAAAAAAAAC0DAAAAAAAA4wMAAAAAAABxAAAAAAAAAGMAAAAAAAAANAEAAAAAAACrAQAAAAAAAC0DAAAAAAAALQMAAAAAAAB1AwAAAAAAABADAAAAAAAABQEAAAAAAACQAwAAAAAAAMsBAAAAAAAAeQIAAAAAAADEAgAAAAAAAJgAAAAAAAAAlgMAAAAAAABrAAAAAAAAANgBAAAAAAAARQMAAAAAAABrAAAAAAAAAKIDAAAAAAAAngIAAAAAAAC+AQAAAAAAAI8BAAAAAAAAawAAAAAAAAAaAgAAAAAAADwDAAAAAAAA+wAAAAAAAABIAQAAAAAAAIwCAAAAAAAA5gMAAAAAAADuAAAAAAAAAKUCAAAAAAAApQIAAAAAAADDAwAAAAAAAOwAAAAAAAAAvgAAAAAAAACEAQAAAAAAAA8DAAAAAAAAFgEAAAAAAACRAAAAAAAAAPcCAAAAAAAALwIAAAAAAAAFAAAAAAAAABQAAAAAAAAA9AMAAAAAAACsAwAAAAAAAF8BAAAAAAAAvQAAAAAAAACrAAAAAAAAABQAAAAAAAAAyAMAAAAAAACDAgAAAAAAAM4AAAAAAAAAdQIAAAAAAACPAwAAAAAAAIkAAAAAAAAA+wAAAAAAAAArAgAAAAAAAMwCAAAAAAAA2QEAAAAAAABhAgAAAAAAAC0DAAAAAAAAdgAAAAAAAAA4AQAAAAAAAGgDAAAAAAAAHQIAAAAAAADLAAAAAAAAACEAAAAAAAAA3gEAAAAAAADgAQAAAAAAABYBAAAAAAAAQgIAAAAAAAAdAQAAAAAAAK0CAAAAAAAAFgEAAAAAAADhAgAAAAAAABoCAAAAAAAA3AIAAAAAAAB9AAAAAAAAALcCAAAAAAAALwIAAAAAAABCAAAAAAAAAF8BAAAAAAAA9gMAAAAAAADOAwAAAAAAAOoCAAAAAAAAZAIAAAAAAABWAwAAAAAAAAIAAAAAAAAAtwIAAAAAAACBAQAAAAAAAJcCAAAAAAAABQAAAAAAAABzAAAAAAAAADoCAAAAAAAAcwAAAAAAAACmAAAAAAAAAIcBAAAAAAAAVAEAAAAAAAAuAQAAAAAAADoAAAAAAAAAUQMAAAAAAADzAQAAAAAAABkCAAAAAAAA+wIAAAAAAADlAgAAAAAAAI4DAAAAAAAAxwAAAAAAAABoAQAAAAAAAMQCAAAAAAAAgQIAAAAAAABJAgAAAAAAAPACAAAAAAAAmQIAAAAAAABMAQAAAAAAANcBAAAAAAAAQgAAAAAAAACiAgAAAAAAABwBAAAAAAAA0gMAAAAAAABUAQAAAAAAAD8AAAAAAAAAgQEAAAAAAAAqAgAAAAAAAGYBAAAAAAAA/gMAAAAAAAC3AgAAAAAAAKsAAAAAAAAA+QEAAAAAAAC0AQAAAAAAAHEAAAAAAAAAMwAAAAAAAACGAgAAAAAAAP0CAAAAAAAALgEAAAAAAAD9AgAAAAAAAC4CAAAAAAAAvgEAAAAAAAB0AQAAAAAAAGUDAAAAAAAA3wAAAAAAAACcAwAAAAAAANcCAAAAAAAAxAIAAAAAAAAnAQAAAAAAAGUCAAAAAAAAggEAAAAAAACiAgAAAAAAACcAAAAAAAAAKgEAAAAAAABNAQAAAAAAAEIAAAAAAAAAPwAAAAAAAAA8AwAAAAAAAD8AAAAAAAAA5QAAAAAAAABbAQAAAAAAAAgAAAAAAAAAuwIAAAAAAABMAgAAAAAAALsCAAAAAAAAYQIAAAAAAADyAQAAAAAAAA8DAAAAAAAAJQEAAAAAAACgAQAAAAAAAOwCAAAAAAAAsgAAAAAAAAB7AgAAAAAAAHcAAAAAAAAATgAAAAAAAAD5AwAAAAAAAHIBAAAAAAAAOgIAAAAAAACJAAAAAAAAAHoAAAAAAAAAtAEAAAAAAACrAAAAAAAAAF8BAAAAAAAA/QIAAAAAAAB2AQAAAAAAADoBAAAAAAAAhwMAAAAAAACYAQAAAAAAAJ0CAAAAAAAABwAAAAAAAACdAgAAAAAAABIBAAAAAAAARwIAAAAAAABQAwAAAAAAAF8BAAAAAAAAVAEAAAAAAAASAQAAAAAAAN4DAAAAAAAAUAIAAAAAAAD5AwAAAAAAALIDAAAAAAAAkAMAAAAAAADZAQAAAAAAAFwAAAAAAAAAMgMAAAAAAAAHAAAAAAAAAGEDAAAAAAAAzgIAAAAAAAAIAAAAAAAAAIMCAAAAAAAApQEAAAAAAAAZAAAAAAAAABoCAAAAAAAAiwEAAAAAAABzAAAAAAAAAA8DAAAAAAAAWAMAAAAAAADVAwAAAAAAAOkCAAAAAAAAWgAAAAAAAAAuAwAAAAAAAK8BAAAAAAAA8wEAAAAAAABSAQAAAAAAADABAAAAAAAApwIAAAAAAABaAgAAAAAAAGcBAAAAAAAAUgAAAAAAAABzAAAAAAAAABACAAAAAAAA7AAAAAAAAADZAgAAAAAAAAkDAAAAAAAA1gIAAAAAAAB3AwAAAAAAAHoBAAAAAAAAxAEAAAAAAACtAwAAAAAAAKACAAAAAAAAsAIAAAAAAAAZAQAAAAAAALYCAAAAAAAAPQEAAAAAAABEAgAAAAAAAKACAAAAAAAA5QEAAAAAAADQAgAAAAAAAE4AAAAAAAAAbwAAAAAAAAAeAQAAAAAAAPEAAAAAAAAA+gIAAAAAAAAOAwAAAAAAANkDAAAAAAAAGQEAAAAAAAApAQAAAAAAAIECAAAAAAAArwMAAAAAAAAQAgAAAAAAAIsBAAAAAAAAAAEAAAAAAABjAgAAAAAAAA4AAAAAAAAA1QAAAAAAAACiAwAAAAAAAPwDAAAAAAAAAwEAAAAAAAB1AQAAAAAAAGsBAAAAAAAA3wIAAAAAAACyAQAAAAAAAPoBAAAAAAAAqwEAAAAAAACYAgAAAAAAANIBAAAAAAAA1QIAAAAAAAApAAAAAAAAAKkAAAAAAAAA4gIAAAAAAADYAgAAAAAAALwDAAAAAAAAIQEAAAAAAADZAQAAAAAAACkAAAAAAAAA9QIAAAAAAADOAgAAAAAAAPwCAAAAAAAAoQMAAAAAAACCAAAAAAAAAMsBAAAAAAAAqwIAAAAAAADSAwAAAAAAACIAAAAAAAAA4AEAAAAAAACeAQAAAAAAAFAAAAAAAAAAQwAAAAAAAABzAAAAAAAAAG4AAAAAAAAApAAAAAAAAAB0AwAAAAAAALABAAAAAAAANAMAAAAAAADpAwAAAAAAAMoAAAAAAAAA5QMAAAAAAABaAQAAAAAAAO8DAAAAAAAAPwMAAAAAAABkAQAAAAAAAPoCAAAAAAAAuAEAAAAAAADxAgAAAAAAAGQBAAAAAAAAYwMAAAAAAABlAQAAAAAAANIAAAAAAAAAugMAAAAAAACYAAAAAAAAAOUDAAAAAAAAbQAAAAAAAAB5AwAAAAAAALwDAAAAAAAAlQIAAAAAAABxAQAAAAAAAFEAAAAAAAAAUAAAAAAAAAAfAgAAAAAAAEoCAAAAAAAABAAAAAAAAADDAQAAAAAAAM4DAAAAAAAAJgAAAAAAAABOAQAAAAAAABkDAAAAAAAA2gMAAAAAAACjAwAAAAAAAEwBAAAAAAAAmgEAAAAAAADwAAAAAAAAAMQBAAAAAAAAHwAAAAAAAABuAwAAAAAAALAAAAAAAAAAiwMAAAAAAAD+AAAAAAAAAPYDAAAAAAAAMgEAAAAAAAA9AgAAAAAAAEwDAAAAAAAA1wAAAAAAAAB5AgAAAAAAAIYBAAAAAAAAyQAAAAAAAABzAAAAAAAAAPwCAAAAAAAAJwEAAAAAAAAQAgAAAAAAANoAAAAAAAAAowMAAAAAAABRAQAAAAAAAEgCAAAAAAAA4AMAAAAAAAD+AwAAAAAAAIkCAAAAAAAAqQEAAAAAAAB9AQAAAAAAAN8CAAAAAAAA1QIAAAAAAACcAQAAAAAAAKsBAAAAAAAAkAAAAAAAAAAEAgAAAAAAALABAAAAAAAAbgMAAAAAAABVAwAAAAAAAFUDAAAAAAAAVAEAAAAAAAAnAQAAAAAAANUBAAAAAAAAQAEAAAAAAAD3AgAAAAAAAGgBAAAAAAAA/gEAAAAAAADZAQAAAAAAAMEBAAAAAAAAbgMAAAAAAAAVAwAAAAAAAHQBAAAAAAAAJgEAAAAAAADcAgAAAAAAANoAAAAAAAAAYAAAAAAAAAATAQAAAAAAAGECAAAAAAAAogMAAAAAAAC6AQAAAAAAACIAAAAAAAAAKgMAAAAAAAAlAgAAAAAAADYDAAAAAAAAHQIAAAAAAADSAwAAAAAAAJ8BAAAAAAAAIQIAAAAAAABOAQAAAAAAAOUCAAAAAAAA6gIAAAAAAACNAAAAAAAAAH0DAAAAAAAAjwIAAAAAAAB4AAAAAAAAAD8AAAAAAAAAzwAAAAAAAAD4AwAAAAAAALcCAAAAAAAA3QMAAAAAAADmAAAAAAAAAL4BAAAAAAAAXAAAAAAAAADQAQAAAAAAALQAAAAAAAAAMAMAAAAAAAC5AAAAAAAAAD8DAAAAAAAA7gIAAAAAAAC7AgAAAAAAALoBAAAAAAAAMQIAAAAAAACMAQAAAAAAAPsAAAAAAAAAVAMAAAAAAAAmAQAAAAAAAHMDAAAAAAAAfQMAAAAAAADPAgAAAAAAAPkCAAAAAAAAmwMAAAAAAACnAQAAAAAAAP0DAAAAAAAAXgMAAAAAAACvAgAAAAAAAGUCAAAAAAAAjAEAAAAAAAACAgAAAAAAAE8CAAAAAAAAKwMAAAAAAAArAwAAAAAAAOQCAAAAAAAAvwAAAAAAAACGAAAAAAAAAN4CAAAAAAAAHAEAAAAAAABJAQAAAAAAACQBAAAAAAAAUAIAAAAAAAAKAwAAAAAAABcCAAAAAAAAOwIAAAAAAAAZAwAAAAAAALICAAAAAAAAqAMAAAAAAADhAAAAAAAAAJsBAAAAAAAAMgIAAAAAAAANAgAAAAAAAA4DAAAAAAAAtQMAAAAAAAAHAgAAAAAAAPUAAAAAAAAAAwMAAAAAAACBAQAAAAAAAGUDAAAAAAAAsAIAAAAAAABIAwAAAAAAAMIDAAAAAAAA/AAAAAAAAADWAAAAAAAAAG4AAAAAAAAAbQEAAAAAAAC1AQAAAAAAAJ4DAAAAAAAAhAEAAAAAAADzAAAAAAAAAAoCAAAAAAAAmAAAAAAAAAAVAQAAAAAAAEEAAAAAAAAAqAIAAAAAAABDAAAAAAAAANcDAAAAAAAATwAAAAAAAACqAAAAAAAAABkAAAAAAAAAbgMAAAAAAACRAQAAAAAAACsDAAAAAAAAhAMAAAAAAADvAwAAAAAAABYDAAAAAAAA9gIAAAAAAAAnAgAAAAAAABcCAAAAAAAAFwIAAAAAAAD0AgAAAAAAAPMAAAAAAAAAqAIAAAAAAADQAAAAAAAAAKMDAAAAAAAASAMAAAAAAADUAwAAAAAAAP4BAAAAAAAAqQEAAAAAAAClAQAAAAAAAHUCAAAAAAAAgAAAAAAAAAAjAwAAAAAAADIDAAAAAAAA8wAAAAAAAABBAAAAAAAAAB8BAAAAAAAABAMAAAAAAAB4AAAAAAAAABAAAAAAAAAAWwIAAAAAAAAYAQAAAAAAAD8DAAAAAAAA/wIAAAAAAABUAQAAAAAAAFkCAAAAAAAAtgEAAAAAAACTAAAAAAAAAIYDAAAAAAAAdgAAAAAAAADcAgAAAAAAAK4AAAAAAAAAmAAAAAAAAABKAwAAAAAAAEcAAAAAAAAA0gMAAAAAAAAcAQAAAAAAAFADAAAAAAAATgAAAAAAAAA3AwAAAAAAAL0DAAAAAAAARwAAAAAAAAC9AwAAAAAAANQAAAAAAAAACwEAAAAAAACfAwAAAAAAAE0CAAAAAAAAWwAAAAAAAABFAAAAAAAAAFwAAAAAAAAAjQEAAAAAAAAjAwAAAAAAALMBAAAAAAAAbQAAAAAAAABsAwAAAAAAAOgDAAAAAAAAnwMAAAAAAAArAwAAAAAAAJkCAAAAAAAAUQAAAAAAAAAbAwAAAAAAAK4CAAAAAAAAVAAAAAAAAAACAgAAAAAAAIoCAAAAAAAATwAAAAAAAABlAwAAAAAAALUDAAAAAAAA8wAAAAAAAACRAQAAAAAAANoAAAAAAAAAzAMAAAAAAAByAAAAAAAAAPgDAAAAAAAAmQIAAAAAAAD6AgAAAAAAAFsAAAAAAAAATgMAAAAAAAAhAAAAAAAAAJMBAAAAAAAAnwEAAAAAAABsAwAAAAAAADcCAAAAAAAAhAEAAAAAAACCAgAAAAAAANQAAAAAAAAA/wMAAAAAAAAeAQAAAAAAAD8DAAAAAAAA7QEAAAAAAABEAwAAAAAAAJsBAAAAAAAAzwAAAAAAAAD3AgAAAAAAAAQCAAAAAAAA+gMAAAAAAAC8AAAAAAAAADAAAAAAAAAAxwEAAAAAAACIAgAAAAAAAJMCAAAAAAAAeAIAAAAAAACNAAAAAAAAAHcDAAAAAAAADQIAAAAAAADAAQAAAAAAAGkCAAAAAAAAJwMAAAAAAAAZAwAAAAAAAMoCAAAAAAAAqwIAAAAAAAAVAAAAAAAAAM8CAAAAAAAAkwIAAAAAAADhAgAAAAAAABkBAAAAAAAAlwMAAAAAAAATAgAAAAAAAFEAAAAAAAAA0wEAAAAAAAB3AAAAAAAAAF8DAAAAAAAAVQAAAAAAAAAzAgAAAAAAAHcCAAAAAAAAMwMAAAAAAAArAwAAAAAAAJwBAAAAAAAA/AAAAAAAAAD2AgAAAAAAAAwBAAAAAAAAaQMAAAAAAAC6AAAAAAAAANEDAAAAAAAAbAAAAAAAAACPAgAAAAAAAKIAAAAAAAAAeQIAAAAAAACGAQAAAAAAAAkAAAAAAAAAawAAAAAAAAAOAgAAAAAAACcDAAAAAAAAUAAAAAAAAAATAQAAAAAAAGkAAAAAAAAAFgEAAAAAAADtAQAAAAAAAFwBAAAAAAAAKwMAAAAAAADqAwAAAAAAAD4BAAAAAAAAPwIAAAAAAADmAQAAAAAAAJoDAAAAAAAAPQAAAAAAAAASAwAAAAAAAG8DAAAAAAAAWwAAAAAAAACOAAAAAAAAAAUAAAAAAAAAdQIAAAAAAABBAwAAAAAAAGgAAAAAAAAAaAIAAAAAAABkAQAAAAAAAOwBAAAAAAAAqgMAAAAAAAB5AQAAAAAAAMQBAAAAAAAAvAAAAAAAAAA4AwAAAAAAAJ0DAAAAAAAATAAAAAAAAAB6AgAAAAAAAC4BAAAAAAAAdwMAAAAAAAB6AgAAAAAAAKECAAAAAAAAgwMAAAAAAABjAgAAAAAAAC0BAAAAAAAAPAEAAAAAAACEAQAAAAAAAEUDAAAAAAAAXgAAAAAAAACvAQAAAAAAACYBAAAAAAAABwEAAAAAAAD3AwAAAAAAAIsDAAAAAAAASQAAAAAAAABUAwAAAAAAAGcBAAAAAAAAvAEAAAAAAADSAQAAAAAAAOYDAAAAAAAArwMAAAAAAABoAQAAAAAAAJsDAAAAAAAAggEAAAAAAABQAwAAAAAAAPcCAAAAAAAAcgMAAAAAAAAuAQAAAAAAAI8BAAAAAAAAnQMAAAAAAAC2AwAAAAAAAFsAAAAAAAAAkAMAAAAAAAB4AgAAAAAAACkCAAAAAAAARgEAAAAAAACQAwAAAAAAAM4DAAAAAAAAiAIAAAAAAAD+AQAAAAAAAOMCAAAAAAAARgMAAAAAAABaAgAAAAAAANgAAAAAAAAAjwAAAAAAAABUAwAAAAAAAMUBAAAAAAAAawIAAAAAAADhAAAAAAAAACECAAAAAAAAAgEAAAAAAAD1AQAAAAAAALYDAAAAAAAAswEAAAAAAAAYAQAAAAAAAMkCAAAAAAAAXQEAAAAAAAD2AwAAAAAAACoCAAAAAAAAFQIAAAAAAADLAAAAAAAAAJ8DAAAAAAAAIgAAAAAAAAAeAwAAAAAAAFsDAAAAAAAAggMAAAAAAADMAQAAAAAAAJIBAAAAAAAA4wAAAAAAAADjAAAAAAAAAIoDAAAAAAAA4QIAAAAAAAC2AwAAAAAAAA0BAAAAAAAAeAMAAAAAAABfAwAAAAAAAPYDAAAAAAAAwwAAAAAAAADHAgAAAAAAAPwBAAAAAAAAGwIAAAAAAAB+AwAAAAAAAP4AAAAAAAAAPwIAAAAAAAAmAQAAAAAAANcBAAAAAAAAQgIAAAAAAADBAQAAAAAAACYBAAAAAAAAEwEAAAAAAAC/AwAAAAAAAAgBAAAAAAAAJAMAAAAAAABkAQAAAAAAAN8CAAAAAAAAXAIAAAAAAACzAgAAAAAAAOwBAAAAAAAAlQAAAAAAAAC+AQAAAAAAAIYBAAAAAAAAKwMAAAAAAAAeAQAAAAAAAO8BAAAAAAAAVQEAAAAAAADCAQAAAAAAABIDAAAAAAAASQIAAAAAAACtAQAAAAAAANACAAAAAAAAewIAAAAAAABeAAAAAAAAANYDAAAAAAAAlQIAAAAAAADLAgAAAAAAAJADAAAAAAAAWwEAAAAAAAALAQAAAAAAAIcBAAAAAAAA7QEAAAAAAAA4AwAAAAAAAKUAAAAAAAAA9AIAAAAAAABxAQAAAAAAABYAAAAAAAAAWgIAAAAAAACTAwAAAAAAAG8CAAAAAAAAJgMAAAAAAABwAwAAAAAAAJ4AAAAAAAAAWgIAAAAAAABCAgAAAAAAAIcDAAAAAAAAhwMAAAAAAAAyAgAAAAAAAMwBAAAAAAAAiwEAAAAAAACxAwAAAAAAAAICAAAAAAAAZwEAAAAAAAD7AQAAAAAAAMgDAAAAAAAA5AEAAAAAAAD5AwAAAAAAAEsBAAAAAAAA0gAAAAAAAAAbAAAAAAAAAGwDAAAAAAAAjgMAAAAAAACCAgAAAAAAAPkCAAAAAAAAQQMAAAAAAACJAAAAAAAAAMsDAAAAAAAAuQIAAAAAAAASAgAAAAAAAAECAAAAAAAAHgEAAAAAAAD1AwAAAAAAAKUCAAAAAAAAjQMAAAAAAABAAQAAAAAAAAIDAAAAAAAAUAEAAAAAAACIAwAAAAAAABIBAAAAAAAAsQAAAAAAAABuAAAAAAAAACABAAAAAAAALQEAAAAAAABSAwAAAAAAAHgDAAAAAAAATAIAAAAAAAAmAAAAAAAAAIcCAAAAAAAAkQIAAAAAAADyAwAAAAAAADgBAAAAAAAAYwMAAAAAAABgAwAAAAAAAOkCAAAAAAAAIgMAAAAAAAD+AQAAAAAAACABAAAAAAAA5gAAAAAAAADIAgAAAAAAAEIAAAAAAAAABwEAAAAAAAAKAgAAAAAAAO8DAAAAAAAALAMAAAAAAAAsAwAAAAAAAIgDAAAAAAAAiAMAAAAAAACcAgAAAAAAALsDAAAAAAAAmgMAAAAAAAAxAQAAAAAAAJ4BAAAAAAAAGQEAAAAAAAAkAAAAAAAAAIgDAAAAAAAA8QMAAAAAAAA2AQAAAAAAABsDAAAAAAAAaQAAAAAAAADNAgAAAAAAAFQAAAAAAAAAOQAAAAAAAADCAAAAAAAAADcDAAAAAAAAIwAAAAAAAADkAQAAAAAAAOEBAAAAAAAAkgAAAAAAAACwAgAAAAAAANIAAAAAAAAAXQEAAAAAAADXAwAAAAAAALcBAAAAAAAAlgAAAAAAAAAXAgAAAAAAAIUCAAAAAAAAFgIAAAAAAADZAAAAAAAAABkAAAAAAAAA2AEAAAAAAACpAAAAAAAAAM4DAAAAAAAAYgIAAAAAAAASAwAAAAAAAAQDAAAAAAAAHgAAAAAAAACnAgAAAAAAANoAAAAAAAAAFwIAAAAAAADaAwAAAAAAACsBAAAAAAAAlQEAAAAAAAAhAQAAAAAAAEMBAAAAAAAAEQEAAAAAAABDAQAAAAAAADABAAAAAAAA4QIAAAAAAAA4AgAAAAAAALoCAAAAAAAAWwEAAAAAAADvAQAAAAAAAKICAAAAAAAALwMAAAAAAADjAgAAAAAAABQDAAAAAAAADwMAAAAAAAD3AwAAAAAAAHYAAAAAAAAAwwMAAAAAAAAwAQAAAAAAAOcDAAAAAAAAUwMAAAAAAAATAwAAAAAAAEQBAAAAAAAA8gEAAAAAAACeAwAAAAAAACkAAAAAAAAAcgIAAAAAAAC4AQAAAAAAAOwCAAAAAAAARAAAAAAAAABjAgAAAAAAAJwCAAAAAAAAEAMAAAAAAACFAwAAAAAAAPkDAAAAAAAAHgMAAAAAAACqAgAAAAAAAMgBAAAAAAAAGgIAAAAAAAA8AQAAAAAAAKUBAAAAAAAALwIAAAAAAADIAAAAAAAAAPAAAAAAAAAAmQAAAAAAAAChAwAAAAAAAKYAAAAAAAAARAIAAAAAAACzAAAAAAAAAEUAAAAAAAAA1wEAAAAAAAB6AgAAAAAAAOwDAAAAAAAAigIAAAAAAABtAQAAAAAAAJ8DAAAAAAAAswIAAAAAAACYAgAAAAAAAMMCAAAAAAAAhAAAAAAAAACvAwAAAAAAAK4DAAAAAAAAbQAAAAAAAAC4AQAAAAAAAIkCAAAAAAAAaAEAAAAAAACWAwAAAAAAAMYBAAAAAAAA+wAAAAAAAADSAAAAAAAAAJYDAAAAAAAAlwAAAAAAAAByAgAAAAAAAFoBAAAAAAAA4gIAAAAAAAD7AwAAAAAAAPsAAAAAAAAAfAAAAAAAAACNAQAAAAAAAFQDAAAAAAAAMQEAAAAAAACzAwAAAAAAAEAAAAAAAAAArAEAAAAAAAB7AAAAAAAAAHwAAAAAAAAA1QIAAAAAAAAlAQAAAAAAACwDAAAAAAAAAQAAAAAAAADUAwAAAAAAAI8BAAAAAAAAmQAAAAAAAAB1AQAAAAAAAHEBAAAAAAAAXAAAAAAAAABgAwAAAAAAAHQDAAAAAAAAUgIAAAAAAADVAAAAAAAAAMEDAAAAAAAA/QAAAAAAAAAkAQAAAAAAAGcCAAAAAAAAwQAAAAAAAAApAAAAAAAAAH8CAAAAAAAAZwMAAAAAAABMAgAAAAAAAKwCAAAAAAAAkAEAAAAAAABwAAAAAAAAAKMBAAAAAAAAdwAAAAAAAAC5AwAAAAAAAIEAAAAAAAAAVgEAAAAAAADzAgAAAAAAACYDAAAAAAAAJwEAAAAAAAB/AgAAAAAAAHUBAAAAAAAAkwMAAAAAAACiAwAAAAAAAFIDAAAAAAAApQEAAAAAAABLAAAAAAAAAMcCAAAAAAAAgQMAAAAAAAC2AwAAAAAAAJUCAAAAAAAAcQMAAAAAAADhAQAAAAAAAHcBAAAAAAAAzgAAAAAAAABRAAAAAAAAAEsDAAAAAAAAGgEAAAAAAAC2AgAAAAAAAJMBAAAAAAAA7QAAAAAAAACnAAAAAAAAAKMCAAAAAAAARQAAAAAAAABCAgAAAAAAAAsBAAAAAAAA4gAAAAAAAAC9AAAAAAAAALwAAAAAAAAAgQMAAAAAAAB0AAAAAAAAAKUCAAAAAAAAzQIAAAAAAADEAQAAAAAAAM4AAAAAAAAAKgAAAAAAAADMAAAAAAAAAOoDAAAAAAAAKAIAAAAAAABBAgAAAAAAAFgCAAAAAAAAqwIAAAAAAAAxAwAAAAAAADAAAAAAAAAAAAAAAAAAAAAsAwAAAAAAAMwBAAAAAAAAVgIAAAAAAAB3AAAAAAAAANwAAAAAAAAA2AEAAAAAAADtAgAAAAAAAN0CAAAAAAAAsAIAAAAAAABvAAAAAAAAAF8AAAAAAAAADwIAAAAAAACeAgAAAAAAAIACAAAAAAAAfAAAAAAAAACEAgAAAAAAAEkCAAAAAAAAPQAAAAAAAAB4AwAAAAAAAEYBAAAAAAAAkgIAAAAAAAAzAQAAAAAAAKUAAAAAAAAA4gAAAAAAAAA8AgAAAAAAAHgCAAAAAAAAfgIAAAAAAABxAAAAAAAAALMCAAAAAAAAsAMAAAAAAAB8AQAAAAAAAI4BAAAAAAAACwEAAAAAAABhAgAAAAAAAN4BAAAAAAAAowAAAAAAAAAJAQAAAAAAAJMDAAAAAAAAugEAAAAAAADFAAAAAAAAAC8DAAAAAAAAjQAAAAAAAADEAgAAAAAAAIIAAAAAAAAAEQEAAAAAAABuAQAAAAAAAMsAAAAAAAAAYQIAAAAAAACdAwAAAAAAAEIAAAAAAAAABwIAAAAAAACyAAAAAAAAACsCAAAAAAAA4gAAAAAAAADtAQAAAAAAADgCAAAAAAAAIwIAAAAAAAB+AgAAAAAAAI4CAAAAAAAA9gMAAAAAAABYAwAAAAAAABkCAAAAAAAAEgIAAAAAAADsAAAAAAAAAEIDAAAAAAAACwEAAAAAAABpAgAAAAAAAEMAAAAAAAAAkwEAAAAAAAAGAQAAAAAAAOoCAAAAAAAAHQIAAAAAAACbAQAAAAAAACQCAAAAAAAAogAAAAAAAACGAAAAAAAAAA0BAAAAAAAA+AMAAAAAAAB4AwAAAAAAAFQDAAAAAAAAqgMAAAAAAAC7AQAAAAAAAM4DAAAAAAAA5wAAAAAAAAD2AAAAAAAAADECAAAAAAAABQIAAAAAAAAxAQAAAAAAAKYAAAAAAAAA9gAAAAAAAAAoAQAAAAAAAJEDAAAAAAAAIgIAAAAAAAC5AQAAAAAAAHUAAAAAAAAAOgAAAAAAAADvAgAAAAAAAH4AAAAAAAAAWgEAAAAAAAAzAwAAAAAAAGsDAAAAAAAA4gMAAAAAAABhAQAAAAAAACYAAAAAAAAAsgIAAAAAAADlAgAAAAAAACAAAAAAAAAAnwEAAAAAAADYAgAAAAAAAMkDAAAAAAAA1wAAAAAAAABNAgAAAAAAAD0BAAAAAAAAUQIAAAAAAABJAwAAAAAAAMIAAAAAAAAAbAEAAAAAAACRAwAAAAAAAOACAAAAAAAA5AIAAAAAAACNAwAAAAAAAGoBAAAAAAAA3QIAAAAAAAA7AwAAAAAAAA4CAAAAAAAAMAMAAAAAAAAJAwAAAAAAANQCAAAAAAAAigMAAAAAAAB/AgAAAAAAAIMAAAAAAAAA+AEAAAAAAACSAgAAAAAAAIQDAAAAAAAA5AEAAAAAAABSAQAAAAAAABkBAAAAAAAAOQMAAAAAAAAqAwAAAAAAAHkDAAAAAAAAuAIAAAAAAACKAQAAAAAAAH0BAAAAAAAAzAEAAAAAAACZAwAAAAAAAJcCAAAAAAAA5QEAAAAAAACEAgAAAAAAAJIDAAAAAAAAIgIAAAAAAAAQAQAAAAAAADAAAAAAAAAA4AIAAAAAAADIAQAAAAAAAL8AAAAAAAAAHgIAAAAAAACaAAAAAAAAAPcAAAAAAAAALAEAAAAAAADnAwAAAAAAAC4DAAAAAAAASwEAAAAAAABUAgAAAAAAACoCAAAAAAAAPAIAAAAAAACYAwAAAAAAALgDAAAAAAAADgIAAAAAAADBAAAAAAAAAE4DAAAAAAAAKwEAAAAAAACsAQAAAAAAAPEDAAAAAAAAsAEAAAAAAAAsAAAAAAAAACcAAAAAAAAAwQMAAAAAAACsAgAAAAAAAKMDAAAAAAAAdwMAAAAAAAD7AQAAAAAAAFUCAAAAAAAADQEAAAAAAADtAAAAAAAAAEkDAAAAAAAA4wEAAAAAAAAuAQAAAAAAAKwDAAAAAAAAMgIAAAAAAAAmAgAAAAAAAHIAAAAAAAAA2wIAAAAAAACFAQAAAAAAAKEDAAAAAAAAowIAAAAAAAADAgAAAAAAAIYBAAAAAAAAmQMAAAAAAACiAAAAAAAAAKkBAAAAAAAAEwEAAAAAAABZAgAAAAAAAP8BAAAAAAAAYQIAAAAAAACkAAAAAAAAAA8BAAAAAAAAEAEAAAAAAADzAAAAAAAAAIoDAAAAAAAAmgMAAAAAAABfAAAAAAAAAIUDAAAAAAAAgwMAAAAAAADGAAAAAAAAAPADAAAAAAAAcgIAAAAAAABBAAAAAAAAAHAAAAAAAAAAgAEAAAAAAADaAQAAAAAAAIAAAAAAAAAASQMAAAAAAADqAgAAAAAAAD0DAAAAAAAAhQMAAAAAAACaAAAAAAAAAPsAAAAAAAAAQgEAAAAAAAAlAQAAAAAAAOgAAAAAAAAAuAMAAAAAAAAwAQAAAAAAALsDAAAAAAAA1QMAAAAAAAClAgAAAAAAAIoDAAAAAAAATgIAAAAAAAAgAgAAAAAAAOcBAAAAAAAADAEAAAAAAAAvAgAAAAAAAAACAAAAAAAAOgEAAAAAAABfAAAAAAAAALIBAAAAAAAAQQEAAAAAAABkAgAAAAAAAIIAAAAAAAAAyQMAAAAAAACFAQAAAAAAAHYBAAAAAAAAcQIAAAAAAACPAAAAAAAAAEUDAAAAAAAAfgIAAAAAAAArAgAAAAAAAHwAAAAAAAAAWAMAAAAAAADpAwAAAAAAAP0AAAAAAAAAyQMAAAAAAACbAgAAAAAAADwDAAAAAAAAyAIAAAAAAABQAwAAAAAAAGcDAAAAAAAAqAEAAAAAAADHAwAAAAAAALABAAAAAAAAUgIAAAAAAADwAAAAAAAAACsBAAAAAAAA3gIAAAAAAADtAAAAAAAAAHYCAAAAAAAAcgMAAAAAAABcAQAAAAAAAEYDAAAAAAAA6QAAAAAAAAAiAgAAAAAAAAwCAAAAAAAAmAMAAAAAAAAwAQAAAAAAAKQDAAAAAAAAzAIAAAAAAAD0AAAAAAAAAIoCAAAAAAAAPgEAAAAAAACHAAAAAAAAAO4AAAAAAAAAigIAAAAAAAC6AgAAAAAAAI0BAAAAAAAAIwAAAAAAAACGAwAAAAAAAAsAAAAAAAAA5AMAAAAAAACdAwAAAAAAAAEDAAAAAAAAZAAAAAAAAAA+AgAAAAAAADABAAAAAAAAewMAAAAAAABIAAAAAAAAAAUDAAAAAAAAMAEAAAAAAAAJAgAAAAAAACICAAAAAAAAJAIAAAAAAADuAQAAAAAAAFEBAAAAAAAAZQIAAAAAAADhAgAAAAAAACcAAAAAAAAArAIAAAAAAACIAAAAAAAAAEICAAAAAAAAvgIAAAAAAAD2AgAAAAAAAOIDAAAAAAAABwEAAAAAAADmAQAAAAAAAFIDAAAAAAAAxwMAAAAAAAApAAAAAAAAAPoCAAAAAAAAdQMAAAAAAAByAwAAAAAAAHwCAAAAAAAABQMAAAAAAAA2AwAAAAAAAKgDAAAAAAAAhgEAAAAAAADHAwAAAAAAAK8DAAAAAAAAZgIAAAAAAAAPAAAAAAAAAEIAAAAAAAAASgMAAAAAAADlAQAAAAAAADYDAAAAAAAAAQEAAAAAAAAAAAAAAAAAAFBLBwgIZAMXgDIAAIAyAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAABQAPgB0cmFuX3RoeS50bXAvdmVyc2lvbkZCOgBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaMwpQSwcI0Z5nVQIAAAACAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAjAC0AdHJhbl90aHkudG1wLy5kYXRhL3NlcmlhbGl6YXRpb25faWRGQikAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWloxNTcyODE5OTUxMTg1NDY0MzAyNjA2MDM3NjkyMzAyMDM1ODI1NzY0UEsHCJm/ON8oAAAAKAAAAFBLAQIAAAAACAgAAAAAAABCtYmqkQEAAJEBAAAVAAAAAAAAAAAAAAAAAAAAAAB0cmFuX3RoeS50bXAvZGF0YS5wa2xQSwECAAAAAAgIAAAAAAAAt+/cgwEAAAABAAAAHAAAAAAAAAAAAAAAAADhAQAAdHJhbl90aHkudG1wLy5mb3JtYXRfdmVyc2lvblBLAQIAAAAACAgAAAAAAAA/d3HpAgAAAAIAAAAfAAAAAAAAAAAAAAAAAFECAAB0cmFuX3RoeS50bXAvLnN0b3JhZ2VfYWxpZ25tZW50UEsBAgAAAAAICAAAAAAAAIU94xkGAAAABgAAABYAAAAAAAAAAAAAAAAA0gIAAHRyYW5fdGh5LnRtcC9ieXRlb3JkZXJQSwECAAAAAAgIAAAAAAAACGQDF4AyAACAMgAAEwAAAAAAAAAAAAAAAABWAwAAdHJhbl90aHkudG1wL2RhdGEvMFBLAQIAAAAACAgAAAAAAADRnmdVAgAAAAIAAAAUAAAAAAAAAAAAAAAAAFA2AAB0cmFuX3RoeS50bXAvdmVyc2lvblBLAQIAAAAACAgAAAAAAACZvzjfKAAAACgAAAAjAAAAAAAAAAAAAAAAANI2AAB0cmFuX3RoeS50bXAvLmRhdGEvc2VyaWFsaXphdGlvbl9pZFBLBgYsAAAAAAAAAB4DLQAAAAAAAAAAAAcAAAAAAAAABwAAAAAAAADyAQAAAAAAAHg3AAAAAAAAUEsGBwAAAABqOQAAAAAAAAEAAABQSwUGAAAAAAcABwDyAQAAeDcAAAAA', 'To_Thanh.pt': 'UEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAPABMAa2UudG1wL2RhdGEucGtsRkIPAFpaWlpaWlpaWlpaWlpaWoACfXEAKFgOAAAAZm9ybWF0X3ZlcnNpb25xAUsBWBAAAAByZWZfYXVkaW9fdG9rZW5zcQJjdG9yY2guX3V0aWxzCl9yZWJ1aWxkX3RlbnNvcl92MgpxAygoWAcAAABzdG9yYWdlcQRjdG9yY2gKTG9uZ1N0b3JhZ2UKcQVYAQAAADBxBlgDAAAAY3B1cQdNMAZ0cQhRSwBLCEvGhnEJS8ZLAYZxColjY29sbGVjdGlvbnMKT3JkZXJlZERpY3QKcQspUnEMdHENUnEOWAgAAAByZWZfdGV4dHEPWMcAAABUw7RpIMSRaSB24bubaSDDtG5nIGPhuqMuIMOUbmcgY+G6oyBsw6AgdGjhu6Mgc8SDbiBj4bunYSBi4bqjbi4gQ+G6oyBi4bqjbiBjaOG7iSBjw7MgbcOsbmggw7RuZyBkw6FtIHbDoG8gcuG7q25nIG3hu5l0IG3DrG5oLiBRdWEga2jhu49pIGPDoWkga2hlIGNo4burbmcgbeG7mXQgdHLEg20gbcOpdCB0aMOsIMO0bmcgY+G6oyBk4burbmcgbOG6oWkucRBYBwAAAHJlZl9ybXNxEUc/wI3vQAAAAHUuUEsHCN36jX/BAQAAwQEAAFBLAwQAAAgIAAAAAAAAAAAAAAAAAAAAAAAAFgA7AGtlLnRtcC8uZm9ybWF0X3ZlcnNpb25GQjcAWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjFQSwcIt+/cgwEAAAABAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAZADgAa2UudG1wLy5zdG9yYWdlX2FsaWdubWVudEZCNABaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaNjRQSwcIP3dx6QIAAAACAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAAQAEAAa2UudG1wL2J5dGVvcmRlckZCPABaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpsaXR0bGVQSwcIhT3jGQYAAAAGAAAAUEsDBAAACAgAAAAAAAAAAAAAAAAAAAAAAAANAD8Aa2UudG1wL2RhdGEvMEZCOwBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWkEBAAAAAAAA9AAAAAAAAADNAgAAAAAAAIEAAAAAAAAAsgAAAAAAAAB7AgAAAAAAAIkCAAAAAAAAKQMAAAAAAAChAQAAAAAAANQDAAAAAAAAiQIAAAAAAACBAAAAAAAAAFoDAAAAAAAABgEAAAAAAACRAgAAAAAAAJQBAAAAAAAAOQIAAAAAAADpAgAAAAAAAIwDAAAAAAAApAIAAAAAAACjAwAAAAAAADMCAAAAAAAA7AMAAAAAAABYAwAAAAAAAO0CAAAAAAAA3gEAAAAAAABEAwAAAAAAAM8BAAAAAAAATwEAAAAAAAD6AAAAAAAAALoCAAAAAAAAXQEAAAAAAAAVAAAAAAAAAC0DAAAAAAAALQMAAAAAAAAtAwAAAAAAACsCAAAAAAAAIgIAAAAAAAC1AwAAAAAAAHsAAAAAAAAA/gEAAAAAAAAUAgAAAAAAACQCAAAAAAAAFAIAAAAAAAC/AgAAAAAAACICAAAAAAAAxAEAAAAAAABPAgAAAAAAAM0CAAAAAAAA6gEAAAAAAAAiAgAAAAAAAJABAAAAAAAAkQIAAAAAAAChAQAAAAAAAB8BAAAAAAAAmwMAAAAAAADsAAAAAAAAAK4BAAAAAAAAEwMAAAAAAABVAAAAAAAAANsDAAAAAAAAqgEAAAAAAADhAQAAAAAAAI0CAAAAAAAABQIAAAAAAABFAQAAAAAAAHwDAAAAAAAAAQIAAAAAAAAIAQAAAAAAAAgBAAAAAAAA5QMAAAAAAABbAwAAAAAAANIAAAAAAAAADAAAAAAAAADzAwAAAAAAAIkCAAAAAAAAPwEAAAAAAACGAAAAAAAAAMEDAAAAAAAA9gAAAAAAAACqAgAAAAAAAA0DAAAAAAAAEwMAAAAAAAAXAAAAAAAAALkCAAAAAAAAxAAAAAAAAAADAwAAAAAAAJQDAAAAAAAAZQEAAAAAAADRAgAAAAAAAHEAAAAAAAAAHgAAAAAAAAAQAAAAAAAAAPYBAAAAAAAACgMAAAAAAAADAwAAAAAAAAECAAAAAAAAfgAAAAAAAACUAQAAAAAAAGICAAAAAAAAoQEAAAAAAABXAQAAAAAAAL0CAAAAAAAA9AMAAAAAAADpAgAAAAAAALUDAAAAAAAALQMAAAAAAABjAQAAAAAAAGMAAAAAAAAAWwEAAAAAAADkAgAAAAAAAAgDAAAAAAAAHQAAAAAAAAD4AQAAAAAAANIDAAAAAAAA5AIAAAAAAACkAgAAAAAAANsDAAAAAAAAywEAAAAAAADkAgAAAAAAAOQCAAAAAAAA5gEAAAAAAABPAwAAAAAAAJIAAAAAAAAAmwMAAAAAAAB8AAAAAAAAAPcDAAAAAAAA9wMAAAAAAAB7AgAAAAAAAGsCAAAAAAAA/AEAAAAAAAD3AwAAAAAAAPcDAAAAAAAAewIAAAAAAABrAgAAAAAAAAEDAAAAAAAAFAEAAAAAAAAeAAAAAAAAAHEAAAAAAAAAIgIAAAAAAACtAAAAAAAAAAsBAAAAAAAAmQMAAAAAAACUAwAAAAAAAJ0DAAAAAAAA2wAAAAAAAACbAQAAAAAAABACAAAAAAAAcQAAAAAAAADKAwAAAAAAAJQBAAAAAAAAyAMAAAAAAADEAAAAAAAAANsAAAAAAAAADAIAAAAAAACyAQAAAAAAACgAAAAAAAAATAAAAAAAAABPAwAAAAAAACADAAAAAAAAyAMAAAAAAAC9AgAAAAAAACkDAAAAAAAAmgAAAAAAAACTAgAAAAAAALwBAAAAAAAAKQMAAAAAAABjAAAAAAAAACcDAAAAAAAA9QMAAAAAAADLAQAAAAAAAG4BAAAAAAAAwwEAAAAAAABNAwAAAAAAAAAAAAAAAAAACgEAAAAAAAC8AQAAAAAAAJIAAAAAAAAA5QAAAAAAAAB8AQAAAAAAACQCAAAAAAAAJgAAAAAAAABGAAAAAAAAAAgDAAAAAAAAxAIAAAAAAAAnAwAAAAAAAIoBAAAAAAAAWwAAAAAAAAATAwAAAAAAAA0DAAAAAAAAmQMAAAAAAACXAwAAAAAAACgBAAAAAAAABgMAAAAAAAD3AwAAAAAAAPcDAAAAAAAAewIAAAAAAABrAgAAAAAAAGIDAAAAAAAAgQMAAAAAAADaAAAAAAAAAEIAAAAAAAAAlgMAAAAAAAAGAQAAAAAAANcBAAAAAAAADAAAAAAAAACuAQAAAAAAAGYDAAAAAAAAeQAAAAAAAACiAAAAAAAAAFEAAAAAAAAAEAAAAAAAAACBAwAAAAAAAL4BAAAAAAAAYgIAAAAAAAAJAQAAAAAAAIwDAAAAAAAAVQMAAAAAAAA2AgAAAAAAACcCAAAAAAAAMQMAAAAAAAB3AwAAAAAAAGkDAAAAAAAA5AIAAAAAAAA6AAAAAAAAAGUBAAAAAAAAIQAAAAAAAADZAAAAAAAAAFABAAAAAAAAbAAAAAAAAAAxAgAAAAAAAIoAAAAAAAAAEAAAAAAAAABUAQAAAAAAAEUBAAAAAAAAWQAAAAAAAADjAAAAAAAAAPQCAAAAAAAA1AAAAAAAAAB8AgAAAAAAACcDAAAAAAAALwMAAAAAAAB2AgAAAAAAAIMDAAAAAAAApwMAAAAAAAALAQAAAAAAAAcBAAAAAAAA3AEAAAAAAACYAgAAAAAAADICAAAAAAAArwAAAAAAAAAiAgAAAAAAAF0AAAAAAAAAmwAAAAAAAADmAwAAAAAAAJYDAAAAAAAA8wIAAAAAAACMAgAAAAAAAIIBAAAAAAAATwIAAAAAAACwAwAAAAAAALAAAAAAAAAAOgAAAAAAAAAiAwAAAAAAAOQCAAAAAAAA7QMAAAAAAAAiAwAAAAAAADoAAAAAAAAAOgAAAAAAAABKAAAAAAAAAMABAAAAAAAAlQMAAAAAAACQAgAAAAAAAAwDAAAAAAAArAEAAAAAAADhAQAAAAAAAMEDAAAAAAAACQEAAAAAAABAAAAAAAAAADQDAAAAAAAALQAAAAAAAACiAwAAAAAAANYBAAAAAAAATQEAAAAAAAD/AQAAAAAAAP8DAAAAAAAAkAAAAAAAAAAQAAAAAAAAADMCAAAAAAAA3AIAAAAAAACfAwAAAAAAAEMCAAAAAAAAxQEAAAAAAABbAgAAAAAAACcDAAAAAAAAXgMAAAAAAADVAwAAAAAAABAAAAAAAAAAMwIAAAAAAAAnAAAAAAAAAGUAAAAAAAAApAAAAAAAAABPAgAAAAAAAF4DAAAAAAAA4QIAAAAAAABXAQAAAAAAAD8AAAAAAAAAdQMAAAAAAABJAwAAAAAAAEIAAAAAAAAABwMAAAAAAAAsAQAAAAAAAOECAAAAAAAARwIAAAAAAADVAwAAAAAAALcDAAAAAAAAKAAAAAAAAAB4AAAAAAAAAIAAAAAAAAAAFwIAAAAAAACdAQAAAAAAAAcAAAAAAAAA3gIAAAAAAAC/AgAAAAAAAHEAAAAAAAAAaAMAAAAAAAB2AQAAAAAAAN4DAAAAAAAA7wEAAAAAAAD7AgAAAAAAADkCAAAAAAAAEAEAAAAAAAAfAQAAAAAAAP0CAAAAAAAAkAAAAAAAAAAdAwAAAAAAAB0DAAAAAAAA9QAAAAAAAACQAAAAAAAAAJAAAAAAAAAApgAAAAAAAAAwAwAAAAAAAF8BAAAAAAAAkQIAAAAAAACUAAAAAAAAAOIBAAAAAAAAjAMAAAAAAABoAAAAAAAAAAcAAAAAAAAAdwIAAAAAAAAjAQAAAAAAAAwDAAAAAAAAqwIAAAAAAACAAAAAAAAAAPUCAAAAAAAAJAAAAAAAAABqAgAAAAAAAK0CAAAAAAAAdwAAAAAAAAAvAwAAAAAAAEYCAAAAAAAAbAEAAAAAAADVAwAAAAAAAGYBAAAAAAAAoQIAAAAAAAAXAgAAAAAAALgBAAAAAAAA9wIAAAAAAABCAwAAAAAAAHQCAAAAAAAASQEAAAAAAACKAgAAAAAAAOAAAAAAAAAAgQAAAAAAAAC9AAAAAAAAAP0CAAAAAAAANgIAAAAAAACTAwAAAAAAAGMBAAAAAAAApAAAAAAAAABpAAAAAAAAAEIAAAAAAAAARwMAAAAAAAA0AwAAAAAAABsBAAAAAAAA+gEAAAAAAACUAQAAAAAAAMwAAAAAAAAAsgIAAAAAAAD0AgAAAAAAABgCAAAAAAAAmQMAAAAAAAD4AgAAAAAAAN4BAAAAAAAAfAEAAAAAAABKAAAAAAAAACoAAAAAAAAAJwAAAAAAAABfAgAAAAAAAM4CAAAAAAAA/gAAAAAAAABaAAAAAAAAAFgDAAAAAAAAcwAAAAAAAAAeAwAAAAAAAMgAAAAAAAAAUgAAAAAAAADUAAAAAAAAAJ0CAAAAAAAAHwAAAAAAAAA/AgAAAAAAAPgCAAAAAAAAmAEAAAAAAACGAgAAAAAAALIAAAAAAAAAMAAAAAAAAACfAgAAAAAAADUAAAAAAAAAuAIAAAAAAAAJAQAAAAAAAAgAAAAAAAAAqwEAAAAAAAAvAgAAAAAAABAAAAAAAAAAEAAAAAAAAAD7AwAAAAAAAGMCAAAAAAAA3wIAAAAAAAAFAgAAAAAAAFgDAAAAAAAAbgMAAAAAAABvAAAAAAAAAKgCAAAAAAAAqQIAAAAAAABEAAAAAAAAAEwBAAAAAAAAsgAAAAAAAAC1AAAAAAAAAAMDAAAAAAAA8gMAAAAAAAD4AAAAAAAAAL8CAAAAAAAAyAEAAAAAAACgAgAAAAAAAOYBAAAAAAAAagMAAAAAAAA+AwAAAAAAALIAAAAAAAAAAAMAAAAAAAAeAwAAAAAAAI0BAAAAAAAANAAAAAAAAAAAAgAAAAAAALQBAAAAAAAAOQMAAAAAAACcAQAAAAAAAF0AAAAAAAAAEAMAAAAAAAAJAwAAAAAAAK8CAAAAAAAAygAAAAAAAAD5AAAAAAAAAFECAAAAAAAAZAEAAAAAAAAwAwAAAAAAAF0DAAAAAAAArwIAAAAAAAB3AAAAAAAAAL8AAAAAAAAAhgIAAAAAAAC9AAAAAAAAAJkAAAAAAAAATAEAAAAAAAAJAwAAAAAAAC8DAAAAAAAAKgIAAAAAAACbAQAAAAAAAAIAAAAAAAAABgAAAAAAAAAvAAAAAAAAAPMAAAAAAAAAmwAAAAAAAABuAwAAAAAAAM8DAAAAAAAAAAIAAAAAAACTAwAAAAAAAIMCAAAAAAAAWAMAAAAAAABJAQAAAAAAANUBAAAAAAAAEAMAAAAAAABqAAAAAAAAADUAAAAAAAAAogIAAAAAAAAeAQAAAAAAANoAAAAAAAAAOAIAAAAAAABSAgAAAAAAABUCAAAAAAAAkgAAAAAAAABgAAAAAAAAAEoDAAAAAAAAowMAAAAAAAB0AwAAAAAAAAkDAAAAAAAAEAMAAAAAAAABAgAAAAAAAAoBAAAAAAAANAIAAAAAAADgAgAAAAAAACIAAAAAAAAAwwEAAAAAAACcAAAAAAAAABcBAAAAAAAAbQAAAAAAAACTAQAAAAAAAFgDAAAAAAAAYgEAAAAAAABnAQAAAAAAADgCAAAAAAAA7gIAAAAAAAD1AgAAAAAAAMoDAAAAAAAArwAAAAAAAADIAgAAAAAAABEAAAAAAAAAgQEAAAAAAACjAwAAAAAAADUBAAAAAAAA6wAAAAAAAABNAAAAAAAAADsAAAAAAAAAJQIAAAAAAABSAQAAAAAAAK4DAAAAAAAAJgIAAAAAAADNAAAAAAAAACMAAAAAAAAANgAAAAAAAACSAgAAAAAAABECAAAAAAAAcwAAAAAAAACSAgAAAAAAAOgCAAAAAAAAZwIAAAAAAAC4AQAAAAAAAFgDAAAAAAAAHwMAAAAAAAACAwAAAAAAAN8AAAAAAAAAQgEAAAAAAAAFAgAAAAAAAGcBAAAAAAAAVAAAAAAAAAAXAQAAAAAAADAAAAAAAAAATAEAAAAAAADrAwAAAAAAAFAAAAAAAAAAnAMAAAAAAACYAAAAAAAAAJMBAAAAAAAAGgEAAAAAAADrAwAAAAAAAIsBAAAAAAAAngEAAAAAAAB4AQAAAAAAAB8AAAAAAAAAmwMAAAAAAADbAQAAAAAAABYBAAAAAAAA1AAAAAAAAAAiAgAAAAAAAKIDAAAAAAAAzgEAAAAAAADOAQAAAAAAAEoCAAAAAAAA0gEAAAAAAACdAAAAAAAAADwAAAAAAAAAzgAAAAAAAACBAAAAAAAAACEBAAAAAAAAAAMAAAAAAABKAAAAAAAAAOgDAAAAAAAAPAAAAAAAAABnAgAAAAAAAJUDAAAAAAAAhgEAAAAAAAC3AQAAAAAAAOsAAAAAAAAANQEAAAAAAADrAAAAAAAAADoBAAAAAAAAtAMAAAAAAAAvAQAAAAAAALYBAAAAAAAALQEAAAAAAAD9AwAAAAAAACADAAAAAAAAHwMAAAAAAACHAQAAAAAAAPsAAAAAAAAAEAMAAAAAAABaAAAAAAAAAMADAAAAAAAA9gMAAAAAAADnAwAAAAAAAPAAAAAAAAAADwEAAAAAAACLAAAAAAAAAC4AAAAAAAAAhQMAAAAAAABCAAAAAAAAAKUAAAAAAAAAeQMAAAAAAACvAgAAAAAAAKMCAAAAAAAA4gEAAAAAAABXAAAAAAAAAJoCAAAAAAAAigAAAAAAAADHAAAAAAAAAGMDAAAAAAAApgAAAAAAAADbAwAAAAAAALwAAAAAAAAAlwMAAAAAAABtAwAAAAAAAEQDAAAAAAAAqQAAAAAAAADKAQAAAAAAAIgDAAAAAAAAhQMAAAAAAACLAgAAAAAAAMUBAAAAAAAARQIAAAAAAACaAwAAAAAAAEwBAAAAAAAADwMAAAAAAADgAAAAAAAAAHgBAAAAAAAAEwEAAAAAAABfAAAAAAAAAPUDAAAAAAAAyQEAAAAAAACDAwAAAAAAADICAAAAAAAAdwIAAAAAAABeAwAAAAAAAAoDAAAAAAAADgEAAAAAAACSAAAAAAAAAHYCAAAAAAAA0QIAAAAAAACtAQAAAAAAABsCAAAAAAAA1wAAAAAAAACdAAAAAAAAAN0DAAAAAAAADQMAAAAAAACAAwAAAAAAAIIAAAAAAAAA5QEAAAAAAACAAAAAAAAAAH8AAAAAAAAAbQMAAAAAAAAZAwAAAAAAABoCAAAAAAAAGQAAAAAAAABmAQAAAAAAADMCAAAAAAAA0QIAAAAAAACwAQAAAAAAAKcBAAAAAAAAxAAAAAAAAABNAAAAAAAAAJoDAAAAAAAA8gIAAAAAAAA9AQAAAAAAAB8BAAAAAAAAqgEAAAAAAABSAwAAAAAAAIsCAAAAAAAAxQAAAAAAAAC0AwAAAAAAAEsCAAAAAAAA0wAAAAAAAAARAQAAAAAAABYBAAAAAAAAqAAAAAAAAACjAAAAAAAAAOcDAAAAAAAA3QEAAAAAAACPAAAAAAAAAAMAAAAAAAAAgQAAAAAAAACIAwAAAAAAAL4CAAAAAAAAuQAAAAAAAAAWAwAAAAAAAIUDAAAAAAAAawMAAAAAAAC2AgAAAAAAADYAAAAAAAAAswEAAAAAAAAcAgAAAAAAAAMCAAAAAAAAIwAAAAAAAAD9AgAAAAAAAIMAAAAAAAAAigMAAAAAAAD3AwAAAAAAALwAAAAAAAAAnwEAAAAAAAC0AQAAAAAAACEAAAAAAAAA8QIAAAAAAAAGAwAAAAAAAL4DAAAAAAAAXwEAAAAAAABXAQAAAAAAAFMAAAAAAAAABAAAAAAAAAC8AwAAAAAAADYDAAAAAAAAowIAAAAAAAAjAgAAAAAAABMBAAAAAAAAbwIAAAAAAAABAAAAAAAAAPsBAAAAAAAAqwIAAAAAAAApAwAAAAAAAC4CAAAAAAAAoAEAAAAAAAAyAgAAAAAAABEBAAAAAAAA9gAAAAAAAABPAQAAAAAAAEwCAAAAAAAAygIAAAAAAAAbAgAAAAAAAB8DAAAAAAAAbQEAAAAAAADyAgAAAAAAAJAAAAAAAAAA5AMAAAAAAAC6AgAAAAAAAB0BAAAAAAAAAAAAAAAAAADiAgAAAAAAAJQBAAAAAAAACgIAAAAAAABDAQAAAAAAAIkDAAAAAAAAvAAAAAAAAAC/AgAAAAAAADEBAAAAAAAA5wMAAAAAAADrAwAAAAAAAFMDAAAAAAAALQAAAAAAAAAuAAAAAAAAAN8BAAAAAAAAqgAAAAAAAADJAwAAAAAAAP4DAAAAAAAAzQMAAAAAAADVAwAAAAAAAPAAAAAAAAAASQIAAAAAAABgAQAAAAAAACIDAAAAAAAA7QEAAAAAAAAgAQAAAAAAAJ8CAAAAAAAA4wAAAAAAAACVAQAAAAAAAPQBAAAAAAAAaQAAAAAAAABEAwAAAAAAALsBAAAAAAAA6AIAAAAAAADCAgAAAAAAAOICAAAAAAAA9gAAAAAAAADyAAAAAAAAAJUAAAAAAAAA1gMAAAAAAABwAwAAAAAAAB8DAAAAAAAA5gMAAAAAAAAeAQAAAAAAAMsAAAAAAAAAiQMAAAAAAADkAwAAAAAAABYCAAAAAAAA/wMAAAAAAABeAwAAAAAAAJ0BAAAAAAAA3QMAAAAAAADTAgAAAAAAAPsCAAAAAAAAmAMAAAAAAADjAAAAAAAAAAkAAAAAAAAAnQMAAAAAAAAuAgAAAAAAAMEDAAAAAAAA+QAAAAAAAACNAwAAAAAAACUBAAAAAAAA+gEAAAAAAADhAQAAAAAAAAEAAAAAAAAAaQEAAAAAAACxAAAAAAAAAD0AAAAAAAAALQEAAAAAAACWAQAAAAAAAFkBAAAAAAAAcQIAAAAAAABjAQAAAAAAALgBAAAAAAAAVAMAAAAAAAByAQAAAAAAABAAAAAAAAAAggEAAAAAAAAJAgAAAAAAAK8BAAAAAAAArwEAAAAAAAAmAgAAAAAAAEoBAAAAAAAAzwMAAAAAAACkAgAAAAAAAOICAAAAAAAAvwIAAAAAAAAZAgAAAAAAAPIAAAAAAAAAIAAAAAAAAAAHAAAAAAAAALwBAAAAAAAAkAAAAAAAAADjAAAAAAAAAIwCAAAAAAAA+QMAAAAAAAByAQAAAAAAAM0CAAAAAAAAxgMAAAAAAABGAgAAAAAAAGcCAAAAAAAA5QAAAAAAAABbAAAAAAAAAPEDAAAAAAAA+AAAAAAAAADPAQAAAAAAABQAAAAAAAAArgEAAAAAAABpAQAAAAAAAP0BAAAAAAAAkwAAAAAAAAASAgAAAAAAANEDAAAAAAAAJwAAAAAAAADgAQAAAAAAAE8DAAAAAAAAPgAAAAAAAADIAAAAAAAAAA0DAAAAAAAAnwEAAAAAAACXAAAAAAAAABUBAAAAAAAA5gIAAAAAAAASAAAAAAAAAGoCAAAAAAAA1gAAAAAAAABtAQAAAAAAAE8CAAAAAAAAJAAAAAAAAACSAQAAAAAAAFMAAAAAAAAAPgAAAAAAAABMAwAAAAAAABwCAAAAAAAA9gEAAAAAAACaAAAAAAAAAHoBAAAAAAAAOAIAAAAAAACgAgAAAAAAAMsDAAAAAAAAoQIAAAAAAACCAQAAAAAAAL0CAAAAAAAAvwEAAAAAAAANAwAAAAAAAE8DAAAAAAAAbwIAAAAAAABGAQAAAAAAALcAAAAAAAAAiQAAAAAAAADsAgAAAAAAAGYDAAAAAAAA7wEAAAAAAAAnAQAAAAAAAIABAAAAAAAAnAMAAAAAAAB4AwAAAAAAANcCAAAAAAAAOgMAAAAAAABgAgAAAAAAAFMDAAAAAAAAbQEAAAAAAAC7AQAAAAAAAFkAAAAAAAAASQMAAAAAAACPAgAAAAAAAAEDAAAAAAAA2AMAAAAAAABqAQAAAAAAAEsDAAAAAAAAYwMAAAAAAAABAgAAAAAAAJ0AAAAAAAAAXgMAAAAAAADwAAAAAAAAALcAAAAAAAAA7wAAAAAAAAA8AQAAAAAAAPACAAAAAAAAtwMAAAAAAABGAgAAAAAAAOADAAAAAAAAmQEAAAAAAAB7AgAAAAAAAOICAAAAAAAA+QIAAAAAAAD7AwAAAAAAACgAAAAAAAAAvwAAAAAAAACCAAAAAAAAAC4BAAAAAAAAlQMAAAAAAACoAQAAAAAAAHUBAAAAAAAACQEAAAAAAABFAAAAAAAAAPoBAAAAAAAAggAAAAAAAACyAQAAAAAAAOEDAAAAAAAAVAMAAAAAAACBAQAAAAAAAC8DAAAAAAAAGAMAAAAAAADYAQAAAAAAANIAAAAAAAAA5gAAAAAAAAAYAQAAAAAAAPIDAAAAAAAAwwIAAAAAAAChAgAAAAAAAF8CAAAAAAAA7wEAAAAAAABMAAAAAAAAAMgBAAAAAAAAxgMAAAAAAADmAgAAAAAAAFwDAAAAAAAA9gMAAAAAAABSAAAAAAAAAAICAAAAAAAAWwEAAAAAAABdAgAAAAAAAG4DAAAAAAAAXQIAAAAAAADFAwAAAAAAAI4AAAAAAAAAwgMAAAAAAAAqAAAAAAAAABAAAAAAAAAADAIAAAAAAAAnAgAAAAAAAG0DAAAAAAAA+wAAAAAAAAA7AQAAAAAAAEIBAAAAAAAA3AEAAAAAAADYAwAAAAAAAEUBAAAAAAAAZwMAAAAAAAC9AgAAAAAAAHMCAAAAAAAA2wIAAAAAAACtAgAAAAAAAHQCAAAAAAAADQIAAAAAAADQAQAAAAAAAOQDAAAAAAAA2gEAAAAAAABZAgAAAAAAAB0CAAAAAAAAnQEAAAAAAAArAAAAAAAAADoBAAAAAAAAVAEAAAAAAABhAwAAAAAAANsCAAAAAAAAygIAAAAAAACgAwAAAAAAAKMBAAAAAAAApQAAAAAAAABHAAAAAAAAAP0CAAAAAAAAKAIAAAAAAACvAwAAAAAAAIICAAAAAAAAhAAAAAAAAABuAwAAAAAAABsDAAAAAAAAtQMAAAAAAACQAgAAAAAAAG0DAAAAAAAAzQIAAAAAAAAvAwAAAAAAANgCAAAAAAAADQAAAAAAAAD+AQAAAAAAAAUDAAAAAAAAdAIAAAAAAACEAwAAAAAAAOwCAAAAAAAAEwEAAAAAAADNAQAAAAAAAMQAAAAAAAAAlAIAAAAAAACBAgAAAAAAAD0DAAAAAAAAYgEAAAAAAAD4AgAAAAAAAPgCAAAAAAAArQAAAAAAAACfAQAAAAAAANUBAAAAAAAA1gAAAAAAAAArAQAAAAAAAEcAAAAAAAAAmAAAAAAAAAA5AQAAAAAAAOAAAAAAAAAAtwIAAAAAAAArAgAAAAAAAEkDAAAAAAAALAIAAAAAAABzAwAAAAAAAEsBAAAAAAAAjAIAAAAAAAD9AwAAAAAAAB0AAAAAAAAANAEAAAAAAADLAgAAAAAAAJkAAAAAAAAA9gIAAAAAAABjAwAAAAAAAGsDAAAAAAAA1AMAAAAAAACXAAAAAAAAAIAAAAAAAAAAxgAAAAAAAAATAAAAAAAAAGsAAAAAAAAAqAAAAAAAAACFAgAAAAAAAFEAAAAAAAAAoAIAAAAAAADSAgAAAAAAAJkCAAAAAAAAYQAAAAAAAABsAgAAAAAAAD0CAAAAAAAAUgMAAAAAAADkAQAAAAAAAD8AAAAAAAAAzQEAAAAAAACDAQAAAAAAANcDAAAAAAAAzwAAAAAAAACPAgAAAAAAABADAAAAAAAAlQIAAAAAAAASAAAAAAAAAOMCAAAAAAAA7QIAAAAAAADgAAAAAAAAACcBAAAAAAAAbwEAAAAAAADKAAAAAAAAAL8BAAAAAAAAoAIAAAAAAAAaAgAAAAAAAD8AAAAAAAAAswMAAAAAAADeAQAAAAAAAAQDAAAAAAAAUgMAAAAAAAD9AAAAAAAAAIMCAAAAAAAAGgIAAAAAAABtAAAAAAAAANQBAAAAAAAAHgMAAAAAAAA/AAAAAAAAAKcBAAAAAAAAOAAAAAAAAACrAQAAAAAAAHIDAAAAAAAALAMAAAAAAABWAQAAAAAAAPMAAAAAAAAAcgMAAAAAAAByAwAAAAAAAMcBAAAAAAAAVgEAAAAAAAAsAwAAAAAAACcBAAAAAAAAxwEAAAAAAABMAgAAAAAAAAABAAAAAAAAVAAAAAAAAAA1AAAAAAAAABYCAAAAAAAArQAAAAAAAADEAwAAAAAAAOcCAAAAAAAAHAMAAAAAAAAWAAAAAAAAADYAAAAAAAAAfgIAAAAAAAC5AgAAAAAAAJoBAAAAAAAA6gAAAAAAAABpAAAAAAAAAD8AAAAAAAAA9gMAAAAAAABCAAAAAAAAAAIBAAAAAAAAEwAAAAAAAADMAgAAAAAAAI0DAAAAAAAAZwIAAAAAAACgAQAAAAAAAOAAAAAAAAAADAIAAAAAAADaAwAAAAAAAMwCAAAAAAAA4AAAAAAAAADgAAAAAAAAAAMDAAAAAAAASQAAAAAAAADzAAAAAAAAAPwBAAAAAAAAWwIAAAAAAACVAgAAAAAAAHACAAAAAAAA/gIAAAAAAADaAwAAAAAAAEECAAAAAAAAoAEAAAAAAADbAgAAAAAAAAUDAAAAAAAADwIAAAAAAACXAwAAAAAAAEcBAAAAAAAA+gEAAAAAAACCAwAAAAAAACwAAAAAAAAAxQAAAAAAAADIAQAAAAAAANADAAAAAAAAYgAAAAAAAADQAQAAAAAAAHMAAAAAAAAArQAAAAAAAAAsAwAAAAAAAD8DAAAAAAAAVgEAAAAAAAD9AwAAAAAAAK8BAAAAAAAArQMAAAAAAACLAwAAAAAAAOACAAAAAAAA1wIAAAAAAABUAQAAAAAAAN0CAAAAAAAAzwIAAAAAAAADAwAAAAAAALQBAAAAAAAAWQIAAAAAAADFAgAAAAAAAEsAAAAAAAAA9QAAAAAAAAASAwAAAAAAAOoDAAAAAAAA3gIAAAAAAACeAAAAAAAAABEDAAAAAAAAAAAAAAAAAADuAQAAAAAAADYCAAAAAAAAuAAAAAAAAAB2AAAAAAAAAL0DAAAAAAAAXwAAAAAAAAD1AQAAAAAAAFQAAAAAAAAAuQAAAAAAAAC6AwAAAAAAAJwBAAAAAAAASAEAAAAAAAD9AAAAAAAAADIDAAAAAAAAbgEAAAAAAAAwAgAAAAAAAOoAAAAAAAAAKQMAAAAAAAD/AAAAAAAAAJgBAAAAAAAAagEAAAAAAABvAQAAAAAAAO0CAAAAAAAAUQIAAAAAAABNAQAAAAAAABAAAAAAAAAAOQEAAAAAAAAYAQAAAAAAAIIDAAAAAAAASAEAAAAAAACvAgAAAAAAAEoDAAAAAAAAagIAAAAAAACNAQAAAAAAAAMBAAAAAAAAsgIAAAAAAADjAQAAAAAAAFcBAAAAAAAAuwAAAAAAAAALAAAAAAAAAOACAAAAAAAAnQMAAAAAAABtAwAAAAAAAK4BAAAAAAAA6AIAAAAAAAAxAwAAAAAAAC8AAAAAAAAAUQMAAAAAAAAXAwAAAAAAAFEDAAAAAAAAwgMAAAAAAABEAAAAAAAAAFEDAAAAAAAANwIAAAAAAABRAwAAAAAAAB8DAAAAAAAApAMAAAAAAAB0AAAAAAAAAF4BAAAAAAAAGQAAAAAAAABrAwAAAAAAADMBAAAAAAAAzQEAAAAAAAAVAwAAAAAAAOkCAAAAAAAAFAEAAAAAAAC+AAAAAAAAAPMAAAAAAAAAlAMAAAAAAAAKAQAAAAAAAOsBAAAAAAAAsQAAAAAAAAD+AgAAAAAAACEBAAAAAAAAjQMAAAAAAABiAAAAAAAAAEYAAAAAAAAAQwMAAAAAAABKAwAAAAAAAL0BAAAAAAAAcgEAAAAAAACDAAAAAAAAAJ8BAAAAAAAA5wMAAAAAAAAmAgAAAAAAAE0DAAAAAAAA+gMAAAAAAADlAwAAAAAAAMECAAAAAAAA7QMAAAAAAACqAgAAAAAAAJACAAAAAAAALAEAAAAAAACFAQAAAAAAABUAAAAAAAAAZwIAAAAAAADYAAAAAAAAADoBAAAAAAAAzwIAAAAAAADzAwAAAAAAABYCAAAAAAAAjgMAAAAAAABzAAAAAAAAANACAAAAAAAABgEAAAAAAABaAAAAAAAAAKwBAAAAAAAAlwAAAAAAAACZAgAAAAAAAJ4BAAAAAAAAYwEAAAAAAABeAwAAAAAAAPQDAAAAAAAABgAAAAAAAADsAAAAAAAAAFgDAAAAAAAAZgMAAAAAAACxAwAAAAAAAAUDAAAAAAAAIwEAAAAAAAC2AwAAAAAAANoAAAAAAAAALwEAAAAAAAApAwAAAAAAAOgAAAAAAAAAbgAAAAAAAABgAQAAAAAAAAIBAAAAAAAAeQIAAAAAAAACAgAAAAAAAAEAAAAAAAAA3wMAAAAAAAAiAgAAAAAAANEDAAAAAAAA6gAAAAAAAADGAQAAAAAAAMQAAAAAAAAA9gMAAAAAAAC/AgAAAAAAAG0AAAAAAAAAcAEAAAAAAAAQAwAAAAAAABkAAAAAAAAADQMAAAAAAACqAwAAAAAAAOYAAAAAAAAATwIAAAAAAAAFAAAAAAAAAGQAAAAAAAAABQAAAAAAAADdAwAAAAAAAL8DAAAAAAAAiQMAAAAAAACBAgAAAAAAABoAAAAAAAAADwEAAAAAAADmAAAAAAAAAEUBAAAAAAAASAAAAAAAAADcAAAAAAAAAJMAAAAAAAAAWQMAAAAAAABSAgAAAAAAAPIDAAAAAAAA+AMAAAAAAAAmAAAAAAAAAKMBAAAAAAAAJAIAAAAAAAAqAgAAAAAAAMsDAAAAAAAAVQIAAAAAAAC7AAAAAAAAAD8DAAAAAAAAigIAAAAAAAAxAgAAAAAAADUCAAAAAAAAyAEAAAAAAADNAwAAAAAAAD0BAAAAAAAATAIAAAAAAABXAwAAAAAAALIBAAAAAAAA0wEAAAAAAADBAwAAAAAAAPMAAAAAAAAAYwEAAAAAAAAqAwAAAAAAAP4DAAAAAAAAeQEAAAAAAACZAwAAAAAAAKsAAAAAAAAA9gAAAAAAAAADAwAAAAAAANoBAAAAAAAAaQMAAAAAAACRAwAAAAAAAFMAAAAAAAAAmAMAAAAAAAC8AAAAAAAAALICAAAAAAAAqQAAAAAAAAB9AgAAAAAAAB4AAAAAAAAA3wMAAAAAAADuAQAAAAAAAEYCAAAAAAAARQIAAAAAAAC2AQAAAAAAAKMBAAAAAAAAKgMAAAAAAAC/AAAAAAAAAIEAAAAAAAAAtgIAAAAAAACOAwAAAAAAAAcAAAAAAAAA3AEAAAAAAAB8AQAAAAAAAIIDAAAAAAAACgMAAAAAAAA7AAAAAAAAADUBAAAAAAAAlAEAAAAAAADaAAAAAAAAABIBAAAAAAAAWAMAAAAAAAAcAwAAAAAAACEDAAAAAAAAfwIAAAAAAAChAQAAAAAAADQDAAAAAAAA2gAAAAAAAADjAQAAAAAAAKcAAAAAAAAAmwIAAAAAAADdAgAAAAAAAJAAAAAAAAAApAAAAAAAAAASAwAAAAAAAMgBAAAAAAAA1gAAAAAAAABBAwAAAAAAAF0CAAAAAAAAcgIAAAAAAADvAQAAAAAAAHABAAAAAAAAAgAAAAAAAABoAAAAAAAAAJMAAAAAAAAAMAEAAAAAAADPAgAAAAAAAHABAAAAAAAAAwMAAAAAAAAkAQAAAAAAAJsCAAAAAAAAeQIAAAAAAAA0AwAAAAAAAI0AAAAAAAAAsQMAAAAAAABZAQAAAAAAACUBAAAAAAAAfgIAAAAAAACdAAAAAAAAADgAAAAAAAAAVwAAAAAAAAA2AgAAAAAAAFUBAAAAAAAAhAIAAAAAAAAeAgAAAAAAAC4AAAAAAAAAqAAAAAAAAABAAAAAAAAAALoBAAAAAAAA0QMAAAAAAABbAgAAAAAAAPwBAAAAAAAAugIAAAAAAAD/AQAAAAAAADoAAAAAAAAAQgIAAAAAAADCAQAAAAAAAFwCAAAAAAAAfQEAAAAAAADgAQAAAAAAADEBAAAAAAAA4wEAAAAAAACxAAAAAAAAAHgBAAAAAAAA0QIAAAAAAACHAwAAAAAAANkDAAAAAAAAYQMAAAAAAACaAwAAAAAAABEBAAAAAAAAwAIAAAAAAABXAgAAAAAAAGkBAAAAAAAAaQAAAAAAAACGAAAAAAAAAEUDAAAAAAAAcQAAAAAAAADfAQAAAAAAAGkCAAAAAAAA0gEAAAAAAADVAQAAAAAAAPEBAAAAAAAAOAEAAAAAAACXAAAAAAAAALwBAAAAAAAA0AAAAAAAAAC+AwAAAAAAAEsBAAAAAAAA3gEAAAAAAAC7AwAAAAAAADQBAAAAAAAAKAMAAAAAAABzAgAAAAAAAHECAAAAAAAAIAEAAAAAAADnAQAAAAAAAE8BAAAAAAAAzAAAAAAAAAAiAQAAAAAAAIsAAAAAAAAADgAAAAAAAADuAAAAAAAAANAAAAAAAAAAywIAAAAAAABbAwAAAAAAALsCAAAAAAAA0AEAAAAAAAByAQAAAAAAAMYBAAAAAAAAxgEAAAAAAACmAwAAAAAAACUCAAAAAAAA+QIAAAAAAADMAgAAAAAAABQAAAAAAAAAFwEAAAAAAAC/AgAAAAAAAMgBAAAAAAAAKgEAAAAAAADTAQAAAAAAAKADAAAAAAAAggEAAAAAAACTAwAAAAAAAJ8CAAAAAAAAgQEAAAAAAABUAAAAAAAAACECAAAAAAAAOQEAAAAAAACXAwAAAAAAAEgCAAAAAAAAUAMAAAAAAAA1AgAAAAAAADYBAAAAAAAA+gMAAAAAAADsAwAAAAAAAH8CAAAAAAAAHwMAAAAAAADkAQAAAAAAAH0CAAAAAAAAoAIAAAAAAABxAwAAAAAAALQBAAAAAAAA6QMAAAAAAAB8AwAAAAAAAHIDAAAAAAAA0QMAAAAAAAB/AgAAAAAAAFoAAAAAAAAAyQIAAAAAAAAiAwAAAAAAAB4AAAAAAAAAzQEAAAAAAABsAgAAAAAAAFBLBwgcKXergDEAAIAxAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAAA4ABABrZS50bXAvdmVyc2lvbkZCAAAzClBLBwjRnmdVAgAAAAIAAABQSwMEAAAICAAAAAAAAAAAAAAAAAAAAAAAAB0AMwBrZS50bXAvLmRhdGEvc2VyaWFsaXphdGlvbl9pZEZCLwBaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWlpaWjE1NzI4MTk5NTExODU0NjQzMDI2MDU0MDI3NDM3NjM1NzE4NDQ5MjhQSwcI1tnJ4ygAAAAoAAAAUEsBAgAAAAAICAAAAAAAAN36jX/BAQAAwQEAAA8AAAAAAAAAAAAAAAAAAAAAAGtlLnRtcC9kYXRhLnBrbFBLAQIAAAAACAgAAAAAAAC379yDAQAAAAEAAAAWAAAAAAAAAAAAAAAAABECAABrZS50bXAvLmZvcm1hdF92ZXJzaW9uUEsBAgAAAAAICAAAAAAAAD93cekCAAAAAgAAABkAAAAAAAAAAAAAAAAAkQIAAGtlLnRtcC8uc3RvcmFnZV9hbGlnbm1lbnRQSwECAAAAAAgIAAAAAAAAhT3jGQYAAAAGAAAAEAAAAAAAAAAAAAAAAAASAwAAa2UudG1wL2J5dGVvcmRlclBLAQIAAAAACAgAAAAAAAAcKXergDEAAIAxAAANAAAAAAAAAAAAAAAAAJYDAABrZS50bXAvZGF0YS8wUEsBAgAAAAAICAAAAAAAANGeZ1UCAAAAAgAAAA4AAAAAAAAAAAAAAAAAkDUAAGtlLnRtcC92ZXJzaW9uUEsBAgAAAAAICAAAAAAAANbZyeMoAAAAKAAAAB0AAAAAAAAAAAAAAAAA0jUAAGtlLnRtcC8uZGF0YS9zZXJpYWxpemF0aW9uX2lkUEsGBiwAAAAAAAAAHgMtAAAAAAAAAAAABwAAAAAAAAAHAAAAAAAAAMgBAAAAAAAAeDYAAAAAAABQSwYHAAAAAEA4AAAAAAAAAQAAAFBLBQYAAAAABwAHAMgBAAB4NgAAAAA=', 'Ut_Linh.pt': '...'}

for fname, b64_str in DEFAULT_VOICES_B64.items():
    dst = VOICES_DIR / fname
    if not dst.exists() or dst.stat().st_size == 0:
        dst.write_bytes(base64.b64decode(b64_str))

manifest_data = {
  "voices": [
    {"id": "Cao_Hung", "name": "Cao Hùng (Ông già)", "file": "Cao_Hung.pt", "gender": "Nam", "note": "Nam lớn tuổi, trầm ấm, dạn dày"},
    {"id": "Linh_Duc_Tu", "name": "Linh Dục Tú (Nữ trẻ)", "file": "Linh_Duc_Tu.pt", "gender": "Nữ", "note": "Nữ trẻ nhẹ nhàng, trong trẻo"},
    {"id": "Minh_Nhi", "name": "Minh Nhí (Trẻ em)", "file": "Minh_Nhi.pt", "gender": "Trẻ em", "note": "Bé gái ngây thơ, tự nhiên, hoạt bát"},
    {"id": "Minh_Quy", "name": "Minh Quý (Trai trẻ)", "file": "Minh_Quy.pt", "gender": "Nam", "note": "Nam thanh niên, rõ ràng, dứt khoát"},
    {"id": "Phuong_Thy", "name": "Phương Thy (Dẫn truyện)", "file": "Phuong_Thy.pt", "gender": "Nữ", "note": "Nữ dẫn truyện sâu lắng, truyền cảm"},
    {"id": "To_Thanh", "name": "Tô Thành (Kể chuyện)", "file": "To_Thanh.pt", "gender": "Kể chuyện", "note": "Giọng kể chuyện lôi cuốn, nhịp nhàng"},
    {"id": "Ut_Linh", "name": "Út Linh (Ma mị / Huyền bí)", "file": "Ut_Linh.pt", "gender": "Huyền bí", "note": "Nữ ma mị, huyền bí, kịch tính"}
  ]
}
(VOICES_DIR / "manifest.json").write_text(json.dumps(manifest_data, ensure_ascii=False, indent=2), encoding="utf-8")

# Đồng thời chép thêm nếu có từ /kaggle/input
import shutil
for p_pt in Path("/kaggle/input").rglob("*.pt"):
    dst = VOICES_DIR / p_pt.name
    if not dst.exists():
        shutil.copy2(p_pt, dst)

ds_pt = sorted(VOICES_DIR.glob("*.pt"))
print(f"\n📁 Thư viện giọng: {VOICES_DIR} (ĐÃ SẴN SÀNG {len(ds_pt)} GIỌNG TIẾNG VIỆT CHUẨN):")
for f in ds_pt:
    print(f"   · {f.stem:20s} ({f.stat().st_size/1024:.1f} KB)")


## 4 · Kiểm thử toàn diện trên GPU (Cả 2 Model & Mọi chức năng)

Ô này chạy thử nghiệm trực tiếp trên GPU Kaggle:
- **TTS tiếng Việt** với thẻ cảm xúc (ví dụ: `[cười]`, `[ngạc nhiên]`).
- **Clone Voice Zero-shot** từ file âm thanh vừa sinh.
- **Design Voice** từ mô tả phong cách.
- **Chuyển đổi Model** giữa OmDiOm và OmniVoice để xác nhận tính năng switch model hoạt động trơn tru trên GPU.


In [ ]:
import os, sys, time, gc
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

sys.path.insert(0, "/kaggle/working/server")
import torch
from engine import TTSEngine

print("🚀 Khởi tạo TTSEngine trên GPU...")
eng = TTSEngine(mock=False)
t0 = time.perf_counter()
eng.load()

print(f"⏱ Thời gian nạp model: {eng.load_time:.2f}s")
print(f"   Model đang active : {eng.model_name}")
print(f"   Thiết bị          : {eng.device}")
print(f"   Sample Rate       : {eng.sample_rate} Hz")
if torch.cuda.is_available():
    print(f"   VRAM đã sử dụng   : {torch.cuda.memory_allocated()/2**30:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

CAU_TEST = "Xin chào! [cười] Tôi là hệ thống giọng nói OmDiOm Việt hóa, đang chạy với tốc độ siêu tốc trên GPU của Kaggle."
print(f"\n── Câu thử nghiệm: \"{CAU_TEST}\" ──")

# 1. Test TTS
print("\n1) Kiểm tra TTS tiếng Việt có thẻ cảm xúc:")
res_tts = eng.synth(text=CAU_TEST, mode="tts", num_step=16)
print(f"   ✓ Độ dài âm thanh: {res_tts.duration:.2f}s | Thời gian sinh: {res_tts.gen_time:.2f}s | RTF: {res_tts.rtf:.3f}")
open("/kaggle/working/test_tts.wav", "wb").write(res_tts.wav)

# 2. Test Clone Voice Zero-shot
print("\n2) Kiểm tra Clone Voice Zero-shot:")
res_clone = eng.synth(
    text="Tôi vừa nhân bản thành công giọng nói này mà không cần huấn luyện thêm bất cứ bước nào.",
    mode="clone",
    ref_audio_bytes=res_tts.wav,
    ref_text=CAU_TEST,
    num_step=16
)
print(f"   ✓ Độ dài âm thanh: {res_clone.duration:.2f}s | Thời gian sinh: {res_clone.gen_time:.2f}s | RTF: {res_clone.rtf:.3f}")
open("/kaggle/working/test_clone.wav", "wb").write(res_clone.wav)

# 3. Test Design Voice
print("\n3) Kiểm tra Design Voice (Mô tả phong cách):")
res_design = eng.synth(
    text="Đây là giọng nói được thiết kế trực tiếp từ mô tả câu lệnh văn bản.",
    mode="design",
    instruct="female, young adult, moderate pitch",
    num_step=16
)
print(f"   ✓ Độ dài âm thanh: {res_design.duration:.2f}s | Thời gian sinh: {res_design.gen_time:.2f}s | RTF: {res_design.rtf:.3f}")
open("/kaggle/working/test_design.wav", "wb").write(res_design.wav)

# 4. Test Chuyển đổi Model sang OmniVoice rồi quay lại OmDiOm
print("\n4) Kiểm tra tính năng chuyển đổi model qua lại:")
try:
    sw1 = eng.switch_model("omnivoice")
    print(f"   ✓ Chuyển sang OmniVoice Base thành công: {sw1['model_name']} (trong {sw1.get('load_time', 0):.2f}s)")
    sw2 = eng.switch_model("omdiom")
    print(f"   ✓ Chuyển lại sang OmDiOm Việt hóa thành công: {sw2['model_name']} (trong {sw2.get('load_time', 0):.2f}s)")
except Exception as e:
    print(f"   Lưu ý chuyển đổi model: {e}")

# Giải phóng bộ nhớ của kernel để nhường trọn VRAM cho tiến trình máy chủ
del eng, res_tts, res_clone, res_design
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Hoàn tất kiểm thử GPU! Đã giải phóng bộ nhớ để sẵn sàng chạy máy chủ FastAPI.")


## 5 · Khởi động máy chủ Backend FastAPI

Máy chủ chạy độc lập ở tiến trình nền, quản lý model trên GPU và phục vụ toàn bộ API cho ứng dụng giao diện.


In [ ]:
import os, sys, time, json, secrets, subprocess, urllib.request
from pathlib import Path

PORT = 8000
SERVER_DIR = Path("/kaggle/working/server")
LOG = Path("/kaggle/working/server.log")

api_key = ""
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("OV_API_KEY")
    print("🔑 Đã tìm thấy khóa bí mật OV_API_KEY từ Kaggle Secrets.")
except Exception:
    api_key = secrets.token_urlsafe(12)
    print("🔑 Chưa đặt Secret 'OV_API_KEY' → Đã tự động tạo khóa phiên ngẫu nhiên.")

env = dict(os.environ)
env["OMNIVOICE_VOICES"] = "/kaggle/working/voices"
env.update({
    "OMNIVOICE_MOCK": "0",
    "OMNIVOICE_PRELOAD": "1",
    "OMNIVOICE_API_KEY": api_key,
    "HF_HOME": "/kaggle/working/models",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONPATH": str(SERVER_DIR),
})

os.system("pkill -f 'uvicorn app:app' 2>/dev/null; sleep 1")

logf = open(LOG, "w")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(PORT), "--log-level", "info"],
    cwd=SERVER_DIR, stdout=logf, stderr=subprocess.STDOUT, env=env
)
print(f"▶ Khởi chạy Uvicorn Backend (PID = {proc.pid}) · Log: {LOG}")

def hoi_health(timeout=2.0):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health", headers={"X-API-Key": api_key})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)

print("⏳ Đang nạp model trên GPU (khoảng 10–30 giây)...")
ok = False
for i in range(120):
    time.sleep(2)
    if proc.poll() is not None:
        print(f"❌ Server thoát bất ngờ (Exit Code: {proc.returncode}). Chi tiết 40 dòng log cuối:")
        print("\n".join(LOG.read_text(errors="ignore").splitlines()[-40:]))
        raise SystemExit("Server không thể khởi động.")
    try:
        d = hoi_health()
        if d.get("loaded") or d.get("mock"):
            print("\n✅ MÁY CHỦ GPU ĐÃ SẴN SÀNG PHỤC VỤ!")
            print(json.dumps(d, ensure_ascii=False, indent=2))
            ok = True
            break
    except Exception:
        if i % 5 == 0:
            print(f"   … đang nạp ({i*2}s)")
        continue

if not ok:
    raise SystemExit("❌ Quá thời gian chờ máy chủ sẵn sàng.")

API_KEY = api_key


## 6 · Mở đường hầm Cloudflared Tunnel (Bảo mật & Miễn phí)

Tạo đường hầm kết nối trực tiếp từ GPU Kaggle về trình duyệt máy tính của bạn.


In [ ]:
import os, re, subprocess, time, urllib.request
from pathlib import Path

CF_BIN = Path("/kaggle/working/cloudflared")
CF_LOG = Path("/kaggle/working/cloudflared.log")

if not CF_BIN.exists():
    print("⬇️ Đang tải Cloudflared...")
    os.system(f"curl -fsSL -o {CF_BIN} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system(f"chmod +x {CF_BIN}")

os.system("pkill -f cloudflared 2>/dev/null; sleep 1")
logf = open(CF_LOG, "w")
subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate", "--logfile", str(CF_LOG), "--loglevel", "info"],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
)

print("⏳ Đang thiết lập đường hầm Cloudflare Tunnel...")
URL = None
for _ in range(60):
    time.sleep(2)
    txt = CF_LOG.read_text(errors="ignore") if CF_LOG.exists() else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        URL = m.group(0)
        break

if not URL:
    print(CF_LOG.read_text(errors="ignore")[-2000:])
    raise SystemExit("❌ Không thể lấy link Cloudflare Tunnel.")

print(f"\n🌐 LINK ĐƯỜNG HẦM : {URL}")

try:
    req = urllib.request.Request(f"{URL}/health", headers={"X-API-Key": api_key})
    with urllib.request.urlopen(req, timeout=10) as r:
        d = json.load(r)
    print("✅ Kết nối Internet thông suốt:", d.get("model"), "| Thiết bị:", d.get("device"))
except Exception as e:
    print("Lưu ý kết nối Tunnel ban đầu:", e)


## 7 · Sao chép thông tin vào giao diện OmDiOm Studio

Sao chép **2 dòng dưới đây** và dán vào tab **⚙️ Cấu hình & Kết nối** trên giao diện web local:


In [ ]:
print("┌" + "─"*72 + "┐")
print("│  DÁN THÔNG TIN VÀO TAB «CẤU HÌNH & KẾT NỐI» TRÊN GIAO DIỆN MÁY TÍNH   │")
print("├" + "─"*72 + "┤")
print(f"│  ĐỊA CHỈ MÁY CHỦ (LINK) : {URL:<45}│")
print(f"│  MÃ KHÓA (API KEY)      : {api_key:<45}│")
print("└" + "─"*72 + "┘")
print("\n💡 Mẹo: Khi giao diện đã báo 'Đã kết nối', bạn có thể chuyển đổi linh hoạt")
print("   giữa model 'OmDiOm (Việt hóa)' và 'OmniVoice (Gốc)' ngay trong ô chọn model của tab Cấu hình!")


## 8 · Kiểm tra đầu-cuối qua đường hầm Cloudflared


In [ ]:
import io, json, urllib.request, uuid
from pathlib import Path

def goi_tts(url, text, key):
    boundary = "----" + uuid.uuid4().hex
    body = io.BytesIO()
    fields = {"text": text, "num_step": "16", "speed": "1.0", "guidance_scale": "2.0"}
    for k, v in fields.items():
        body.write(f"--{boundary}\r\nContent-Disposition: form-data; name=\"{k}\"\r\n\r\n{v}\r\n".encode())
    body.write(f"--{boundary}--\r\n".encode())
    h = {"Content-Type": f"multipart/form-data; boundary={boundary}", "X-API-Key": key}
    req = urllib.request.Request(f"{url}/api/tts", data=body.getvalue(), headers=h, method="POST")
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read(), dict(r.headers)

cau_e2e = "Hệ thống tổng hợp và nhân bản giọng nói OmDiOm Việt hóa đã kết nối hoàn hảo."
print("Đang gọi thử API qua đường hầm công khai...")
wav_bytes, headers = goi_tts(URL, cau_e2e, api_key)
print(f"✅ Thành công! Nhận {len(wav_bytes):,} bytes âm thanh WAV.")

for k in ["X-Gen-Time", "X-Audio-Duration", "X-RTF", "X-Mode"]:
    val = headers.get(k) or headers.get(k.lower())
    print(f"   {k:18s}: {val}")

try:
    from IPython.display import Audio, display
    display(Audio(wav_bytes, rate=24000))
except Exception:
    pass


## 9 · Giữ phiên hoạt động (Keep-Alive & GPU Monitor)

Ô này chạy vòng lặp gửi nhịp tim mỗi 3 phút để giữ phiên Kaggle không bị đóng do nhàn rỗi (idle).
> **Bạn có thể bấm nút ■ Stop bất cứ lúc nào: Máy chủ và link kết nối vẫn hoạt động bình thường!**


In [ ]:
import time, json, urllib.request, subprocess
from datetime import timedelta

t0 = time.time()
print(f"🟢 Đang theo dõi máy chủ Kaggle GPU. LINK = {URL}")
print("Bấm ■ Stop khi muốn dừng theo dõi (máy chủ uvicorn vẫn chạy nền bình thường).\n")

try:
    while True:
        elapsed = str(timedelta(seconds=int(time.time() - t0)))
        try:
            req = urllib.request.Request(f"http://127.0.0.1:8000/health", headers={"X-API-Key": api_key})
            with urllib.request.urlopen(req, timeout=5) as r:
                d = json.load(r)
            smi = subprocess.run(
                ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,temperature.gpu", "--format=csv,noheader,nounits"],
                capture_output=True, text=True
            ).stdout.strip().split("\n")[0]
            parts = [p.strip() for p in smi.split(",")]
            gpu_str = f"GPU: {parts[0]}% · VRAM: {parts[1]} MB · Temp: {parts[2]}°C" if len(parts) >= 3 else ""
            print(f"[{elapsed:>8}] ❤️ Sẵn sàng · Model: {d.get('model')} · {gpu_str}")
        except Exception as e:
            print(f"[{elapsed:>8}] ⚠️ Lỗi kiểm tra nhịp tim: {e}")
        time.sleep(180)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng vòng lặp theo dõi. Máy chủ uvicorn và Cloudflare Tunnel vẫn đang chạy!")
